# 3 · End-to-End Pipeline (fully instrumented)

The **third** of three notebooks. It trains the models and runs every experiment: **honeypot log →
trained stacked ensemble → LLM hold-out → red-team probes → packaged results.**

| # | Notebook | Makes | Needs |
|---|---|---|---|
| 1 | `1_honeypot_generator` | the training corpus | 5 WEB-IDS23 CSVs |
| 2 | `2_llm_attack_generator` | the held-out test attacks | GPU + Ollama |
| **3** | **end_to_end_pipeline** ← you are here | trained models + all results | outputs of 1 and 2 |

It can also generate 1 and 2's data itself (Stages 2 and 6 below) — those notebooks just do it
standalone, with more knobs. If you already ran 1 and 2, leave Stages 2 and 6 off here and this
notebook trains and evaluates on their output.

## Why this notebook exists

`full_pipeline.ipynb` starts at *generation* and shells out with bare `!python`, so when a step
misbehaves you get a wall of stdout with no way to tell which stage produced it, what the inputs
were, or whether the artifact on disk is the one you think it is. This notebook is built the
other way round: **nothing runs outside the instrumentation harness defined in Stage 0.**

Every stage emits, automatically:

| What | Why it matters when debugging |
|---|---|
| Stage banner with inputs + params | You see what a stage was *given*, not just what it printed |
| SHA-256 + size + mtime of every input and output | Proves the artifact you're reading is the one just written |
| Wall-clock duration | Catches the "it silently did nothing in 0.2 s" failure |
| Live-streamed, tagged subprocess output | Subprocess noise is attributed to a stage, not lost |
| Explicit PASS/FAIL content checks | A stage that "ran fine" but produced 0 rows fails loudly |
| A JSON record appended to the run log | Machine-readable audit trail of the whole run |
| Environment + library versions (Stage 1) | The #1 cause of "different numbers on a different machine" |

**Design rule:** a stage never silently skips. If an input is missing it prints a `SKIP` block
naming *exactly which file* is absent and *which stage produces it*.

## Stage map

| # | Stage | Script | Needs | Skippable? |
|---|---|---|---|---|
| 0 | Instrumentation harness | *(this notebook)* | — | no |
| 1 | Environment + repo + versions | *(this notebook)* | — | no |
| 2 | Synthetic honeypot generation | `webids23_to_honeypot_log_v9.py` | WEB-IDS23 CSVs | **yes** — log is committed |
| 3 | Corpus audit | *(this notebook)* | honeypot log | no |
| 4 | Feature/split preparation | `prepare_honeypot_for_training.py` | honeypot log | diagnostic only |
| 5 | Train stacked ensemble | `18_train_stacked.py` | honeypot log | no |
| 6 | Ollama + LLM payload generation | `25_*`, `26_*` | GPU + Ollama | **yes** — CSVs committed |
| 7 | Assemble LLM hold-out | `27_assemble_llm_holdout.py` | stage-6 CSVs | no |
| 8 | Evaluate | `17_*`, `19_*`, `20_*` | models + hold-out | no |
| 9 | Ablation + significance | `14_*`, `13_*` | prepared splits | yes |
| 10 | Red-team probes | `16_*`, `23_*`, `24_*` | models | yes |
| 11 | ModSecurity baseline | `21_*` | Docker | **yes** — no Docker on Colab |
| 12 | Multi-seed variance | `22_*` | GPU | yes (slow) |
| 13 | Run report + package | *(this notebook)* | — | no |

> **Colab:** Runtime → Change runtime type → **T4 GPU** before running. Keep the tab open.
> **Local:** run it from anywhere inside the repo; Stage 1 detects the checkout and skips cloning.

---
## Stage 0 — Instrumentation harness

Run this first. It defines the machinery every later stage uses. It touches no data — it only
sets up logging, hashing and the run record, so it is safe to re-run at any point.

**Objects this cell defines:**

- `RUN_ID` — timestamped id for this execution; every artifact of this run is tagged with it.
- `stage(...)` — context manager: banner, timing, input/output fingerprinting, error capture.
- `sh(...)` — subprocess runner with **live** tagged output that is *also* captured to the log.
- `describe(path)` — fingerprint (size, sha256, mtime) of any file or directory.
- `check(cond, msg)` — recorded assertion, printing PASS/FAIL instead of failing silently.
- `note(msg)` / `skip(reason)` — attach human-readable context to the current stage record.
- `RUN_LOG` — in-memory list of stage records, also appended to disk as JSONL at each stage exit.

### Why a harness instead of `!python`

`!` magics cannot be captured, do not run conditionally inside a Python `if`, and lose their
output to the run record. `sh()` fixes all three: output streams live *and* is retained.

In [ ]:
# ==========================================================================
# STAGE 0 - INSTRUMENTATION HARNESS
# Defines the logging / hashing / timing machinery every later stage uses.
# Touches no data. Safe to re-run: it resets the harness, never the artifacts.
# ==========================================================================
import os, sys, json, time, hashlib, platform, subprocess, traceback, shutil, textwrap
from pathlib import Path
from datetime import datetime, timezone
from contextlib import contextmanager

# Windows consoles default to cp1252; attack payloads routinely contain
# characters outside it (full-width quotes, CJK, emoji). Without this, printing
# a payload raises UnicodeEncodeError and kills an otherwise healthy run.
try:
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass          # Python < 3.7 or a stream that does not support it
os.environ.setdefault("PYTHONIOENCODING", "utf-8")

RUN_ID      = datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%SZ")
RUN_STARTED = time.time()
RUN_LOG     = []      # list of stage records (dicts)
_STAGE_SEQ  = [0]     # mutable counter so stages number themselves

IS_COLAB = ("google.colab" in sys.modules) or os.path.isdir("/content")

# Re-pointed at the repo in Stage 1, once ROOT is known.
RUN_LOG_PATH = Path("/content" if IS_COLAB else ".") / ("aigis_%s.jsonl" % RUN_ID)

try:
    _W = min(shutil.get_terminal_size().columns, 100)
except Exception:
    _W = 100
_W = max(_W, 70)

def rule(ch="="):
    print(ch * _W)

def fmt_bytes(n):
    """Human-readable byte count (keeps stage banners scannable)."""
    n = float(n)
    for unit in ("B", "KB", "MB", "GB"):
        if n < 1024.0 or unit == "GB":
            return ("%d %s" % (n, unit)) if unit == "B" else ("%.1f %s" % (n, unit))
        n /= 1024.0

def sha256_file(path, chunk=1 << 20):
    """Full-file SHA-256 - used to prove an artifact changed (or did not)."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()

def describe(path, label=None):
    """Fingerprint a file/dir: existence, size, sha256, mtime.

    Returns a dict that is embedded in the run log. It deliberately prints
    nothing itself - stage() does the printing so ordering stays consistent.
    """
    p = Path(path)
    label = label or str(p)
    if not p.exists():
        return {"label": label, "path": str(p), "exists": False}
    st = p.stat()
    if p.is_dir():
        files = sorted(x for x in p.rglob("*") if x.is_file())
        return {"label": label, "path": str(p), "exists": True, "is_dir": True,
                "n_files": len(files),
                "bytes": sum(x.stat().st_size for x in files)}
    return {"label": label, "path": str(p), "exists": True, "is_dir": False,
            "bytes": st.st_size,
            "sha256": sha256_file(p),
            "mtime": datetime.fromtimestamp(st.st_mtime, timezone.utc).isoformat()}

def _print_desc(d, prefix="   "):
    if not d["exists"]:
        print("%s[MISSING] %s  ->  %s" % (prefix, d["label"], d["path"]))
    elif d.get("is_dir"):
        print("%s[dir ] %s: %d files, %s" % (prefix, d["label"], d["n_files"],
                                             fmt_bytes(d["bytes"])))
    else:
        print("%s[file] %s: %s  sha256=%s...  mtime=%sZ"
              % (prefix, d["label"], fmt_bytes(d["bytes"]),
                 d["sha256"][:16], d["mtime"][:19]))

_CURRENT = {"rec": None}   # the stage record currently being filled

def check(cond, msg, fatal=False):
    """Explicit, recorded assertion.

    A stage that 'ran without error' but produced an empty file is the single
    most common silent failure here, so stages assert on the CONTENT of what
    they produced, not merely on an exit code.
    """
    ok = bool(cond)
    print("   [%s] %s" % ("PASS" if ok else "FAIL", msg))
    rec = _CURRENT["rec"]
    if rec is not None:
        rec["checks"].append({"ok": ok, "msg": msg})
    if not ok and fatal:
        raise AssertionError(msg)
    return ok

def sh(cmd, cwd=None, env=None, timeout=None, echo=True):
    """Run a command, streaming output live AND capturing it to the run log.

    Returns (returncode, lines). Use this instead of ! magics everywhere.
    """
    if isinstance(cmd, str):
        shown = cmd
        popen_args = {"args": cmd, "shell": True}
    else:
        shown = " ".join(str(c) for c in cmd)
        popen_args = {"args": [str(c) for c in cmd], "shell": False}
    if echo:
        print("   $ %s" % shown)
    t0 = time.time()
    lines = []
    proc = subprocess.Popen(stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            cwd=str(cwd) if cwd else None, env=env,
                            text=True, bufsize=1, errors="replace", **popen_args)
    try:
        for line in proc.stdout:
            line = line.rstrip("\n")
            lines.append(line)
            print("   | %s" % line)
        proc.wait(timeout=timeout)
    except subprocess.TimeoutExpired:
        proc.kill()
        lines.append("*** TIMEOUT after %ss ***" % timeout)
        print("   | *** TIMEOUT after %ss ***" % timeout)
    dur = time.time() - t0
    rc = proc.returncode
    print("   -> exit=%s in %.1fs (%d lines)" % (rc, dur, len(lines)))
    rec = _CURRENT["rec"]
    if rec is not None:
        rec["commands"].append({"cmd": shown, "returncode": rc,
                                "seconds": round(dur, 2),
                                "output_tail": lines[-40:]})
    return rc, lines

@contextmanager
def stage(name, purpose, inputs=None, outputs=None, params=None, optional=False):
    """Wrap one pipeline stage in a documented, timed, fingerprinted block.

    Prints a banner (purpose / params / input fingerprints), runs the body, then
    re-fingerprints the declared outputs so you can see exactly what changed.
    Exceptions are captured into the record and re-raised unless optional=True.
    """
    _STAGE_SEQ[0] += 1
    n = _STAGE_SEQ[0]
    rec = {"run_id": RUN_ID, "stage_seq": n, "name": name, "purpose": purpose,
           "params": params or {}, "optional": optional,
           "started_utc": datetime.now(timezone.utc).isoformat(),
           "commands": [], "checks": [], "notes": [],
           "inputs": [], "outputs": [], "status": "running"}
    prev = _CURRENT["rec"]
    _CURRENT["rec"] = rec

    rule("=")
    print("STAGE %d: %s" % (n, name))
    rule("=")
    print("PURPOSE : %s" % purpose)
    if params:
        print("PARAMS  :")
        for k, v in params.items():
            print("   %s = %r" % (k, v))
    if inputs:
        print("INPUTS  :")
        for lbl, p in inputs.items():
            d = describe(p, lbl); rec["inputs"].append(d); _print_desc(d)
    print("START   : %sZ" % rec["started_utc"][:19])
    rule("-")

    t0 = time.time()
    try:
        yield rec
        if rec["status"] == "running":
            rec["status"] = "ok"
    except Exception as e:
        rec["status"] = "error"
        rec["error"] = {"type": type(e).__name__, "message": str(e),
                        "traceback": traceback.format_exc()}
        rule("-")
        print("!! STAGE %d FAILED: %s: %s" % (n, type(e).__name__, e))
        print(textwrap.indent(traceback.format_exc(), "   "))
        if not optional:
            _finish(rec, t0, outputs, prev)
            raise
    _finish(rec, t0, outputs, prev)

def _finish(rec, t0, outputs, prev):
    rec["seconds"] = round(time.time() - t0, 2)
    rule("-")
    if outputs:
        print("OUTPUTS :")
        for lbl, p in outputs.items():
            d = describe(p, lbl); rec["outputs"].append(d); _print_desc(d)
    n_fail = sum(1 for c in rec["checks"] if not c["ok"])
    verdict = rec["status"].upper()
    if rec["status"] == "ok" and n_fail:
        verdict = "OK (with %d FAILED check%s)" % (n_fail, "s" if n_fail > 1 else "")
    print("RESULT  : %s   duration=%ss   checks=%d passed=%d failed=%d"
          % (verdict, rec["seconds"], len(rec["checks"]),
             len(rec["checks"]) - n_fail, n_fail))
    rule("=")
    print()
    RUN_LOG.append(rec)
    _CURRENT["rec"] = prev
    try:
        with open(RUN_LOG_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps(rec, ensure_ascii=False, default=str) + "\n")
    except Exception as e:
        print("   [warn] could not append to run log: %s" % e)

def note(msg):
    """Attach a human-readable note to the current stage record (and print it)."""
    print("   * %s" % msg)
    if _CURRENT["rec"] is not None:
        _CURRENT["rec"]["notes"].append(msg)

def skip(reason, produced_by=None):
    """Record that a stage was skipped, naming what would produce its input."""
    print("   [SKIP] %s" % reason)
    if produced_by:
        print("          -> produced by: %s" % produced_by)
    if _CURRENT["rec"] is not None:
        _CURRENT["rec"]["status"] = "skipped"
        _CURRENT["rec"]["notes"].append("SKIPPED: " + reason)

def preview_json(path, max_chars=1400):
    """Print a JSON report inline so results live next to the code that made them."""
    p = Path(path)
    if not p.exists():
        print("   [MISSING] %s" % p); return None
    obj = json.loads(p.read_text(encoding="utf-8"))
    # ensure_ascii=True: results JSONs embed raw attack payloads. Escaping them
    # keeps this printable on any console, and the returned object is unescaped.
    txt = json.dumps(obj, indent=2, ensure_ascii=True)
    print(textwrap.indent(txt[:max_chars], "   "))
    if len(txt) > max_chars:
        print("   ... (%d more chars; full object returned)" % (len(txt) - max_chars))
    return obj

print("Harness ready.  RUN_ID=%s" % RUN_ID)
print("Environment   : %s" % ("Google Colab" if IS_COLAB else "local"))
print("Run log       : %s" % RUN_LOG_PATH)
print("Defined       : stage(), sh(), check(), describe(), note(), skip(),")
print("                preview_json(), RUN_LOG")

---
## Stage 1 — Environment, repository, and version pinning

Locates the repo (clone on Colab, auto-detect locally), sets `ROOT` to `dashboard/`, and records
the full environment.

### Why the version record is not optional

`requirements.txt` pins **`scikit-learn==1.8.0`** with a comment explaining why: the committed
models in `models/` were pickled under 1.8.0. Unpickling a scikit-learn estimator under a
different minor version raises `InconsistentVersionWarning` and **can silently change
predictions** — the model loads, scores, and reports plausible-but-wrong numbers.

So this stage does two things beyond printing versions:

1. **Records** every relevant library version into the run log, so a result JSON can always be
   traced back to the stack that produced it.
2. **Actively warns** if the installed scikit-learn differs from the pin, *before* any model is
   loaded — the point at which the bug would otherwise become invisible.

`GPU_AVAILABLE` set here decides later whether LSTM stages are fast (T4, ~10–20 s/epoch) or slow
(CPU, ~165 s/epoch).

In [ ]:
# ==========================================================================
# STAGE 1 - ENVIRONMENT, REPOSITORY, VERSION PINNING
# ==========================================================================
REPO_URL   = "https://github.com/Judge09/AI-GIS_dashboard.git"
SKLEARN_PIN = "1.8.0"          # must match requirements.txt - see markdown above
INSTALL_DEPS = IS_COLAB        # locally we assume the venv is already set up

with stage(
    "Environment + repository",
    "Locate dashboard/, record the exact software stack, and flag any drift from "
    "the scikit-learn pin that would silently change model predictions.",
    params={"REPO_URL": REPO_URL, "SKLEARN_PIN": SKLEARN_PIN,
            "IS_COLAB": IS_COLAB, "INSTALL_DEPS": INSTALL_DEPS},
) as rec:

    # ---- 1a. locate or clone the repo -------------------------------------
    def find_root():
        """Return the dashboard/ dir, searching cwd and its parents.

        Anchored on DATA (data/honeypot_final.log), never on a script: the
        scripts are embedded in this notebook and written out in Stage 1b, so
        scripts/ may legitimately not exist yet when this runs. Anchoring on a
        script would make root detection depend on the very files this notebook
        is responsible for creating.
        """
        ANCHORS = ("data/honeypot_final.log", "data/eval/holdout_eval.csv")
        for base in [Path.cwd()] + list(Path.cwd().parents):
            for a in ANCHORS:
                if (base / "dashboard" / a).exists():
                    return base / "dashboard"
                if (base / a).exists():
                    return base
        return None

    ROOT = find_root()
    if ROOT is None and IS_COLAB:
        note("No checkout found - cloning the repo (Colab).")
        sh(["git", "clone", "--depth", "1", REPO_URL, "/content/AI-GIS_dashboard"])
        ROOT = Path("/content/AI-GIS_dashboard/dashboard")
    if ROOT is None:
        raise FileNotFoundError(
            "Could not find dashboard/scripts/18_train_stacked.py in the cwd or any "
            "parent. Run this notebook from inside the repo, or set ROOT by hand.")

    ROOT    = ROOT.resolve()
    SCRIPTS = ROOT / "scripts"
    DATA    = ROOT / "data"
    EVAL    = DATA / "eval"
    PREP    = DATA / "prepared"
    MODELS  = ROOT / "models"
    REPORTS = ROOT / "reports"
    for d in (DATA, EVAL, PREP, MODELS, REPORTS):
        d.mkdir(parents=True, exist_ok=True)
    os.chdir(ROOT)          # every script resolves paths relative to dashboard/
    note("ROOT = %s" % ROOT)

    # Move the run log into the repo now that we know where it lives.
    _old = RUN_LOG_PATH
    RUN_LOG_PATH = REPORTS / ("run_log_%s.jsonl" % RUN_ID)
    if _old.exists():
        RUN_LOG_PATH.write_text(_old.read_text(encoding="utf-8"), encoding="utf-8")
    note("Run log now at %s" % RUN_LOG_PATH)

    # ---- 1b. optional dependency install (Colab) --------------------------
    if INSTALL_DEPS:
        note("Installing pinned deps on Colab (quiet).")
        sh("pip -q install 'scikit-learn==%s' pandas numpy scipy tensorflow "
           "imbalanced-learn" % SKLEARN_PIN)

    # ---- 1c. record the environment ---------------------------------------
    ENV = {"python": sys.version.split()[0], "platform": platform.platform(),
           "processor": platform.processor() or "unknown", "cwd": str(Path.cwd())}
    for mod in ("numpy", "pandas", "scipy", "sklearn", "tensorflow",
                "imblearn", "matplotlib"):
        try:
            m = __import__(mod)
            ENV[mod] = getattr(m, "__version__", "?")
        except Exception as e:
            ENV[mod] = "NOT INSTALLED (%s)" % type(e).__name__

    rc, git_lines = sh("git rev-parse HEAD && git status --porcelain | head -20",
                       cwd=ROOT, echo=True)
    ENV["git"] = git_lines[0] if git_lines and rc == 0 else "not a git checkout"
    ENV["git_dirty_files"] = [l for l in git_lines[1:]] if rc == 0 else []

    print("\n   ENVIRONMENT")
    for k, v in ENV.items():
        if k != "git_dirty_files":
            print("      %-12s %s" % (k, v))
    if ENV["git_dirty_files"]:
        print("      %-12s %d file(s) modified vs HEAD:" % ("git-dirty", len(ENV["git_dirty_files"])))
        for l in ENV["git_dirty_files"]:
            print("                   %s" % l)
    rec["environment"] = ENV

    # ---- 1d. the pin check that actually matters --------------------------
    print()
    sk = ENV.get("sklearn", "")
    check(sk == SKLEARN_PIN,
          "scikit-learn == %s (installed: %s) - a mismatch can SILENTLY change "
          "predictions from the committed pickles" % (SKLEARN_PIN, sk))
    if sk != SKLEARN_PIN:
        note("MITIGATION: either `pip install scikit-learn==%s` and restart the "
             "runtime, or retrain from scratch in Stage 5 so the models match "
             "the installed version. Do NOT trust scores from committed pickles "
             "under a mismatched version." % SKLEARN_PIN)

    # ---- 1e. GPU -----------------------------------------------------------
    GPU_AVAILABLE = False
    try:
        import tensorflow as tf
        gpus = tf.config.list_physical_devices("GPU")
        GPU_AVAILABLE = len(gpus) > 0
        print("   TensorFlow sees %d GPU(s): %s" % (len(gpus), gpus))
    except Exception as e:
        print("   TensorFlow GPU probe failed: %s" % e)
    note("GPU_AVAILABLE=%s -> LSTM epochs are ~%s"
         % (GPU_AVAILABLE, "10-20 s (T4)" if GPU_AVAILABLE else "~165 s (CPU)"))
    rec["environment"]["gpu_available"] = GPU_AVAILABLE

    # ---- 1f. inventory ------------------------------------------------------
    print("\n   REPO INVENTORY (what already exists before this run)")
    KEY_ARTIFACTS = {
        "honeypot log":        DATA / "honeypot_final.log",
        "hold-out eval CSV":   EVAL / "holdout_eval.csv",
        "RF model":            MODELS / "rf2.pkl",
        "LSTM model":          MODELS / "lstm_best.keras",
        "meta-learner":        MODELS / "meta.pkl",
        "n-gram vectorizer":   MODELS / "ngram_vectorizer.pkl",
        "prepared splits dir": PREP,
        "reports dir":         REPORTS,
    }
    for lbl, p in KEY_ARTIFACTS.items():
        _print_desc(describe(p, lbl))
    check((DATA / "honeypot_final.log").exists(),
          "training corpus data/honeypot_final.log is present "
          "(if missing, Stage 2 must generate it from the WEB-IDS23 CSVs)")

---
## Stage 1b — Materialise the embedded scripts

**This notebook carries its own scripts.** All 23 `.py`/`.json` files the pipeline needs are
gzip+base64 embedded in the next cell and written to `scripts/` when you run it. Nothing is
downloaded; no external file is required.

### Why write files instead of pasting the code into cells

The scripts import each other (`from text_normalize import normalize_text`) and several are
launched as **subprocesses** — so a crash cannot kill your kernel, and each stage can capture its
own stdout. Both mechanisms need real files on disk. Embedding-then-writing keeps ~4,800 lines of
working code working, without restructuring any of it into notebook cells.

### It will not silently clobber your edits

A file is overwritten **only if its content differs** from the embedded copy, and every such
overwrite is listed. If you edit a script locally and re-run this cell, you will be told the file
was replaced — so you can re-apply the change and re-embed rather than losing it quietly.

In [ ]:
# ========================================================================
# EMBEDDED SCRIPTS - this notebook carries its own scripts.
#
# Every .py/.json below is gzip+base64 embedded in this cell and written
# to scripts/ at runtime. Nothing is downloaded and no external file is
# required: the notebook is the single source of truth.
#
# WHY write to disk instead of exec()-ing them inline:
#   - the scripts import each other (`from text_normalize import ...`)
#   - several are launched as subprocesses so a crash cannot kill the
#     kernel, and so their stdout can be captured per stage
#   Writing real files keeps both mechanisms working unchanged.
#
# Existing files are only overwritten when the content differs, and any
# difference is reported - so a local edit is never silently clobbered.
#
# 23 files, 210 KB raw, 106 KB embedded
# ========================================================================
import base64, gzip, hashlib, os
from pathlib import Path

_EMBEDDED = {
    "13_statistical_significance.py":
        "H4sIAAX0jGoC/9Ua227bRvadX3GWwdZkK9OSkhSosWrhOnLqRRIbkrIooDUIihrarHkrZ6REzbroR/RH9hf2U/ole85cyKEu"
        "DvZlURuOJQ7Pbc59zuTZX05WvD5ZpMUJK9ZQbcRdWTx3XNd1Bs9DLiKRcpHGURby9LZIE/xaxCyoNo5zzeqkrHMOdXpb1uWK"
        "gwUOguGX4hbKAsQdk4/AmYCqZss0FmlZcBAlPpZr5oi7SEiwqYjie7aEccFZvsjYEYdKsSG2cBuliBbVzGaVbaCVTThxmVcI"
        "sSTqRHIRcQZ5uWQZB29yAVGxhDfT2Vs/cJwJ+3mV1ixnhcCX53dRJVgNzyFnqIZlmZW3G//UARgE8H1ZCi7qqIJvXv4Vzssi"
        "SZeMZLosEGcdIXUUEy4GPbi4nvQkm7P358eTq3PwBr1+vw8141FeZYwja4BhAG/jdyyPatzkjNTjRVl1F436Qf+lT9Lrnchd"
        "VBHnJ0mUZhDd1kxKDHkk6jRmHInhT5ns1SDCk9IELEpxZysDt3+1EtVKcNphzaqyFvzkoMV/4mXhOO+nZ6/H4CV1mZPpfmKx"
        "gBo1I7WknAd4XKcVkvqM/5CLOWlObEF9ZOkiWIk0M6uSpf5epfF9xswT33DzVaQ5c5Q8kbhDEpoYXONjw6BY5dUGIg5F1VBE"
        "C+EC/lZLRYAjh6gugiwt8DOUajLU3pS3ciMThurnHN23i4Meg7bgBjwZhDwua9ZD7cRhtIrNY4yOsyL0UFrvY7OjOK02AemL"
        "O84z+OP33/AXpgwt6U1WaMahrxef1q/ZX43aLnNnOh6/ghG8GDpqIeC4Q49Wfaeogt3FVhdkT66V8fUTVIZzPbn6+/h8Fk6u"
        "rmaoA9qPF4ZJmrEw9AOK9ELoD+fV2ewsfHU5kZGNsDbqCbjLSEQufcFsKpOd67y9ejV+M9U4uxgq5l1nMr6+msw03C6YTgMd"
        "uCC/X6a1pyTjo1m9Qj9mHzEcwvJePvrOdHaGwFfvZ9fvaW82F6T6aFLBLGA5vIHk8APLMPXzJ2NoZ8kSyOOCMnpI5c7bhELq"
        "ahNS0RuYL0OZLAEoAdLnOab5FSJY5UBWS+9DihkbEwbW0VUqNvi1rpksnb6sNeJD2eRyIqT5yNKjWclaicU9qjedyuv1AQkM"
        "/KAjiWYwQAt6hthoBGofvg0ybEGGNoiEeQaLUyUYDAxCTy8M4UNdFrcSboFEMOj5Kvcazl/Ar4aFr4nFLTGJ25LSkEqwltiv"
        "FrWWmIRKE+T6FQGPoK/MQD81E6u6gE9ufJcO3VPAEtzD4AqxrK8YPg/ks9Vl4NoFlnw0rrvA72khvIWPD7F+iP0HowttVThs"
        "TSU/cpb1kjTrRQvuLeAYYh//IHcfvvySSsAJeFJ+pRstIGIgCAJaRSQgekG8TLyGcA/trYQy2232b/aNXVyx9JKsjESLh/t6"
        "4fda2FYtClA/2yBdTS3KMjNQ8DeQ/Y0FbCnQEqhRpFxDbcr4ilWshElVb4WXDqo4R21sF9kt0B5k0QKjZjRHqw5uFAdR9CCp"
        "8B9+ioqI5FiO1izzfFtnSUU2wL9fIYZP/tQ+fAt9QLKMdqjlXZiuMYzTjhTlIupBEabYO0YyJEcDbBGxbWxbwANZ4mD7uU6j"
        "ll+FvXc3uAtMvb+Qr2Ss8OyQNr0Kx3fzG7VU1TtrTROj16yUY5IBbgq+xcAKXvpBxMWmYh7aUDsdZawUbUqtwC3z7L37bSQ+"
        "g6nskVW4YDnKolh2uw1EWiyp6VXhrhsG+iBv6ff0Ns2n36Cp1jtUGweTseaa2s0unN6Y+vIoHG17ZKy6C2nt7arINsaHyXcU"
        "EbjDJlS253GGbT7uzasxbyNoVEDK+UprI4vqWwbv2j0hBTIm6mFVpD+vmNfZpE8eOWhV27F1gC7CKNb1Qhe111VDD35hdRku"
        "03VKUTXq+36XauMuhqwdp49R3iLU+pgh1GmfHyGFmve1o1Hc3TJBEadoWe6FGitK6rZpvauapgbkLCqaGpCVH1jdPK1QqPYJ"
        "s/ptqkEfGloSQ3knAsfoutjhaUl6+oyHOWQokzpg1LcqkOQPow4wxT+GToJTllUU6MkooAWRIiNAHn2URabBOVaCY7KnZSXJ"
        "cUPSstNO8ZBJRivNLiANbrd+SHCjVhve8N8BNlq3geXaPuDGKB1R5KIEb6AfDtXCZIDY2oGaeLFrEzq1BdH4vg1CLouu24K1"
        "nm2DtSkQISmUW36dspfjEd7Tbtz0yjNW8LK+QLWZU+efvUOWdbbeWOGoz/ByJwntBDOhSJr3ImmPhCK0joVN5qEzuIWuCd6j"
        "WtVYhH2MWSXgUq6P67qsW+5VTTXDnY8nk6vJja1PyhE0NImyjC0DF7sCDMQR33Bsrpasrq2ygmt4FhKe6a000ZGL0fm1dje9"
        "dnZ5/PpyCvAvgMdnI67/OVIAs4jfn25PpS65bP4fmSu5TY+uvegfrE6TDYaBnIQt8cAg0iSKhTlqv/j/HrV1TGphRnZcTi5g"
        "PVS9P0aL+rFOvXR6TYZBdZ+5VoTRpA97axbfV2Uqu9FtpIyLPFygngLpNTYyccTUuB42/ACsk7nkSBkuXGOrzddbqPIQZ+Pu"
        "oBLEHlwpM5K1uG7hSpkRIiiqX3YwiepBgSUmQVioqnbl2GnQwBabu3vZrN33YC37NW2NALNVzj3fFNF1IIcAuHJjzlaahnWw"
        "ilLshyergsZ0Mvy8pAm4t5ojBRc/hU9HPTgKfkIzeZqO/7Djrm/KyPbRJzMcsGL3n8V8cPLiRm6Fto/hmxbo7HpGTQFM8x3g"
        "VZbiFoMg0PlANoEltkXeAbfHrwvXpxyaWCZI0KRqfBpgMcSSqKiRJ+Ab6fSBHiPQezX39DCreI8Hium3VADIqVO1DGqGJNCh"
        "vcbvdsJEH7iU838GzQ4RhfcjUurhyYMiE3E13WVdVth1Zqscj1JzVx7w3Bus+Ea+Zi2Q51BVHX4k8nggk6E6MiI9QkxD7CXW"
        "aDXUomETJhVub6kTua0hQi3BQZQmZLcDQp8L1WRf9sGLFP0mZX+O4LAdfz4kt3/NCtn0oOfbE6nte6LW7bXzqHOWdLM6CTSq"
        "WvXIJ/z5KZ3mDYq0lMHZh0IANo6xjEEayRWD5ZnXc/dH9wY9Yc3qRckZnoWCJIsE9iDeljmbk+EuHXqtCe2loxIyE4S9e/Pg"
        "qZ5IdhBs1DZEhBAk2IugFynvJZD43pvbKuxt7ZT8moJJ0eDq+qi7AUm4q759LFqc3rYWbnyjau293+/OJKnm6M7FGFGewfUM"
        "3Osadd+coVG/wUOsbXMcwjMbN/OMPYo4ONpogrFtx6gVe0qXE3aYPqcwVXmFovTgwGnnWtMKWtGnfIpFP6A/XlfHcYovu6Mx"
        "lYh3lb5nSGY3wgl2wq/KglGf8slih0dX0T8NBskD38mYW93xkzTRi66Julvi3Zvk1ijmdsKoec1D7A4oTLr3Fl1jqAmQjsaD"
        "hHQ78XlCTYBu22X8UZ7dvoDpKs8pOTwRm6D/rzLBu4eVJVPX4Oi12Iu77X8tmFr/R2JqHftgosjY7XwbIt+8xChBQttTF30Y"
        "crdv/d2tqUhn0NCG4dx+cbOFI+cgNiwu7MDIUUgHCFe2odppiA1pVttJ6YO1d+1IXOUC6c9hP+y/3FVCx5vx9V4v7x3EIX/c"
        "j0VvrHFR+7dtxO17T2y/P+x033TDGSxXeeVpN+lB0qMZNivEyNxKJXTjtq1pM4jffkX6bfvN5kzhwlewd1QwnY2vYfAc/vjt"
        "dyBxL6ezy/OzNzC9fP3u8gK/vjsfw2Q8ff9mNv3s4IHy7fbMAbxt7/NP3W0kgIvB8ZRGW4AGTAbzIxoPHt2cBi+SB/jPv/Wa"
        "nNOZVZjLRTkXNGvHClCO//TazT5m1+rqXDKr6j3c5OIOO1rd4UeLhxmi7mFn4oLV3yjFugvLNrBg2OLV3+3V0JrD5EIJXR2b"
        "m71Pe915fqTv00ikIUPhPUwm3x2EtoQ4unnw9zOX04PPMqeo+F/YK/jHBSAVqvQv+zY0mh1YDwfc0kkTCMMiylkY0oWuG4Y0"
        "KA1DV8Wfmpo6/wW56KPt4yYAAA=="
    ,
    "14_ablation_study.py":
        "H4sIAAX0jGoC/9Va607kRhb+j8Q7VJw/dtIYmgG0w26PxM40I1ZMYLsZKSsWldzdZXBw246rDHQIUh4iT5gn2e/UxZe+kES7"
        "KyWImcblc746dc6pc6nqL7/YrWS5O0myXZE9sGKh7vLszfaW53nbW/0DHk3SSCV5xqWqZouwWGxvbW8Nn8S0UkIydSeYN06e"
        "dt7n2SwhOnZiGdiYGDwWSTYTcZKJGUsy9v4uKpQo2ZsQKA9RWkUOZprPizwTmZIsj/XIWEXTe7ANMynmk1SAfyYKgf8ylS6Y"
        "yllR5g9ie0vdRYpJok6yWwBlqkwmWr65iDKMxVXKClHGeTmPsqlgE7GAvCzBXFKVeXYrpGKTSAo2z2ciDWmN9Yrk8fYWY/2Q"
        "jU5ZlEJE5j/sBzS2H7Lz8dUnM0oDbzCQ3yZSJVM2ErelkJI0ocpIr3+WlGJKotMYFtjvm0ljEakKxMz/Ox5T0Gr4Az3l1+yT"
        "UBHz9Uw/iDIXM/320E7u3oO09fYoZKdVmgLga021rEta4UWlikqZ5ZWiyEsld7vm5hCqSpUMv5N5Riyfxycfh8yPy3xOyv8O"
        "y2FlnqtAgxjnYXJaJgWw1nuPdqztrWROEzLzkSaTsFJJWg+b+exDkUzvSWL7KBey/lslc7ww4kTqDjgWkV3isTVPVs2LBfli"
        "VjS4UTbDCH6LmQWRmCgqs5BsEJVce4NDdJZtDLvENBfwu6l09HGfy2leih40NOVRNXWPBbwgkVor9r2YRmnqnuC/caVfzyMA"
        "PtEqvmS//PwTftlYwIj+qMJu2A/s4J/rt1Z/CfXn8+2t8XD4gQ3Ywf72lhkKJRbp0zBcOSvCNaMtjZChpVXJ0Z9QJdtbl6OL"
        "fwzfX/HRxcUVFEEL8jmPk1RwHoRFVCLe2Y/trQ8nVyf8w9mI0Q+I27y7zJtFKvLoDzgZscyw2T5dfBiejy3TKot2cgm684uP"
        "/PTsfPhr0NjjYlHkCiJmURqm+S2YR8PLi9GVnWWV2caXLmE4v0dI9M3S5OCqrOD+4gm7jOf3+hGmPvn7+cnV2cU3/OIzaac9"
        "D3BfiVc6zGxvIfkwnubRjHazn0Vz2oFQsQlZjMlCTIHbDUQhjXLa3cYOaT7V01h+bSINEhgQQLNVEIzCKw0MAfr0X9BMG5JY"
        "ogwF0ik3xD4+LEUpkBQygqZlcBtVOTS2yUlANd/bx9tmvc4PuM5BSIVcm7HHOniNv/DGuHlZMyFwe5BKo8LcNAMmCt3z9pY0"
        "AZHflnlViBlWmybKka192RiHEj+SNbfh019gWnKEBUeGmUQ9pEqY9S5PZ4O98DBgO++QSKfKmo+oEBIHzLf07N2g4QjCSKpF"
        "IfwkU1at0zmIl2NsZ1Ix67E0mmBTDK73eqx/YzlV1mNxgX/4VAWhzBGbHkTqW4K4INPEBfTp4/+vwRGwJG49vGN7DLiC7YV7"
        "tH7iAkEq4FlFWGXJ95WwsgQB+xvbt6ukH2QQoGtGeiSYlbedTLOsycDNaD3rueH24r537B6AUmUzP4Z5le+y2IqGqNiA8zzo"
        "TDbYC4IeOwh6bciibDA7kEW5QkxCQ3jD0CbGixXiOoESeZt4KbP+fplNJrZit5HbKfq3wRrUl8bP59hLvos6qly0jOeqGZHJ"
        "vMSEj1SUqLghUHGTBRVvZ0JHoSuRFoCFvBdlJK3DPE1FodiZfjEsy7xsSVCU2CG+dz0cjS5GN+xKA50SUJajSstQXaepmIWI"
        "HBRzBijCQqkQvMqWDDSI8K38fu1qFnfgsa/YkVOKHTw52/l4NmbsR1TXa0pFL9gAYYbrAuAcUYhBOai4sa1Rz1coSq+ooh8L"
        "xR4TdcdOo3mSJiiw/w/Zu629/u7hTVsMakco3sJo1K6IJwRUCBcpqsZZ7KQKw9AtFsGAFO5ScaizoazdRrtllCCAjKqMil9t"
        "Rz+uDXdRJreUlRmF6XmCwJvdHrNnh/fi5pnFvIweETFcDPcdiSXgPfpFD6X4LAbd2iDuG5geI48ctDzSrJDXKxw4pGuv/Wrh"
        "3YTUBAq53qqfdP19Uqokht4k+6NWcG0f2CcfIOnJ+K7zMzWW8dLTXd2RaQV2bK9dNUeD67cqNqqd4v2wuE+x9bxy4gUUG+K2"
        "P5B5TI+kM7IfW7xUKsp1OgSERoLQFQZIWnDRpYmIgU9gqFDzeEG92fQ6+APVFsUsLAVApvLBr4tRLSV3VCHeuUXBvMTHXuc0"
        "VG0+7S/E+PqMhsoxGtZvtRwIz/qTfM/JNSvzwp/maTXPkNs9neS9G4RrR1GP1W6pAfHQo1oDn3olVtyNcG7ZG+BIahAt6JO5"
        "vfEqnqVYxWsszbVwAzS4xgvamnLvw6z4wWt5h9bfazz03jHZnS2lQF7x69fXHnYxGwzMclBwpaiH4KouIp2TxBSKUGlN777w"
        "OjBU+Bg+QqCnpcjRRmoFVmwpU2LK5AfRAV8KI+NqguAr2adI3kv2p2oLzTqu7gRzNbkO6qYytRFWssmCyWqyY8Ip81FCIkjT"
        "qRLqEqlyXZ8gxgQOr0TjIEpUyFFmj92iEunEk9+nibfrPUnpsWkK84A9R81SNPAUyJHFIvM+NIjjf56f8dOTT2fnZ8MxXOnZ"
        "a4lAIWuS56mIMJAmmR6gtGVeN2OC0pjjaUJb58eT5gyLo0YuFxopqlQOnegH/H3HJ4sConkvdqONx41oWrZSxKkg9yEOox76"
        "C4WVnfzFedAcDsNJKWZ7RGUZLfzrGPvkUZR+QGeZ3aWjX2IxDS95sOseNCDUyzYDduT9NTxbKdRyhlG2AAy46mHMZkZXCj0U"
        "DI9Rmd0w6zdzvT30JsLWem5AZTX3gxdaaoKg8FzD2nFIjDL5Uf6VectWi70cW698AJ4THYUIqkpo38eeXNnpQV2cfMlOsYyJ"
        "yJLbTMP32COd/k7TaibIa+ekF31ETHJRViVBpN3r8FHbU7LTy1FL+xaRekUT+NBO2SltU44lcwMDqsYFfmzzG1IowVE6UjLu"
        "EuVSNPooMiRWyEXnvOwS3odGlo6Y2R+ylHlDpYyVmaoZfVhdtKSmQ2fSPhU0QbuUsSndNOQ6ZZZxaDnNqE9JNbg+pv665tFp"
        "zjGt49Gpos3kMpvjGugRx+a719fet94N8uiDKCe5FOjRwhjtBvolfzkdWqR1QCbdaaT1QM7c+kT+nA6FEWp9VwTmGZUFTONP"
        "4POKvN66JzEM1hwx+6btQ1MEt2lX2MQRxijBEUtMycB1fPSv27rvLSmIigmqYla7qOYC583x2ksM/Pb7xnFP3X3F/8LJDsjJ"
        "ruxBU1sM7W50EM/885GdvnNd0vE4xIykxJ4Gka2g6BrJlIIsKnXYWL5tQb1xq+56TGQKldeCjt7RVckmXRp2DsyB/TtM0nxK"
        "Dnjc76+p6ohU01LdtJHUepxJeL/R7D0ElieeKFEO+nt7LmxZDO0ItbCu8u3SLDm3ZVyzv4BQb7ElJ3E3dlqZR42x/qDx65Bc"
        "y8pMzoXSFFLXF5Qt8VuO5JZMb1n/uL71s6eH/VqFnXi1hnf/eOl2ENz7nejyOj/twxHz4VG1w1uUN8tmfB3o4Pi1e0TgHdR4"
        "Oqp0fWJdfGmm61EtQ2iSp8m98JeWFdwEnXDdkevw+LUbTMh1+Lvk6grSkTLoLev7NbmOjs3d6eqdqZbq6L/R1mYxLHrjkwN2"
        "3dRUvkei8T4fnfITciiUrc4X2yeYlm6fk2Ybyv2NlG/4+Qg7nlNg5xTYJdG/2Uh/QBJcppXkZDT+Ta5nIp6DjTyHRpoO1+iU"
        "eA438hxxsgG3NuDOBsR01GWqdUeHrHR6wevj//qk1jC0ymDP864cFUXWBD0khYjTfujg6KeOFS276G8bkKexvFLohsgLdlKb"
        "7JMMdSfzzb7bPQyYvEtiRfGyXYK7AgC92zRKhW61InYXlbOdaT6Dy+2Fh2wa0VU6WhS8oi8v6K8O7CAx6G85mKvpSDWwp/3B"
        "XoiMTce7RCGi6V0jN0j19x3yx0wrqbkjoS9JSJ0QO1Immf66hObtfD8DgpUlShcU5XcJpkgk/oj0Gpk7xKVB1MF0ddTRefNg"
        "DNUzn3Ff32wc9thO391v6B4CbQCdPrPmcoTug/W5PJ1ZJJksoqnw98I9ZL298O3bHnv7Frt9P2gbW0PRFCuXGX5zZdS5Klo9"
        "0++iofkC4Dsn/fFqy7q6PnORQNhxv3V6Z+5iDHlzQaOvMallta0sacLeXWqJk7ZLtqanvmV5D5jjJncF1Eys57gm1BuaqbsG"
        "7ML5hMpW73j1hq6N2NvAxyPFu5IA6fmrr14Da9/1qWDTQcD6H689j3pZFos6RX6RpYtN67luNX83Vp7O2MpKqV1/FbHpEWvA"
        "1lDLFvWpQ3ME3L727jHvcfXcl+64w1k1L3xrSjhWT39HKlOD/eUbmH9nHhLt+ouY8dXwkvUP2C8//cyOdt5ffPPhjKZmTgY2"
        "vvr84V/s/cWny/Ph1XDjxUzbUclDr6fXezd6ZNp12JvWMvTe6DjjdeN7+Dvuezctan3F+hp5Ubbp7bXoZnp38XizekTC2DMx"
        "hOl3lVT+/l+CF3ZMUfYZGz48iF/Q7J9ejvBYlO755PP7wTMg9bO3ZIMYRmBs+FToMxBmbp7ptAL7om3ul1duvhB4OCepOKfj"
        "C49zulbk3Dt25xyJ7kT/A0EI2OnXJwAA"
    ,
    "16_claude_redteam.py":
        "H4sIAAX0jGoC/7VXbW/bNhD+rl9xUzFU6mz5pWk7eNMAI3GKYQk6xG5XQBAEWqJszjKpkXTSLPB/35GU1Nhxhu7D/MGReXfP"
        "3T33IubFd4OdkoMl4wPKb6G+12vBX3u+79/Qoq8p2YJeU1Ca5BtaAOWKbpcVhTum10DgvCK7gvbJDs2kkd8SxQQHRXXkeedE"
        "UQUVu6XA8CyXrNZqgHoGN8uNNPpToXqQVGRJqx5ossIv+kWnP4E9glFMtHHeg2G8pJyteBh5M5KvgSmEtF71Wordag03l3A7"
        "hh/gar64xj9bqkm/okRyKoF+Ibmu7oEom5ANitQ1FAKD8G4oKfqCo/wukpGOYCsKWqkI/pBMYw6S1kJi7LnNN2tTkFTtKu2S"
        "wHw/zqfvZxCUUmyhIGq9FEQWg3ACDa8dBaO32SFSVN8bzj22NX7A4LXPNcs3FW1/qXvlWfya6HXFltCc/44/vVaJ77a1zZTX"
        "HQrhGJE5qwvPu/nwYQGxNQqyrGQVzbIwqomkXDd/PPQUGScRw5pLHQx72AUysKYD8JtU/DB08TSVN5Qw7Bae0yxf03zTBkj5"
        "inFKZSbLrKRE71AR4AVw8ReZwOxsOHY4yx2risdKGZa0TV7LXY6HpHoEYT9PcUwTZVzILanY37RF6A4yI4ennwMgz3a5qCnv"
        "0nZ9MZDlOKo3ld8DXy790PBaTjwDgD0YNzWLKkGKoAyfhzEd+gzO9Wwx/Q9IfCWxH29proXE9OQzqJ9m509BG764ErKsxF3L"
        "1YZKclwiO1mxE0XNjBiYzD4fB1Upvc2WVOnIGvih92mM1hU2SFAXkcShy3J125kVRJNBjbOGHVggx5mWhHFsgAi1/DAqpKiD"
        "XFS7LVdx4tsF4adh1ByFuHGm89kcXdiBtPkdkPX8CkKuKM9Fwfgq9ne67P9oOtvzCloaQaB7wOPxcBg6Hgn64HWERAsV8B4U"
        "+r6mMZ4wrl+PQ6tTCgmsB7nZfRRHEinQNNDJhKcNivkIRBKyCPKwOyIJS80psBK/fo5hNH4HyCeFkdWRFFufA2nCs0sw0A2m"
        "NpEddDmKrEQZEdJ+gSxfYrfQIDkxUKidOn2yeqJ/aogNNf772cJv7fgTO+y6CEvJFVKyDRKNJdO4GiW5D8KeByc/XZlL3/X2"
        "A9v7jlNDKKKtaPAaC9J4/eyc5oLnRAeJeY8oU7RV2gPyhal4FCafxqkjsERlbHVUvLmMsOEKluuslmJJgs9hMunBKE2GDXCl"
        "OmXT/a16gMW2b0XDSm5Z68EtlUuhaDwMo7LC9xa2XtgBqU0HZIb7yC/COUaSRJY99Jqm4XEoTeGdHPPbYAdIcWcCTFLPcFMt"
        "u5eoYcmOw6TJuTVC7bZlnAQRInwVUl4ED81QTRySj1D4bAF900vmx4mK+bJECb6EEcI4OkMufDP93alxfXaq2H5zteg01aax"
        "RwZXKyvAmH+JYRi92eNIEm0ySKTtBWl7wVCAsyK7lQAxDk3q4X3hm1SHqZcTvECYAVG7bYDS1nn61RYd466s/1UHPYYeirdE"
        "3qPig83XdzcYlWmhieUW+8Kg9b6KnH+UuYfeoeGWKWWZaC2hf6jobkYHDkwohkWCiyOrhWIarzwKhWXdGBVU4/vCvrRxM3UF"
        "aKgYPAqzrZxf1rLTQyoGjzwZnb3XrdpvuDHhRr8z1yu3pawDu7cLvL4obMWGR9MA7gnTsRvbhnCn9j2kvMC7Sjw221vhNsqI"
        "yhmLL03STcxP1rrn1RLXdODHPryCd7itm9/nV9OPFzO4mV30F7PpNT7MP14t5n74jEHpA0xdhZpqwAQe3NN+8NDSt8cgguZ4"
        "0B6+Gg2Hk2hY7r8P/cd4NnLoyoV4Zd1gGZodVlkP2oNTOH7/KDG8Tfw6n88uYLpYTM9/m0OAt7UCL85dB4QTtHZddjQxZtxw"
        "YLjQ8Ljl3bqxGs7MbZmviQAkDzJ5iavjZToZvVX7tP0nIjbnzTPKorMSs7K62AYv02Ty5m26x3Aar0/hLXrABaePckbqplfz"
        "GUyvpjfXmKGbCGgCNlcgN0w205IcZWk2hVsLpzIsyf+W3SH0ycy6rvsHbYxFjawNAAA="
    ,
    "17_evaluate_csv.py":
        "H4sIAAX0jGoC/61YfW/buBn/35+C02GLdKfIjtteO6Mu0G1J0UN3LZx0O8AwBNqibF0kSiPpJL4swD7EPuE+yX4PSdmy6+Su"
        "w4IAlsjnnc/Lj/rmd/21Vv15IftC3rBmY1a1fNYLgqB39jIVN7xccyPShb5Jmg1j4fuqUfWNqIQ07FPJJfvPv/7Nrri+ZsOo"
        "17taCTb5/OF8krDLRa2EZlxuWMnnoixFxv58+TdmVqpeL1dscsFuhuw79uHy6q/4MeDUhi+uRdarhOGnpeBKChUzcccXptww"
        "ri1RWdwIxpuGZbXQMeRnrFGFNJplwoiFKWrJFEyGzItPE5bXqif4YsWqOhNlwj5rATEFZNVMCdiYsT+dX3ycnPffXlydT5hc"
        "V3OhYDeshGhxI9SGLVZcLgUrZI881ezZ6Yuk13svm7WxPlVrbdiKw7BFXa4rqUeMGXFnYus5Q9jcw9mYG/IxZoPxXMhiKRGz"
        "z5dv352zMFd1xTKuV/Oaq6wfjXrMnwbTC1U0RvePHcjRv28Qi5yvS2PdpKit6jI7rWGuFuY3CT49xQNruFn1TZ3g+X/jwgpp"
        "VaKpFViqTarWMvlZ17LXmwgOmySO9jZRiUmsofaYtM2qwjBJ8We3qjDCuuJ2+4nNz6IioTipZcOVFu27Fe6fm2JxXW539Eb3"
        "bJjJwLKYM7/+Ca+9lggJ0Nhkk81WCpIMC/hvMpj98eMVG1umME3zohRpGiUwASXhf3rQlJCSpJBaKBMOYiS3Ci1rnwU+gkEU"
        "OXsQRo28TVEwBYpALhDQlVhctwYKuSykECpVeZoLbtZUWThlWf+Dj9j588HQyZmvizLrEqUosdZ5o9YLLPKyI8Jny6Ecyt1U"
        "1qriZfGLaCVsF1LaP5p2HUG9HnKQlTXPUndqoU1pxm4Ls2J1I+Q2HP5UVT5MmusyiFmg5kFE8c4dC/2pHEF355mQ1DCPnhZH"
        "XeQJebT9lRLlUvEqvUGTqRXCoJ6QDqLjwn18pa5VXta3bWyvheLaEpTaVGC1C4lTnOyieGgSUadzoU1iGQKn5GaYohFpiCmR"
        "T2GTJQq1RkW65c+44f0GdYmEzRD61CheSOQLVa0Xc/CXZKpuQt/hxtPA9rRgFiV+yTEpgdxCC85jG+KYQhFbr+LWLp8b6L2F"
        "RDoKuYAzoW2YrOJ3aSnkeDgY+HzhSsER2SSIeK1DTxCzzGwaMcY6mv+zoY9urVgRswVaNYpmXQmaBFbydOQZZ9HulEgvZMOQ"
        "cLFzGQqnxQzrdrvI3e/rMTsbvmQIumBnXU9B7h3SNPJSXpZWI4bTU0HwZlhK6JruV1doIuuMIU8szcwWWMUNZpk9q0IuMePa"
        "cWiFuRqnxMuSv+CAL5CvIpweKf0j8l0A+HJJP4cijnWgEMcVvDu/Ch4RZcvlC1GIRAL7pQZP5QIVJQYzT/FNGMU99ujfNvPy"
        "wFXiffEQuCMn1YpmdPgMeeP1/+RUL2q54CacwjOUAeZfWshM3IWUzeMrtRbHlbqgfQ2HNeoRhhmQyl2hx2fR1J//zOVQnjY2"
        "3CpPUI1ZsTApANachz9F01HMzmbbpgDCsX1oCUPkvoVM4fSwlL48kacCixk7r7UYD6IkL4FQ0Pkin0+QbvW6ZrpnILS7E0m9"
        "EeRL7E2dRV3ztz1hRxC3wn3tQIEqFjrcxMyKjwkojgfJC18npBt2hHaTvRnTdpRwTU0gRAdwBhuyFW9hGDoGFG3E/sDCjXuM"
        "Er2uwsg3iyeJB/vERn5BPHiUOH+S+MAMmXJzDXrY/h04/RrgIa1JWmu6QbzfHmTgwKROTW14GYycpHi3vuAA2gYbpumsVoXW"
        "IsNqLndJETg42hE1px4b5Bz9Lm1qXRg0Gk1cTYdrC7lTarTYtag5hCv91prnEbVQ56Rtnj/WUnRE5I3a8uWOz6pu+SgQB3wP"
        "bc6gC7aogtNhtlAweauWa7qifKI3FWbCIS4YOg7sxYTx47cSgqBv35++e3/psWbiZyFvEp5lKfeCw8BCXUx/j7THXXxHwzWg"
        "B8LH9oHwN4CwBcxuwh4vyJUom3FAFlkU0rlE+O7HQq9wtMP0JJSAffS4rSB7xFYPzb+wksAo3SAsWH/a3L+vBCIKeG6Buo3h"
        "D5cff2RI8oqrTWuVWtKgg3H2jMg8YMKe3UNEUsLLLa6mPQqT46Q8qM2WKBHopGaLJ21t8AIZcrnRRlTnd4UJ82B6Ppl8nMzs"
        "6RJ3Thk2YvetlIfA685yNye2EKml2FN+H9BZENpzsOchQRmt54h6mOVbBPSbLNqakEgMxIdjt8fdufvgWSOhiCaK5KHTDAxm"
        "jfLzroUSWd6ut/0R500z1oJBR7vxdB7DdRspCFN7CWoPx16vw2AcsG/Z96+izloe+Fph/2Ts6O00PHAWBwAM5sf+A8r+VrdZ"
        "+7iaYHrWH87YB6BgAj3+ipgkbW0+BbNoZnbvIHsuTYcklxoCye0KPDKoIOnrAF7Pt20qIzB3GndBXw7Q9fZi0+mJFBZs7yLV"
        "3cux05mWZGt3n2zYp3B+dGn8h5Z9Mu9m1PZYd6Pz30iopoFpipsiW6PZ2Bmima7RRQ1u6MjyRrMFl2hDgt2uuDnBrinKEj2h"
        "lstOJKaBG0Cpn0cBIe3p1jbrL8HvA1y3i8UOqKM0NxapY6baD0HeBVp7zYAdLOVsT7kbaLzkqvo/qR4cqgY42em2P9RTv2hv"
        "WIz2dttvCNV1VqjQvWgLH+kjGGo3ra8dmtxns13X3RqoVycZileHzuGYTg1yxkPIkHpN6asXRTG+oDg8gQotkkRVjIO1yU9f"
        "BXuF86sFOzm//PzhCn0hxGAVmuYKxeSpUl9ldM3LA8buT2yxnozOBvoBbw7GnIzenA3tO0BHQnADK3+0CxefaHPgnyd4eaUf"
        "go4myN5TjBIArgpOrf7O1fFabOjYQyqy2JdSvCuXTnunK7pPKTDNtuttX4QXWHceBAdBzoP7anqyD9JOZqM3zx/63Q0LxbD+"
        "+uVRESFI9+HXyQw3WQDNb88Gg9Gbl8lZ/vD7R5QfwLqO9i4OJOUvHlUO4Lav8XurMfjVSO9idOkC69pJ5kaDj+rJfpc4mWFa"
        "+Od4z6I8sKP2gLtb5pbXLjC3kASHhlys0akQTV6UFsTgCgQ8g2HVFtnDI5nb66ETpCn17zSlbhCkKcHSNA1csjiM2vsv/3Y5"
        "uWcXAAA="
    ,
    "18_train_stacked.py":
        "H4sIAAX0jGoC/61a627bSLL+r6fo5fxY8oxMXyebMQ4XMBI7CJBMAtuzZwDDIFpkS+Kat3S3ZGuCAPsQ5wn3Sfar6iZFXezk"
        "xxoZWyKrqqurq7669Pz0l8OF0YeToj5U9VK0Kztv6tNREASj49ep1bKoU2Nl9qDyuF0JEb6vWt0sVaVqKz6Xshb//tf/C62y"
        "pjZWLzKrcsFcSkej0bXiz0bYuRLTRVkKL0uYlbGqElPdVPwSq6pV21hRNjMh61xgDf2oC6scc9XkqhxNi9J/l20LUpmbc+He"
        "mUM9PYnbh3LcfS+NrdKJMjZ+UFqa/nmlrGTCkX9Qz7Ss0qXKbKOLP5WmlzEpv7GpRlTSZnNePWuqqrD02IkQ6klmtlydj4S4"
        "vhLLEyHOxTW20VRXjYYKYZ3idwEJjTbJydERtJFPaa5aO8fXschKaUz6qIrZ3CbBRMKymcqDCAL7n6YWx78Kp9FCy1JMlcQH"
        "WORncXp0JLK51KI+oN2I26uD92+vRCJOwdLRxZD24eb2oyD1LquJyvOinoXHJ6/HpyeROPg7vw1fnY2N+sLf3+qmbRa2f+XJ"
        "/OOhcvRU1UaFx2NhilnVFHk0JpUbjUVkeaDqDLaCIdWTFWGpasFmWDaZnAioEJF2H3E2rN2HZlbAYNm1mkFxU5CgWtzpaQr3"
        "m8ix4NPlz/drL/rHxQdh2rLY0Cw0jXMhyD4oldTwTVEr+Jcwauhe5q/OdWEToZtHA41GNyRNFAakrAWs2yxabCM0slIkICeH"
        "mje5kE4UHbdopvy5LVpVIhZgCdOMWPjhUpaHlmi8Cjg0uHMnn8OpbgT0fJAzBQ1uyV5QoIWDkPnmUGA2B42uZAl3TcmgYQSF"
        "FPTJdNFac0jP0p4EkRuJiZrCFUe06KrziLEocgRykcmyXIm5Ik0Qe0XN2pfFkgMNSvx+c/HuUoRs6Fya+aSROj+MyN8dZPQr"
        "70WN3Z+fxAI7hiwrD7vYT6fkKDEQ4MfEHhyotsnmRrzGRzoJcXbCyFVUbaOtkHrWSm1U970tsoey/wYEGvF+WmnnZTER/vln"
        "fB11RPWiwkI42brtpcBCeIB/be4EmAf2qpjcv5qUqpM0BIA3FN/FtFB6k8cfRIrz0oAQeEDMAeJF3E6LfPqPHpk2ecmzpE7Z"
        "dzuG3bjZ5IGv6iIzHfn0ODUZucXo+tOnW8AF7T5MUwLaNI1i2A/+4f+Mbt5cv/98ewMypj4UgT+eYPTx09vLD8M3LqKC0duL"
        "24vBYzrwYATbx2T2GLlBaRsCBwBroZcfRb393R+cTrywRTkajXI1FSkBf0j8Y1EjCtkNBQJfZVhpkyWmpymZwO2pBNyQlUNi"
        "HPPZO4yFvrvMeLgAE7OToJB+Rf1qMSmCfKGesIajDfHHEWiFg61JLtQGUqkW8p3qnR1hD3oO46brGGh02qEQnDwYOxpkAj7H"
        "yaIo8xQw2GF6imTTeXSfGfq3fbTVzRcJzD87OnFy1FKSc6Sggb9QskmzucoeOlmqnsG7lB4utVfQJtJ07JvotDf+B4JG11fp"
        "m0/sPndBK1dkpBQZgjYPv0OyYTsgGPkUCmyQcp3pHubFrIATjoeozy8Wbat0Jo1iM0pK81mzqC19LUzaNoY/zqVJzZeySB/U"
        "6hH5aksSvX5Cdu7fiuDLorFqLYvqASia2uZB1d3jHRlWLmwDdFulcDurdN2t7aQ5gCZFXtCjW0lOIcAxdlLmtiqxxixtWjLd"
        "DiuQH4xzgFIJ1oEKZVPPAFG9bfVim31o9VRTAK2N4L7ej357d33xMf2NSo6jo9HHiz8+XNIXJHkftr4QSF0hEJJjuCoIJ504"
        "eh/JUmtw1m0MyGtM6GnGIrerViV4XtQWlQjTwmaiQP1ESUvhzFHqWSf87twz3nux9ENLQzZ0CbN1cYUF74p7POfXxdT9/d8E"
        "dcnfBGBMieNhTIPc7wnRgZpOF0+8ImpMVJEdHFnIa/P4LRDvCp6nwrs9ARraiLdgOemSjHunlpztsO+LyRA2DN5d3gbPiKl3"
        "xEDDGABTG9BXTu0otsjnWq7CaOPcBz9ZUy6q2iR308CVy1+Lb4EzPi0JeTMVeheI/NoTBJ5bHYU0YDe8k7MYOiubFnWunsIc"
        "kZ3c6gXXR/aZV/djIZ8KkxxvwOpAKC1z5xEExPXs+4Lc4TkspUKSaqc8OTvxR+eBrTaNnpbNYwdq3EbsI3ANRkdWypXSju4n"
        "cYs66tP1+3fvf0Npah+bA37bt0FUVI+pluAIQ3nQF1/4D8gscvye1bGT5mVewNOlLgsqYxGB1DsUFZeGYFhKXUj0ZoHrwrhx"
        "ufj9jTiKfxGhKyw15ZYo8NK44LPiEUogz5QSbYeYrKgiBRWKGVY4xkakFXmDLIA1iVIj32XqXCxgZi2kl0aGVDnisSHcLksI"
        "A5w41aTO5mjoKAKoTJZIOKRvmcpFBv1+/fXXVxzFVNT5gHPqHcdH+CGt3LvTmO2KmmhGiCKmsihJJu1BlhVQXWQoK7B/VLWy"
        "9pIWtVMOfklWLiwQrTBcDYx5U1JgU0gYdtXV7xsaHxygQkai9OLovXpCe5BRg+DF/emaCmqrUHCTa2ycXCXNQ0qgxh5JVT2l"
        "uoMJzgQc5+KIHpHgzxdvUZqPeffcOxfaeM8S1cJYL4/0lTMAnmtiIJ2Omz4bal7RpbhwSdHPLUh340IB3QhKH+7+XBvi5XWH"
        "4pYk08JRrXKboD0inNnX48GGTfyubHRjf68LhhSOJfrlC6ue54aVILbwrocZFy3x+7pd2BDNEAA+dMlgHA3QyJOtm9aCGJD5"
        "q4TaV4FutPt+ejLeMvSunK7PFXvtMyaFa1Wmg10m9HlXkO+Ew6P4dCzWO9+/IGn2X5fcd93UPizZnZPAd+CBJ/eAXAEzqxZl"
        "cNi0tqh4cXc0/XcTX+SyCo/VwelOLigbY5JgAr/QqzRDZjZ9cbZF6buM5M5J919j4BDX3UmAiA+6NNEVyh6VK4Rt2BUCVDN3"
        "bVx8oWcLKn4+0zcd+jzZxnDkVPp3YXBwgBIL5QlkyUVpE2osuAlBtb3baQbRs2JccwlJXHWg5FjLfP0sE53RXpYzX6+A2NCm"
        "2pg3RcwGW3ER5hKI5r7R2YY/xiQ1JMp4HVcog15460UNkhiNJ6b8zk7XnKj69nC/mACZogVgYcNJIP5HvPLm8M/2tepuSNhP"
        "V9bjQB4DBtF3ZE6Du+PDV/fobiVjFg0Jf94ZyfDkh+cgufja7+hbJOI49kvkU2rCqKGKXZPRzEJueZkc37w7IMm2LZIHKXpz"
        "8fGya2hcuqDZRS4kwf5UacINgfABbCxaMh0aZr/YXZB27QwVWgGVmHuextKQw4Rw1SiuZBtutk9OJWfTfDrmlEl/aYqUDnbk"
        "7ZF6e6Rsj5Aot463M6oLVhacfEWhHHZrRN9oEffMrYYntJwncwtH3wLvuBY96xMlh07A7hbR+aDR9EG7lJ7eCf8ONZoML92t"
        "+x3yldUbmpQSmZiJUh7ndGRLOVDgOSKrhuvuoRp67t0JOelVYS056Z5xLFyH9XJD0IbKk7VvLnl8sTXxCSWAakVAHXDz9TgJ"
        "qMSlCpxr7iRETjl7tmxnMEYX1LULiS/RxwihR9caJ1cSDU6vQjyFz7jz3NrcKW3utgvh4XCLBt7hcNA8DLk/+DgGjRLLdp2S"
        "I+CDWBM45xgS8CEMW60tAk0xsH/a9r3B+wt2843P/sH89xkdxKZcQCV9AOLw0n82E5McdP3MlC0OK43JczdsDtg72zA6D+9D"
        "h20uPX1zJTE6Tx7ftA34xITPxBXWw6O4tL6vZmQO77Y78mEPyafkM/Qln9AP8bnD6/j44H5sPTXgo5YMjIP+bAvAsFHCAz90"
        "RGLfuutxvU02oUy7LjVdMUKz7gnUMfFHGlG+6c1G8BuS5IiuilCTNQg6b8UfOO8+3CA1CeBkCFQjl4rVSinWt2vRbX0u0dSt"
        "bmzTtlTg7qowFP2D6rTIV5SfktMxX0zQgIn1cQ5third97Znj7z0Hsn5psg58aU0vk1C+MOY8HMbdpwrJgP3HKPZstk8NcAy"
        "V6j3201wPBTDetIAhE42Tt6XjTxCdomaP7vToezMHr72+E2g+mUjZjaufQDAdEfE90aTAumxQLezDhBgjAejGBk1LzLrrphC"
        "YFR0dz4Wx2sjOUo2l6d1dul2dBTF05KGbLXPJaQIOHYn9IP3bHkEjBu3pD5uWK1xt+p95Iy/uelXtOkbueQtu0vJn8WXRZE9"
        "cPpCYeoNNtjuY2HngqaF4TqS/C0qzfeQayIqG6fr8Zm7QolzJL9Qo7CYPi+ou2b9AUlE+qKsfTe0PyAXDL3Yn+BUqu3ujv/q"
        "r578+D0Xa9kiQ0dDkGRWdYZ2umH2voXoOAJUag95oUN3N9I1juqpoGh/cIG1taEdGf+1vV0rvit3l47dnrj+M9Tr580j3WQr"
        "yYP/srtVE+ExcOH4LKLGnwcNXhyXvHQFetjP/mke1d2Tike6KewGTDyjgIvhY1PH4v+wXboyppHPelSy4kkHjRNKJeK6/fMw"
        "zszSNRpS5MWUC2nra3iCC5brp0xiJvWEr0Ody/Ndyo41u3IjpuQ/q0Ou1hJKqlSwYbmQOdnJfYuyPCE1YG4eFg5LIUT8lhyE"
        "3B45hM4vSLFqWxu1VxvC5OfFUApFKvmzZ2Eo4C2QLcHyR+Lwmrf7Ag/UXXMwjPPGXlqFsup6EeUWUR5+WPe9iAmSbcRkyk3E"
        "JHnPIibDn+NyaLKxwj6YJHF+qftouD4VGv7Kj2L7Lgyur7AjZgGehgGVVkHPzI9uXP9KedwrEt0PgpFCLBFhK/6e0HQ16ro4"
        "4HE0oBq2Wl9JhfPXxjVT4uo4+drdwIYr0p2ERufx2bRvrV5otN82tYq7/6sFvScQDd22+A0Nu1YHbuyqhF6USp9vtti71+t/"
        "SwkYFpJutcySLh739+SjUTEVaUr7SFORJCJIU5rYpGngTOPGN6P/ADI0JLBYJAAA"
    ,
    "19_eval_by_type.py":
        "H4sIAAX0jGoC/9Va/3PbNpb/XX8Fjpk7kYlEy06TNt4qO27qdLubTXKxe9M7WcuBSFDmmiJZAJKtRv7f7/MAkCIl2c12Z/bm"
        "PJmIBB4e3nt4Xz4A+OTfjpZKHs2y4kgUK1at9XVZPO95ntc7fhWJFc+j2TrS60qE1Zox/yxZZUpIlmmxYM9P2cWHj1+xI/bp"
        "w0nQ630nBb9RTKyEXDMau+Q6KwtWLjWbrdnZ5eXZm7+wy//+eM78i/98l7GVYj9fXAQsK5QWPGFlyvS16C2LJEtTIUWhM65F"
        "wrjWPL4ZrtRwJopsXrBVJm4HjBcJq2SZLGOhaCDEeMPipVzhlQTrxeVikWn0lWGvdwklWKYYVwoswBUiKS2XsV5KnjO5zAXz"
        "lRAszokkXUd22uCUhGSZkY1VfJ2XPOnFXMoM8/zp8q/vhprPYQToXejhNaTKYaEj9ne+4iqWWaVPm4nEgBnNwS3TzDHpoYmp"
        "daH53YB5RR5lhQYrj6WltJPmPCuGOS/mSz4XTJa3CoQluqTHzM8tViVklzCe0YPUrKo8g5a6NCzef/j017N3P16cf8+0uNNs"
        "ODTNii+EbaC3RZmIXDEYAfY6y1XJpKhKqa11y1m6VLFZECxcURbDVsuM1j4pbwvm87aPHI8CUqNHDLSEGlkxH+blnKV8kUFA"
        "ZZfxVoJatRYQAnxY6mqp1WmP1WIcdRzy7wq+Zf4qIYfUxBKhRUw+N2BvP34asLOf3gzYx09D/La4yDLeenUxZ+6v5T1+CR/m"
        "ec6eEW9GlPDvny7OfoDrprJcsISr61nJZXIE/3Bxw+xqq6P90DERlS1ofsblvOJSifqd9Kifqyy+yZse2TypteqZeSuur/Ns"
        "xlz7R7z2aqJiuUCQcqxN1fCDcTn5PKuSXu/Thw+XbGwG+VGUZrmIoiCEMPA299PDTCFNEiIohdT+aEDO65uhR8xzKnpBYOWB"
        "ngr2jqRQmdK8iEUUX4v4phZQFPOsEEJGMo1SwSkEFGz9BP7zCz9l51+NTiyj2TLLkzZVtDqpmWzDtM3D/O0zIm+OilIueJ79"
        "KmoWTUNkvH3/r8PIclJYCi6LcCG0zGJVsyL34cs4UnFJ8Uyvxm3gyeQ1cxFVUsSZMYuh6fWeIN6Gxo3q7JLFNjcOf8dfD/ko"
        "+nSOlZQiRI6rsJB+j5SQnv/tlXp69Ef859ul2mSL+Uat5ptZmaw3WSoR8ZusQGRt+IYvk6zcrLJElBvEDs9ytVlw+ctSiOBq"
        "5jmem6tZWUz48NfpM/AdN83bBIfm06aZJCDqs+H/TN3vaPhq+nTyt9fTp68Db2AJRfjjD8hK52/OLs4RXMiBPz6o1NVsWcBa"
        "VzOIokSOGDePiSwr82B91TwuqwQJyXaDkh5beqhciAryXfl4ueWZRm4ytHdVFC8SdS3y3LzHZbEilo4UPiN5rKmmtRkOh5uj"
        "q6dkH3C5Us+uEmMhGpU8a5qf9id/60+f9l1X33QkQmLAbN0RLytS8lNtXedaLDiE+QOxu33Ibu/fPWS1q5lPBtpYM2ys3TbG"
        "kJvZuoIbbqjs5huk7mQjl8XGmnEj7kS8xIgE+WRTSycF4q7YqOvydjPPVmKTCh1fb3JE/cZYdRPnpRIbqrnII+Q+gz1he4lI"
        "d+urTwFpKmTAhq/p99RMiYxpSuWRKb9HrKmLeDY1b8AQP1RXXGwrqnbgFVKuJQ4aZunGvZkrMJ2Ziu6UAsWsLHPfRlSoEO/x"
        "ta+Dhkb9ktc0zkP3iNKaFxWyotRumNXCLhmZjnmYxGsNIdatIeCwP4Qs4B2YZneKJ4wwSIMyWFIusgJxgPIqwnnIvkUWYErG"
        "4zsYTUhZynEYhtYl8PD6YVlVhVzGyQhqufCPDSaJgdhgXYhEssdhpniO4uMHjT7UpioeC39rJeOpW+sZ2pr7t2O24Hf+yYDl"
        "oqDOp2wUjl4E+xbZwqNeu9mCIedhhNAiC2Z8x+I209esrMC8rmS2/0imJ2F1k8NZPTnzAiqVaWvWFIrbohwSVz8NHmeHWsEf"
        "4Ufd/yDHYo6UHa2wUKWEH8tHuIPoMHNXGAtVyjQvb+tKdoNypQxBrvQCQ01DaCcOt1bcFYmoo5lQOjQDPDvJ6iSKy5xCipKC"
        "XyUhJZYoVqtmPFIzP0J1JKyRwPSRgYSo9CGoHJudv5BymA/Gy0WhxhMv5zORe9MgdE1B2w1kOjAmHpApBkarQS2X8w1kXkRG"
        "HokihjImIQzI9yI43vhkNHL+AmRO2aMKYfFS+Y5gwBIq4mO0wwmfnzjrIiSygY0KgTiATbTlPDl1A6ctT6Z5wRuC+PFWZUw4"
        "yaZoN90IF/OLuDg++ZrB6IIdtzUFuVPIQIwIYNXMiIB/zAhODEOJuSa76TGwew4T30QztTnA5BVyrST8Hkv4lkCEPzkAyw5w"
        "sCry+Xxv/CFs6GM1vB/OL70H+BR7bNreH8KfCkU11BojCDUwuuRrPxi07O98KfVsbH3O7u1eK6PZwGEufEpDxHAudC1bVGA6"
        "FWE3i6QWOHl+tuIALgDQ+ROoCa9XGJQVibgzBXh8KZciGDgrPtR9yPeLh5hNgTXvMjU+3grxM0gtWa2fW/IBA9DPIwNdxiMX"
        "LmlUEWZIQwRjksUakLWccf/nYHI6YMd21c9tAChTpCe7cfPA+pjUQLzpoebun5MvyhnwASQI0xyVH/YNnHMZeps4O9JgcqtL"
        "5GQgsQduimnQlrWJ/y3BwHB2QeIgfASZ/fWAoVNfy/EorCvM2urKlfWWtRWt6rZWrhVCosOv2OsxsQlAQFnBR0qwFJoG4s33"
        "fUuMKA7YfzB/bR+DkGqpq4xpsUc8epj4Uc6jLrF+lPMOMR3SjNlnr/BOzRgKgHUAt/V0hSYNk3kpdaYFPVFbSm2a2nTR8V+v"
        "2YdHlApBIMtlkfiwyxGj/5+BC1h/FVCiaxpsmnsPgNLlllayYZEaFimN0C0WdcOWxX0NO0gRxJC/Ri4whQnBS/qfbDMydJ94"
        "2NJ5lH/dPKh+2u9s9YzjBGbO7shKRocGP7Ab3GXjXBecal9FSaxhC6f1rg8MwjM5XxK4/khv0k+E3X2B9dj7KOTQnZOZfeb2"
        "/C10hZXDk5Mk4o6J7w2HVHVR00TKl7ket/f5plITj6PrMk8gnDnRsGX6YK5i2DpVY+8dlecczvbm4r8spjEV1lTtOvMy7yAD"
        "u9i+wavb6fN8EdUipMvciGDxzMnX0dOgUU7OqaBBR2Mq0hLYr+cClhzaG3tAlV+PgnbboUPOIa2JgkEUm8nyRjQHmNa8Zhvv"
        "BQc5m8YktTWhQT8kC4kduO4GxICu9baXRqjPTEaEk71tk/F8jDVJwDq+Z89HD9nXZOuBoUfK/jWrfMMei+OhmrTECKYdq10V"
        "f4L1h2QC2l+WKjPnFjAHiXbqbSHQDZI88d6KHZqigyq0LDRWI6QTwQaRb+dIPTx+vjk9PlH37PPq3gt2JHgHJJqh8rvzyWcG"
        "81ADti5eXdEeBj1Ujdo7gqYEdksFbXEaLLU1TpM2fgNZNUsm08aiIKifTQkTic0SB8oUjXan28YrOm6CbF3XOeuYlKnrqGgt"
        "C9Ik7Gjy9ypoFuXhaOv+fcna3d/3mgU39qRRRmfUAqMvfhtd29s3I/jEjJka+d0ZK2RuV+eW3sZwdkBw39rq0pFsKxZRfq65"
        "No995ToUHVGfvXtXm9RX17TlYIWYIyeuhNqmcNKEm5RpNDE77oHdAHdO45sT95ZWBr8sZ3bF/J0ls4W5HcRjO1Ew7TCoa9Ry"
        "ZurS6HRvvYAudVYsRafDQRaLPAVt9v0J3lH6lF/zg882Wxhqs9ZowGsThgdYYbizvfUE4Eg7utu6w6m7zBOj7tSiuw7+Cn6f"
        "F9X54qr43B/3TdK9x7NhES4rOAY8tN3Z2lNuc83nvhJzKoF9l3L6Rf/09Qvz1CAXtByP0LSbS1Pvc//tx0/o/sYMOPvpTfNs"
        "bxrs697MmNhD6vKGVCxennQdEAJZE9RBMWD/uCfCkYiPOU8qdpbiC51qYU4SO2sIll1/hYlABTssJv0u0OtPodrxaHQaHqf3"
        "/+6RQIvJLhic0u0UyUgozZWtYdfKAHzNDHg+zJZQ4RfwAjJzvGgv5/fx3seWyCC0fsGLfhCchl+l991RleSdURbifcFAyt4Q"
        "riDR7HIgpJtVtSKiH5h6CmcgNUC6M3XjpxjuPLQ4fZHgB6Z0bvkZ6ju3g2DuCVLXvtezqZKuC3Yu7uJccEB4eyTUupVjdCu3"
        "f3lX3zk8HH4fvnv708Wbs8vz74n9+w/vh60Wv3Ppp4XSTFV5poMDUarleuum9QWWPSwCBCzEuio1ZZCoEZkrRvC6akbRJGPb"
        "Zs+v0NDFtFtGtJ0F0bwVqXibeJG7441s/TfoYb+5Rmt0bh0ueOV3j1K2PKMB/dOikUsJZXYDc2wWKpFExh4+5sCWoJUWJL+l"
        "MrktfPsnhA8pNGBmlw4Ljb2lToff7J0U1nknzwpT99L9/EAhSFeTxo4oH6DcP6CDkBNpQsSr1coSSlBeMK3PzV3/1gu9YMvI"
        "2EXT3UW19tvNExpgkS8mMSxQgt5yBJBFNsoePWBhWjNPp62jW4OxDMDqYjszaGc1G5D3BWDpt2FgrUIb9EGKpg+V1Kg9IaJ9"
        "lGd2lrOULlW7LkB6pzmHp9BZlKkYPh0IUUVomXfAfGMoNJtox2IcwiyY3N8RwGKWlvXHZr7/U8TyxXBla+42ZNlv3mHnLD0h"
        "kx7GKrsVsjPioeRNnTZ7F2MqZLrq25TfT4v+1OTzgxvhdFswx4+V2BPUQkZfOIzbpS1oA4+6kjfOYQ4hKN0Yj3RqGHJxF4tK"
        "s3PzQxs85Asg6+Za/Lt356PR8QEwNbnlspiy1gw2wTN1kwGQJdiRiN2C1PrSYvg7/x4sGAuuq7zUeTbbXro0TeFSCd87m7dz"
        "/t64sFrTk/loIte9bfBlczp2NTGJjhAuR3TKPx6wEzplnStk/7F//HzAXoQvgmA7lN/R2QSGTloR3oBfd3Uw8R0AfnLy4uVz"
        "MTNx3KDhJ8mrr78evbSNtUdT++jFq5cvX+2FOPBBhOSk7U9kzqbchwoP7bG6uOkuJO38Lh+kuFKO48NZ0rAcE2gifvfMN865"
        "Ayf7Dgv18UgOOzUgKmgtSD3zZETHuwNW/3o3wyE0zm/Ho/AbrEReXXNzhNseSaflOtM5lpm8DODHfcnjP3/1cijLW3btjjN2"
        "pqSBd0YD3zPJc2g31SvBDGzdI147Ysq/j9HmYi6KBPU9Hnt5eQtsJbP5NaH4FEnSuMyrzoC5zBK/1u75QS86nu5vX7deVG8c"
        "On7kdhFbN3q40PndvcaTJD55efJyz8NQBey+eWfDcWjj9mhd+Kd30P+CrfDh2kLb4d8qL48F4n55+WfCzpjGlJta6d0Y7DtZ"
        "+/XW/F8Tg25WoFIlFrMcrnrwNPX/TyQi14eaxkY5X5sLwfaNQFSZW8oapbvPDj3z3P300OtwVHwl8Os7FkjNVTY+fjEKDpx/"
        "dD5XpK8n4axMl6i2bvRBHEDT07Sm+lu6kHBKGwL8aOrhOX0jsnvwYuZ1BX9bL138Y5Hp4P8PtvDT7owkJJq6+tsrpkNm2fuu"
        "06sHhObLUHs7bXYj9FGS8p1GA0ZXnoUenwRBr4tL/nzx4X3LMFYNa577h07ve0hpkbnmjSKzb48iuoyJIs9awt7M9P4XFzFY"
        "+TUtAAA="
    ,
    "20_significance_holdout.py":
        "H4sIAAX0jGoC/61ZbXPbxhH+zl+xRSY14EAQRcXxiBNmRrXpVB3HciWnzZTh3ODlIKECAfQOlMwq9G/vs3cAAVKkOn3RjMfk"
        "3e2ze3v78tzxq98dL7U6jrLiWBb3VK3q27I4HTiOMxgNhc5uiizN4rCIpbgt86Rc1kG1InLPk/tMS0VZLRc0GlMsi1qFOV39"
        "mY7p+vLjqTcYXMkjtSw0LcpE5kf3+sh8oD4olQXVt5JOz747UuUDnb/9y/Tq+vzq4vw9sb4jKCQVYoka1LehXazDeokxmVCc"
        "Sx6TuiZd5Vnt0wNWSpL3Uq2oVVdrCmt6d0JfJjQMzs7OKCySwU/xB7kIFd2GmM4XJTCKkqryQaqAba9KVWuf0lKRDONbqsJM"
        "jQdER9RKys9hXBNcVy4ybN2Y4calUhLDLKcXYZ5TkmkMJmFRU1wui1p7BoV30p+Sea4pgvdinlpomd9LbVZO372bvvlE1xd/"
        "m17zDjNYw9JhcwahvoMvWGGYl8WNzhJJ1dF9mC+lZov57yWVSaLZlVlJ0XG8GVaZvoMdaQq/8YFkBUmlgMUehrasvqWQorKs"
        "NQ64orNXX9Obi434m/JWFi803SAmfouO3eib2IPJw+DVb55vrJSAhj909k9YpcoKBtdl60OzP/YsNtDpeHPRxsXbC+z9avrh"
        "zZQNe/fxio+OElkDMsMaNnIw+Pn6/McpuakqF5SE+jYqQ5Uce2Nq4pl0rLKq1seHQ9pEfLbgQ6e/67JoP1dZfJfL9pte6YHR"
        "UiEk8yyiZvwjvg7aRcVygRRBVBXVBgVWc5hpqhLE1uXlJ5oYIVeINMulEF5QhTiAuvlvAE0BKwmyAkdcu0Of4BvXiB6T02zI"
        "8Txrj7wPNfwhlNSZrs3e4lsZ37UGyuImK6RUQqUilZw/UuP8vkLI/yMc0/Tb4cgCRcssT/qrxP2oBYEBy5hTL+9jmL+nQLX8"
        "XIuiVEgBPvnWN+2A4Hl6+rcFZJF0nMGfGxvCWg8Gf7AuHA2Hw8H1dPoWn78dDVRxgw9FFSj4u1wEiUzDZV4LjLu8CjVpgDHK"
        "yzARpjpo17MZYuIcwVlsXGznj1U6Cqq73PHJUZHj8RmmbVIRqRQKbYwEjOqm3vNwC1mHz+Dx9H+IWNyocCHukRClgmPVM+hY"
        "tB+8ObFClyrNUYYbZ99JFWqzINf1AqJmILCKg86LuybxahGhHAZGwLFK7kciLlHjJpQjRt0qCZQEQqzvN/JJWIfHlZKcBAlc"
        "L1AOsgIhGGBVA7PzFySoKai6+XJR6MnMycNI5s7cC5ohK6QkwhXlIvWNi312hW925bd2NbGBcpwViHAUQ2zG5TCFTPhZ5LKY"
        "IN6aeAmVsqEGj5fabRb4lNSrSk4wnhX16ajxLqpp5qOuc21FdZBctQzybNwIzr3ulFgvsGGIG3dbhsJZNse4mc5S+//3EzoZ"
        "vSY4XdJJf6dY3myIO4wUaERGIzrac05ozDAroWu2na9u7ZnN1LwTs2ZultvCwKGVBG9xhO8QkdKd7akXexDsFsObmyfy+4qW"
        "i9Nwfpx+cg7gFE9g+tEfIJ4KDcGFdYYX1GgVKly5nt/zfxNLqWNz6zFbO/YQWRsQbqSLM3MZ8EbWrW2igDot0E9cz/Mae36x"
        "5sRlEYe1O8M2EfUaQlmRyM8uB+/kk1pKz2+8eGh6X+wXh8DmPoWfMz056Yz4BUvtsnZ/zZGD4mR5LgxdmAybdElFBRmVBkjG"
        "JItrgc4dhe4v3mzs04k99alNAFTk+M6d7ebNgfMxpYGx+UOL7k45FlVUarYgSPOwRjFyvSa4zHpbOLesgXK7F9HYwGb7jYq5"
        "17d1k//dAt8gN0myiAumI4KrrLCMpWVyAonSfox84kPmIfN/1OQLyEO75CVSTzKXMdTUBJce0wMKMfhr3XBSld3c1sxxDLME"
        "8Q2YfjBShK2idPDe9HLR2UC/py8bKzzPdMrQ4vgQelAgfgYg3gb40kd4CmDkGMAg2QwSzEqBEtE3FA8Gtitvc91xC8V8u3jK"
        "cI0PDPs14qhWLSoYeFfp+FxPgqH5ziVsewadCCljGn5gFDPBdvkIDJbPJBO0yQSuZ+1kjissx52Qa8i0Z2ol/UBDWyUtqpMV"
        "qWOmoidTRVg0rYbpagtklXq9vRgZ2GDdzjxYC64+YaRdI2losNd6cB+9NUz7iNtBn4K7lhCjUksdLqpccoQ0jazg1EFubI7V"
        "DgNIMHHojnszHPWHIzPMukx5n2/6k+hKG3OrXjvKks9cClBrEFbyRirNZBStruiakwEMwgr0JHGNLTOIzYMFcsDl+4AxpD/m"
        "bRmCYLVl2Aw0laL06Tazk7g08AUTTNmu8Gk2Cl75dPY6eDVvPNyk+OPGKKTkAkQiA5t3xpQ6jzZ113SvyX6O1k5XV51IhMKk"
        "gYiEyQtIRb35GPNmHPNmHebj3rwNC5sHmGqitJuvbI0VJpWwQCFVEhd16NtedXe6+wkOsRbDYPiK7UA9cSv6nvh7f3kX8bCq"
        "RBEVMZa7Frub9Ol0L3va+uPQBnHRaVbgWt+T9mys8wWkN9g3ow3/zbbagZ3dmYBneSn4IDfLbeqZyNkJmjZengcSZ6/iDGiz"
        "PlxeWqktHbeZGZxbrHVT/zk5BVqMymKD5658XNrwLzJcCcN8mVUTTuhNyf94+N66faEOGwyKZP0gUTDrh5IaFt0W/lWTBtom"
        "wmo73Zuv/Bryf0zbFXYZYpcR14iVSU/etvlAP0x4x+yC/vdBL1iaTaGmO2mlnPFWgBWSa+FqZUr+YDfM5I3pTt5OQ+goGJK9"
        "WMqtCb6oNyUmDGeA6NWXKOoPdDvc7iqmppd6Y9fJrl2Y/F/tAsS2Xb2B/iFuSh6+bt1RtkJ4u/ZhqU8ofLvZYKSfl+JKacXm"
        "LeXBzaq9/VaKKYMzceglvR55/bHnXgGPaO9jXv8RD+zU8fbqsA0gtfx4/01QomAeNwoFf+ldAzlfknRz30Pi8M3LhQ4m9cI8"
        "wriNFqs6dX4t/ti8KY7pkdMqSb21aa7kPvIS14aFZ4MAcyCiYJZsUM/ZaCXd4mG3OJIFvOFt7/c9bskZbiU23UGpuEXwQBAE"
        "Tsu2D1/IOP/7rxUber5NY7Gsu+exW5hzwyt1ae7a3r+59bVukgkrfHSUqcxGi8Mr8a3V5hiuLRMMseJ1w34sG4TsHbqPe8/V"
        "wtCz3rGwt1b2WnAH1Vy5jMaA35CxtXXbx/Uyr60dHAUcQWNydsOA3NOz78zZ+fwQivqGVh/mnvPkpuRw++ezaE4cznA2NVtA"
        "uWlnhcYqrqGYZaLJOvc/9Drr3ah6fDF5YaJ6/WvxcXp19NPl2+l7ml5dXV7Rm8ufP3y67q9xuleBO3bCjN3dOrrn4HlXf9Dt"
        "dMPrN4x+djdvQ6+jzY1JxOdwMtRryzD15JERxqfJ+riLewrjGJfzeDV57CDb0vWSTobDcTBK11/33l2as8HN+PFuLSy2w48S"
        "DP+sV84vrv56cT3ld9/pT+dXyIP+q/Ye9zAHNhzV3TiEH7XA2n3aGjJuM4M9N3rzfizNHEZ7yLQ0xvb6Jzc/E4isrfesxxfO"
        "w1fCWTjf3AlnEV+2AdPz0oz7YY+XsNK9DIPD34DZTwxlWukW1ua1+79FTDY1+IBP2valnkbSrwViSc1edFz6xXzt7Is46l0D"
        "QVgnLPWEUkOY3N17Ky6jzk7Spk5sAJ5w7gZg+97qHTCoItekrtcBT8xutqg4Q+6zoMfEjS17mPlhX2z/2tJTvYewH0bp/bqy"
        "vYGWXh8W3bladqI7zNns/unm7c88+wRMAB7Wy7/RGIUNwkbvVko8E0Sb33YMCiNY+T1pYFF6pKplD8r+eueYn0r2sRf+icdp"
        "BYMHhS5gXzh5IkhAHrTbJIpP/GyGGBh53mA3Nf50ffmBWLxmVl+CVgBvfYjxDEAxhXkmFMLQZiGYhQnRcGdLyQb/AlQqEz2U"
        "HQAA"
    ,
    "21_modsec_holdout.py":
        "H4sIAAX0jGoC/6VYbW/bRhL+LkD/YY7B1WQiUpKdVwXqQXXkJhcnciUFuSINhJW4lBhTJE0u7biG+9vvmeWrZPtyQBNAJndn"
        "np33neGjf3SzNOku/bArw0uKr9UmCo/aLcMw2q3D/mIbualcLTZR4EaZcuJrInPkXvqpTMhXcktPBzSbnPWpS9Pfjvh3cmS1"
        "W+3W6bv3Y/tkMrX5gT5E7kyussRX17SKtrFI/DQKHSacbyRJkQQ+EJcilYEfSvL8RLr012HnxdERrQIpQkrkRSZTlZLaJFG2"
        "3uxgitBtt3Jg8PmgOjmbklgLP0wVOHBEmMrtMpAH+VYUkqA3705OxtPxxzn1X720k+iKljL01yGlUjksmVDkpwCnKHFlYkee"
        "vRXr0FeZKxtqdCiMFOBWUaiSKAggQRTKQjnwp6vEj5XWKdWyjP8zOp7TbPRhTEevnuuD2cA2LMxHkwlxSCglVuewKL/kcsGw"
        "9yn/hCafR7MzOp7OAC929SUtQJRooTqURiQvZXJNYbZdyqTd8kNNXmvDj5DTS6It+a4Mlb8SAflhnKlU69Q8OpEqS0KYiBBB"
        "ArCuXPmpDxTzsNejyXu6TOlp74hOomTpu4CztAy+oo1IYbd2K06ipVj6AcNpQQc0+nTMYmehKz1Eg8vu5YVExlGi+B2sWRBQ"
        "IiB7wkqH5Ilz6WoBx2K1oVhcB5HQbCmUoF8m87fMJwhhlFzbqUr8cE2/jucaXe+cTWZzWkbudUevAbvdKsIOVslChBXojGUQ"
        "rXCWQb5H0tcSaOG+yRULZ9vaE4i8GAspANWm3QKPUDBMWmHXPlfyuw60ksZGkEUpDK/VOZsiRn/79G72bj6eDdotIpfPTyiU"
        "6ipKzmmVSKEk5alqY5V++qmkYb9GqaQsJttlXvNNvvFGpucqimmbQbmlpCQLQxjktU4eOLPagC/PRvO3js7qT7PRr2MydXC4"
        "It0sI5G4XUtLldeOItrT7r3Fw7Y3UEwrGugn22aX0svey15RdPytXhHJGgGZymrhG2Kzekmv0+o5S4LAXzoySaJkf3EXolgs"
        "XNpuaTVixBBWqaA5E+ytiiWGs4R2e+zy8nQymdNQU5mLhecHcrGw+BjEWPGHydqtRzoIkIxnp4d5XOe5f3z6DgUnLxnfsqI4"
        "FcHq0Jm2oZ0LilKFBBBZoBjuE0quPVpzLCN0YwRchux+1T/q9/pk5lbniF6hkoaqQ0+eWVWkiaXOgdFqJWMNtpHC5cTRQK8O"
        "e0dIVvPJoUX2z2CKtiK4phf085Ce8UpeMcpMQKAWGYDw8KAYAza0YALmwI8IMkQm9JrJ0GXhUEfzOrFE0eM7JBckBU8UgDTN"
        "k+4RfR6dQP1vmbuWWxY98vQRZ6PfTyejNx262vhIcpx0VRa8RgXjmr3kiGu3fplOPs/G08Xb8ejNGA4Z0g3HKpFR29MYkGl8"
        "iP70g0B0nzmwxGc/dCEf8c3Qc3qvCQvPn76m78+fWjSK40B+lsv3vuo+O3rhHD0nI8fc+2eY79/OP5x2KPDPJf0qV+eRRcco"
        "4FvZ7R8+dXr8n2bCg9gFkmF1CvFyX2nRuDx0N2obdASO9vMa0f3OK0++769ug9cXw57zqnNXJsPH7SW74tL3OvnjlVzGncfd"
        "x5rl5f7h9qkI1xnoIIUhQ/vTrCPDHN0A5W0e6AhRWkSxDE0ESIeUv5UwfV4SiBAfdN6hS5RD2vOFw/1DapaU/A8IjnDdRR4V"
        "JjNa+a5Krht0V6i6e9ns4PWOFMNSGk7gZLBrkfzuosRZS7WKXGkWR8nvrPxOWXHezudnY35iILkjsQaRDiPs8I/1H/hkUNI+"
        "QtJfiAH9cjru9fp3MOx+bVC+FKUJERbcEg0It1Wl1IA8pJlCKHNs6pRdRlEwuM9QD9ioBL5rqDtSzZNM/h92GdzLWBv7EcrK"
        "NTE59tMY9yAuLLR1eVFEtusLP/57FjwRAVf70oioee4CzjUbNixq1A8t6oeqUAn6YsMzbhjltvuvi+FN83JxLrJISbPARXMj"
        "PDk8OLBui/wrRFuUdm+6Y5r/5eUOrIHS7w4NdCRIRKodsqMOLvO/qY8rlMDOjg54kaHOgRvjAtleoN5aTrFslcpc1Kz7WrBU"
        "6EWNrtHRhwz5p9aLmyujxmnmuXGMzhmF2J5fxxLcxk5Fs6+urmzUka1dienWQA3r7tSf2m5bDAFVlRExFCh7C2eUrDO+Xs74"
        "LSm1FLEWThSbppF3LaxWfh0PjaqBMR7m4f4BPAo6DX2+kUtubnYeZlull42T4FZT9xxdMtieXb5Uu0VHteAXhxmsh/FA9wBe"
        "3k6n3b02jRutGjBZp2yxOI8URk5NbVze1S7n5NgoFQ+63Rvedtgut4P8mU8oUyFGxw2RhgY9phelBYrF+0dNm/Tdyd639SV6"
        "/xxpWD88wDNQGiERmuMB5alsVGo8QtZiqrseYAaIA4QLup9yEszHBN0Mf4TZ0BAXzU8HzUzexosklGlaQqF/loGnuyTJrMXw"
        "1RxVGs0Sj1TMh1GCO8K87rN4zWuxVOGPkGg8nU6mAxDvFVPdy9aK7bJC+Rm0V1WxxSyaqsG9hD8eL/43287E0SRFz+7I774y"
        "D3fdRWX5xzmrjcDM6vwR1s5xPURY7KLcCHeBWDd1XOHBKve/GIFYysD4CsLGmyNSzj0Th9wJhXL0QiygpJiuZ90Sd6S7nZxn"
        "mDfMgf0vBxr1AGegqlpOmm1NC0zVlL7HeC9fr+YrBvqmomVTPaQvX+vmye+QyQ0gukgGsriRkhjeJQJKmn/6MR+iW0Tja6ep"
        "Pu6QfjOI1gDeuRE7HNwa22o6tCSrbpr76fx0UQtsrlk9zPkWJ4UZl281eUHroLID3Ky5m5DIGvonPetpW+11a7XviG78227t"
        "Nz3aOI6DKucFWboZcufRCJ8v1cDOAVI8l9tCnedRsxNGcHHhAvjp3v1esa/YXiwZgBoHFX4u3BgWNCwy6HAhg6/cLAFw0kMA"
        "qgkAOgbw4lpFyVc9JOlSdQJsWT1LdEX0MQpleWLCNbsk13gFuX7eI48TuSrhTfw+4aOZoX7RHOg0yjuZ6VkoxEK16vU5Tg5R"
        "mTXgY00GRP32hN+sHLWx0AAuodN6gtOTCt+JqeQ5ydi/FMksv7Dlqf3gVzHL6DQQQ2CVtkZ8Wc29PNvTBgUbGJlm5Pnc2GBT"
        "7vCqGLsqBq3HdF7IT7zm8ZriNRU2GWBBueIWaMGpju0kypA6WO7QU20rtrGf6ouD/VV7rgkDd1e8eC55OQp+yMu+0J/zKgRe"
        "YQgWvl/j9vVa01LZCrv7eFhd4ERZTNz1XS63/Nmp+oqo06D6llh+X9UOe/3AsM1T632fDV2JXlUXS1zqjOOJZcK9JYqRsSPy"
        "1k9Tid6p8jGn9F+7ef21LLUOt67m4TPLUVHgo1LuQHk8iCxEIJItA3Fq76b3j3Fuy5ivL/+bg+GB7mxu/wg/TN7Mxsefpu/m"
        "v+98/43y7mL/s3KT2bhzG1aRxl2K5AaJI+sxpofewOl7t/8kMm9UXBRdDvlb6y6KVhpdb1qhcIztoXglCufHfShVyOWmBAqv"
        "DJyn3j2Cn/R3IoCP7FekObH+UqdbBp4KnCuEm1yw7U3ucx0328YpJoe0g8zlr93DQ8vaNzzRv2eTj8S8GFNIRTiohLx9sPnU"
        "XxE9WixCsZWLBd8axmLB08hiYRS3Wz6btFv/BUXs1MeHGQAA"
    ,
    "22_multirun_variance.py":
        "H4sIAAX0jGoC/81ZW2/cNhZ+16/gqti1tB3LHjtNs4OdAkFjB10kTuC43QKzA4KWqJFq3SJybM8a/u/7HZKakeaStA8LdBA4"
        "EnnO4blfqG/+crJU7cltXp3I6p41K53V1bnn+753dsbLZaHzdlnxe9Hmoopl1KwYC14n97mSLcu1LNnLCbv+cBZ63qe8WhSS"
        "LSslZSITBjzFRCtZVWuWyFRWKr8FgFAszStRsGpZ3spWRewmyxVTcZs3mhGy8m4AXLeXRf0wYlfL8uOKiSphHw13I9ZK3Yoc"
        "1HUm2UNWg6jSIr5jV0znpVTsIdcZE+zNT5eXF9cXVzceUWVSxBlxNWLyXhRLoQEp72W7okVWV4bc+T9eHrf1AwPV5Lhe6pE5"
        "uZVN3WrllVJU7NuTY/bpDUvrFmJpGescuC3Ijdjlx2v8GRuc1z//GHneBR2gM+iGyUJJBkkzWSRQwaNMJuZIJUrJinox2rwp"
        "qRSoHi/aetmAddUUucbpkD/pgYk2znLiYNnKiH2oipXdI2nL+l6qEVO1XWpaKbAohQKs8owCiamqrkiItsSrKkfWWEILZkxO"
        "okGInz+9fnvBgrStS2yq7LYWbXISTjzmPMZZT50ccprjY+MO53iAKuNMsZd/GPm7HjIe9UN9XIiVbD3v321OxnRWOtkl8puC"
        "HOxNDaCrDzcMqmkfCAdaSmCWE5CDNzPrVl5eQVFlDcfotJflTWN0Cmgy4bLS9TLOZBKZUMlLOhjmWDSiVbJ7p1O751p1Ty2c"
        "oy67N7VSntFrI3RW5LfMrX/Eq9cBIVKgBgRO1XRLDahgAf+axPNqFSF88xZSKqkRbAIqCPybS/7jx4/8/U9X/N2Ht/zdxS8X"
        "7/wR8899xOv1B2hiag4KOE/zQnIeRhBAVtr954G7iBiLoBbZ6uAUDqXbwKCeMN8Zzg9DKwPiityWw8NyRCQ0z6EkBKZjWlaL"
        "vJKy5W3KUynIbRVj7Bt43WcxYRcvTs8sodtlXiR9KH5/1hEBA0vyeFH0aZjfLiEtHzWv6rYURf5f2ZFYL3DaZ7u/AaFO43Av"
        "aIXDY+WqqTVHAuDrOIIhOEF8iZBlSd0VUrQVDKZkSQnR0b82fnFZQx79YyGQANIczj3AcQJzcN0Kk3giI4EjcZPmSfoL8kHd"
        "Qrgt3AKqFy13PmwR3tULGCqPr+WitSlniIOM0+ax6sDTMVcxGEQKrmMulrF99bz3r399d3EFbzo7PfWu3l6/fs/p7Rxv3jWc"
        "8MO7T3id+Y1YFbVIeCEr8kO4WFs3K3qEi3PVyDiHWeMMUdQtJvkih4uNvJ5WzcYSIdnGQkmChGVEyeMagUmvueJNrcxjJhRX"
        "n4uc38nVQ90mW5Ro+1GpzS7zPy9rLTe04roswSjX9Z2suuUdGlosdY0kvuKIF+TTqjvbUruV8BZJjHyBj+4kkYKAReyoZLos"
        "cMaC1w2pbgcVdQyIGVyoAGqPhaKuFnCotW6RFrfQ+1rnLaX8jRLs69zzPOQUBq6pcHNZxXCigDxvxErxSOacWhcwNYEhE7aw"
        "d9VE8MJaBQ5mxBK9auQU63mlz9E0ECzV0XzEYoa0K2FYSaXUEJ9NHOLckaUfHQ3a4CWIw/UqDpzlc6yb7Ty1//9zysZn39vC"
        "OzawaB2WbUXgTibkmFLAyR/NiSiY9zJ2pykNek0SvUE5vIR7yWC2J/cEOjQiaOLf0JhbtsRiB31f/gugQ//txY1/gMwtPNwS"
        "iusqFjqYiUUEPKl5jtL9GCQIoelNu5QhJecDW3P0MY+5mo4t0WqHN4gdIadUCkyUVhdhpFHnW7EKwoHH9H5xXSzLSk1nqV8t"
        "KAKf8mffWpTkAL2FDGBAQ34BvroEVuFMxdFfBWEYOkmdbXqSkuwzlz8gQbX4unTWqrZ+FEqXATVDaJkeam6ahemlgDc4C7sS"
        "Qb1mil6zS3N3cEG1DyAyOx2YoWfhUAPIWcx2RC+5KS9obt8WdVvrn6vc6JWYmdIfKzH8dM3YxsPLNaVP8vMScQ1iwWxgAXt0"
        "9FPVQIMqEwiqwAbgKNwylgO9QK+dIHwXwfjs1Yidn1HkqjtOAeq8Zx/au08374OXL0bOOFwZjmKpDM6I+KxkwXsiT+l5P7E3"
        "MBiZ/DQ6H7GNKg4fTFz+3054QwU4GMN1UEjvTas79VW+KOs88XsozjspifzJbfRnUVWJAC4bNJRB3WAgM4xYZa3fVfQ6EWUw"
        "lsfnO8mlqJWa+hhJRbviMeqHWvcJW5CuPZnOLHX3GmHyCijBTH20KP5WeildinDAwWrEGhqp2ulp9J3LCytbvISy6W9lCaC9"
        "S7AR9HaakP0wJeQQS1TbAhQ2C60bwOItCAKLiFoUsr+xYGUfw0gtS2Q/m2eqHeDTw8BfpHw6BNZfpLwF7FT0tFazvx5wqRWQ"
        "/gSN37JKAgh3wujvt2AddedFSLlsvWAr7hUa5Y3F/LRp1/ipwU8JXPfwu4X9+OMNOppIHXTdqDEhZAsNoR4GWX+IMmhbDd4O"
        "km6Ao+ESflrhKa3oidZSWtO0pisL/9z1RZWkvsqVGjui9krOyFwmjGjE50nqXMzVkF7xQWHR6dfLkjWUmRUiOjDYlBM45v4N"
        "nW420AoON23s3IoY7S1KLzX+3N0/BKE9TkOGewGRqBMxY07kILi7oeDmhiKwIkKjZ6EZfsyqvetgQmPZkeP6keqlbmc+72YC"
        "6jn8OXqOAiNJYHm7Fw7yXnwFcqVbR7AQt7Iw29zMzR3AvXB09gDYw2QMiK0ZKhDoeFeUwXzTID/c+tSIUK9juptpcDZ0oN0f"
        "NbFdtzd1wxF5w4MdX1xH0rEQpdCj1ZDj61cjWq9RNZu2U7UARrQNgNVaD6BNsb9/vgwqjvccmHWrphjfbEefyEZneP2SXK4F"
        "JFL8QeaLTCNri4LG/sT/OqL1SK40EsvUhk7Ff6tv1fR43LFtlPEruR/s69Rxod1sYe78gtn2VNLvo42mXP6/MFr6XXhWgQ6P"
        "ukggHm4pN2CG3QvHLoVMkSemWnK6UpsGFxREcMSwyxJTlyyGyroVOs64gvtNTSNwL9vbGo5y6jSA0kWi7A7vztVp37ACYW2H"
        "zp3M0ChlyjzWiOL6VgRwnXA2GbHx/GsGM/I5ZCvIhq0oLWjixPnz0ApoGf1mfdPaXa9216pmO3PBPRvexwxMQQgzfyvi5zai"
        "M+Da/S6ke0V4O/5/zQYxku2ECG8MwLaCMqcf60PZ73WhbOBBhvZQg9l+BbrJ0yAYOw7Z2WNRjrpEJ8zDjtNBLffbFCVr3e3g"
        "WEIJ6WoA/GxtEZ2tXOabgxDRQ0hiMewKYClysO5Gf+K8uwmNXreLJd1pfKS31okn0EAlCRduL/DtBS8Sq7kegPVGzN1hTs8P"
        "otjQ2Yv08iDS+tLYt82saWQ1Xc1gspcHslYmi2bqL9GSkAujLizM94szS4k5BTHqw/2DJ8Pp/Q2H/VvUL19b+643Ay1Fmm0i"
        "o1mirbrK1bTU4flTn/2dfX8a9tZS/+BnnGPGnohIRLp/Zr2vNvCNnh5S/+nIiXpEXZrBWSc/26odjR3Ac18L22zZYDAtwrqP"
        "sNeB9WKtDcqUJ5s7VtI0gBaOpMXe6QUouvbvdCkBCg+jUjTBMNdYqpRE7IUIfSXhsbofskNZ7ISAYEROLxFAfCeQ+WJFWezF"
        "GbrXfPv2Y61il7XMpwxAz/uXX2p4+WVIIpR7116dOf9THR8fmw9WT/nzSd+A9nLhST2HDCDwtbRYqszOjWsyVDrX7erI2rJr"
        "WIeG3epaNxRmPh1kdK42q2AhEk2DFjJoN8AkXkmyzSgNdTlntEkq88kg4jop6fmpnLyCWJhApk/trJzPjobDyNEcbjU+PZ18"
        "F43T57+ygdNunBdjR4ePxx2kdLzeHR/NJ9GL9PkQJQwOHSweHfCWnq1HLMsSsysUhOxLiW3C1naiW2+buCYD7W+f6Pe/7QHW"
        "dyHoHwpBf9wlth1Sxp1Aw7mV39AN5JImGceRYxgLT8/Pnjf0t6ej6ZEJ4Of/VO8vXl913z/pI9oggxjPDrY/oIZ9Cv4wN8HI"
        "R+Z7xNFkfAoKTxsTH01+GL8yS5cfr3svY3p+aZ4x5ruXYb6BxAhE/9iknVebS+bf64jEPUz3PPDiO4u8NQ+P7Fg7MtPpyE6c"
        "Wy6NbKHcdYK5MpgZD7qbG6qtyRPQ27y7GzdzajigAH4InryJvj9vjbREP6L1YHuU3faCZB+m0kmQJHU6HYdmBjeXtdgI2Q9s"
        "bD3rNDr9IuEy389T/lWWMGnsxRSPDnNjA+eis7WvzqFGatjqB283eyB3GIfaDmTUMtLmTiaZHZEG18nhVXSGNEB+fghcJWvg"
        "FwS8J/10Z5ms85UDHMwfoTreEJ18b9JWj9y4o3YgoXVUTB47SMbtrul0Fc98MjaBT/EdmS/ptnM339kTdN0qcHYaMbqsr/T0"
        "LAy3Egtj//r04YoROlpfpuuJSyeg+nyohfDgo9x8PuCcbrB8zqn75Ny3YWdbUe9/3P1DC9giAAA="
    ,
    "23_claude_evasion_probe.py":
        "H4sIAAX0jGoC/9Va3XbbuLW+11NgmM4R2UqypPxM60Q5R7GVxK1jp7bTtJV1uCARlDimSA5AStakXmvuzzp3fcJ5kn4bACnK"
        "lifTy2o5EQlu7P1h/wPUk28OCiUPplFyIJIVyzb5Ik2eNhzHafSf+rOYF4HwxYqrKE38TKZT0ck2jcYwYcPjv4wuLocXJ8NT"
        "JkXQzgVfsjRkw5P2u5NLNt0wnrDT0w/MPdJcPMZnebQS8YblchMlc5anLBCh4HkjyltM8nwhJMsXmCZFFnNNw1kY3YqAzVKZ"
        "FarDRishNyzjmzjlAZuKOF2zNVcMs4I2L4AeWBrgnHM5Fznmq0zMojCatVjCl+C0FvwmEUqxKIEwwZZpIOKmAhQVzZPDRoOx"
        "NrvCg4u37EZsFEsTpjbLaRqDJFFRvmGuEng8Gh5/GLHnnWdei6mUnZ5/bl/+7cOb81PG85zPbhQUEIAbY+9P3r0vn01FAjEs"
        "F7cAJ4WGkPC8kDxm0zhKAgBOc9WpYJxeXn1gEEhASHcYCiOpctbvdtlswSXUKqTSGD4Oj49Pzt5VANZRvrASNRAYUGmRCVb/"
        "vYA9oOKY54KpRVrEAYOlA8Gi3IjXys4XRBQpdvb2T0ftJJVLHkdKBFpgksKg7FMSzaBF2DUiqUTIghSIz86vWJjGRgvuj0Km"
        "7XUUANP3aZRo0Dmf7yzi4uqUgaeUEezhsSXfMFXIFfxmq5GkPZfwtRXwpxJQJKELozyHdVNj1FzyKCHcq3TGp0XM5UbjtY6j"
        "NKBpEcU5C2W63EKApn8oRDIDeD5VIrHPiadxwVJTCreCwf06jcaIzxZsxqHaGQdyzCV6GzXs6v3o/OJvUCqMrnJjKM6WEbkg"
        "3C7i8yRVfBoLGwINHQLLjQKcKCWnvyRZAZjKtJgvNHOoJ8mjGZwmwyTGswxxyQol4DmNT5fDdyPmauQBV4tpymVw4B0yG98A"
        "L6MsVwe/EOKfZQS4FImpBOVesu9VmnR0roiWRMZooLzO4AuxKO/URjU0HoIbR1Nmxz/itlESJcUSi0A0J1nFBTGEAfxlQaNx"
        "cQ6HGuhJru+HUSx83+tkiKMkt18NSOqQkE6UwDVytwuF59LVUw+YY5fueJ7BU65IIvxVzmF5f7YQs5sSoEjmcFUhfRn6lKsK"
        "EMJ3nsD1f+CHbPSs2zeMyJuCOpW/6pdMAKCY6Riv89Cfh4woNfg2zn4UJYtqwNep4+Fnh1Gj8YT9/M+f8Fe5/CFzaSZCThbC"
        "jzlyZwv+mot5SsFR6gHOhXvPzv53/yB2y571YCukD8VQD+DSJitRVmSX788/nR4jNQEBXDpnro2IQRU26yjxdtl1S3b4ezM6"
        "O3l3pnnZgKRcA35hzOdzBAsY0iWmhDxGZHLkgKXXOBpeji4xOG4YnbXxYcMOO03XbZvlD5ADlpyii13++TSC2nQpMUFtaNpl"
        "JYBsmc+K3NOMTJpzeuz8Amt/M7r6PBqd4Wp4dsx6Tov1WvC/H+LoIK6kOS09iTkldY4ilsbpfHMIe7IfihRR2KLLWbpcUj7K"
        "0xsIb0FdSQSPMPUN35TC4NatCkQq2fNutzug/74iHIGHXDPbCjeVI6aFMyREsAfJVGfoCtVWFg8Ahq2Ru4S219ek2XL3+f3o"
        "YsSuLj6NUP0FXBRGWqaobHF0I9gomaPOLOor+sc/WDb3VSxExlzW6zLvK5I+gtucgs3M0YtSGafsPoXEG92FwKqXp6PRR9eo"
        "1paXutwiIYfkMZQtYlQdKCOOt/99BcSns5PzMyN7ncqANWlOk+wWx1SxiryyL9dKrbvmG7gmYrRtyzVgHCCig4CQ7zgmuoE2"
        "uYDpFxA8Qbque6XrXKGs3LBNWqAkS/pG3UxQY1CUqBei+0yKZVQsUXKnJk+SQGTRVTQTHeaYBTHns9CtC/qmCLFLlYngJ2hL"
        "loavFDNyVawWFZViFPVJkpui09AEFStMQxNSEA9Z1ICgKQHKpe4F9PoyjqdK4BaesxHwdWo243SD24rbUmgnXUTW1ijX5JSA"
        "zn7HnCbFZbPXHDR7UHPNZlsFV0ZDvVNqJyJQ+xeAqXFs2wVqwrB6oWY8M81cTf+VD7nOudEKNEKNZxBR4kUXkIlEm7LIAhoA"
        "J60ePpulBRQIpYXAT+1ITf0fY0G9hhSrSKy1TJ1ZCZ0q4ERokHluGEFPcUQGhp5X6AOJlTZIxYxa7U3dgLrVgavMESTkLGud"
        "XXVIQp0ghC3NDG0WmVIZ3qKDnl8Z33nNY6q/Pe/VgR0otX6r1F6lGzLdEz6q7fsBctSp2s+HfeduiJS9K9duDfui3aDL3eT9"
        "80//X3eUmp8URk77x/X3FeQHPS2sq6JAWJ/VpfelQXaDFKTY3z//sWXa2ixGS2iSTS2XWuX9/NP/7dHfVn1bLCrbB0ZnuToW"
        "UiptgGqSrj+hMohu97unj9nroTyw2SY2q/er4TumoNm4rZj76Xcj4uhRJCOKAQJh2lRN9vNP/8TYKlIRWt2W2U2g4cLNfZMe"
        "Y5uXgHOZnWuN/oKaw5QSARL1rnHtpoD6/R2D/teT2/53L+n/7svzC/NtxnrVo6dBOVS39yJfxm1qs/NNteb3Vx9OWVkuzTN8"
        "WbC00q3r6BUHglSkN5scNSddYxVbE1xf3z6dGWXTpSgtoMcPag9qlliI27ZJNxWoKW32kLAW7Vvku1ubjZTO8whk7CymCKAb"
        "kdf87BKbtWQ+blLLeYQAOwLM5sR90W31es9bf/gDvp61et3n+O7j34vWi75XQ0ExqR3Cct5iMffYKOYmzpAhMqxqQ0kN2+6W"
        "3TOaWqsWtI579h912BEluAO0FLQJ0a6sZ4FLPVSb1/l5eiGv895gx3BBWsCtgnYsctpVVtjsOLPjrEmzm/YQAvZBQOry0kwx"
        "ah2Kh5R4aHOnE0rIo7jeZTW/7Qbfdvn5hfm+B2Qm47CtA73CcHRx+pYVkhyL9EdnGPmavNn0dfd18bbD3mPzVh0bLADUNs86"
        "yVGLW8WB7nTbWYrWFBtm2jiHUN1ONFyOTkdHVzpiUlPeNcsp1Qj4KTxuavI+ylVKcaoPC2YxNnPsyNbqDpbYreytgR1guW3q"
        "bvDIaRohTdq1oe23jRzCWE4pDj6doNZkm60Sj+Eh9lwDrkoFu0XNqxGKOmPCxoBCl5QVZBHj29gd8fleQAGYbhGRCIOHxCMt"
        "6g0Zlr9F8Uk3ebaD0B0M6ZCz3vNvsUlXpiDbNlA3Nso0J7953t0nX/eMWwCau0WQgFVNLpbWbpNy6GwGq2qvaNEZwhO7TzpZ"
        "ULnIXjI0faAL5KYti8RQ6AZgn3Cork2uoUj20ekJquKsoN2DCUndG8G57JairYv71v8MLjpp4VKiUmG7xcfRBLumKX39lvVf"
        "avVHZM1up1O2/gl7jT0adRcRe8WSvcAoZdDmiTKaoy+YriEa0qDFXrXARHeM29bL4Ite1ZQ2WiIOWdqcSrQ0/yNu+TJDE4L1"
        "kAJNS9PcCFSdbWPzEuMhezJs9wZv2v194PS2ZouOI5By+M2CNmDlRkxvhvCNQrTQzWoikDaSelkNohXT7ePg2pkhdq+d16+Q"
        "w5LXn0UMhEjGyJAtOquR36DU0pNXB5j0el9Y8VC0qQ4RoFjMEdhLahR1GUKnd1NkevNgO6dU6oPQuF5jLsQcJQF6WKH3oS7z"
        "kP2v+9+Dzm/Hw/bfJ56+vL4OPLrl7R9xOfny+9bdb7SJ0X8p8mG1T1+SOBOwkgqpBzG7YfpBi+mj2gW2CPfOTwnbpNFoIO8y"
        "6o98fQSrXO9Q89eugJqRVGc25vmBDPud7EarQk4dj4IpPGyU5x8yhIOaQ6cOcXVD75fZLUXOf4EfPf43OSZUMXzbrqAZ+wXu"
        "INrP3B4BJSqVIR1u2+OfGxQmc2oZq3yJqXqgYwR3tlq8D4mofcruHT3BMUJWfXBAUs7dLOjQ1tufqVU1FU7CD5Bd6EAtgNZ9"
        "fZrqr/odUFkO20+HMq07S+NimajB2NFnNc7E69ghQy8F9vwJjNTSiiXnmLX0WnDZt84AF4oSHvumNNrjqiW/9WORDND7WwdB"
        "VgL8JOtQv6tcS9BiQb7JxADjyJxP+1adlKdabEaxKnTfhgjQnMeHduLE25pF56IBAXFn24VCoMl/+nEUmu9XA9brf8cEnS71"
        "6osEuV2QPiS263hk6VZ4TivaOeLT0zzdCWwPe832i456YR/sInMfCUgzUMQB1jyG8d7CDYU73nPq6ObexKxLH43t0u876nSB"
        "3Hk3unLKecmDeVhOBx6SKNqwu+Mcps9TXTlcm4Qefip3CR0TNF+iO2dbVCRtP92nsLiV+lcjFKkcG2d3zOfwWiVyH1tpceuS"
        "Bw6uZCHoTUz+2KN9UJLHGE1ajN9GatDzxqv+xFg39DM6TkSw5a4MO6UF6Cycu3/1xofo/CbjroUcqxo5Wbuc4MJDFZ2EuuP7"
        "Do/l6rRJrcCg63VQw9GpJq5XcVV5javJYDswwNuofjwmvC0NYzLx7qOr4rGkaWnW1m9RXZMyHetGxHUGDgr/d8+8+tjjrw+Y"
        "3vygZ8FiFJd0PvnwHaHjPSrgkWChrFUvGKZVpvaQjnQnVcSbiNueces2NkU1gm/pE+B62djVABj9mqit5pP6MadHaUHPfz1A"
        "T/TcpIVuRYan5hB7ANrDHVfU79yIh2GF6fVDfXtoruIoy7bvgHYYpEWu+4oBc0Z/GR6Pjh0SZ9lqHI45ZXeqaTT6FRS9Ogq7"
        "/VjLFM38pjxkfwzF2+Hp5ag9PB1efHgIJb3ZwiDLdZDQRBK4X5zyZYRzaGzmbM/+MWSt6Vh/FwHG6PqRFOPIEARQFlgbG9N7"
        "WocMWI0bqz97LEs5Okq1IENP9rV87GrxyF49xoPcCFTGmxzjhnSvL+6sB5fva+HEZmcjdRYkv4b65LiuiYk2jvF1a5ZfPa1r"
        "plW2rk8rMZiZ5fr0tNKtbIDxezMtin0T654wadTCPXSuk6GVyNgh+4JC7FoI2LAyujUwvbuDnYd35Ssjp6bx0HG/1KfYJXo7"
        "KSZ03hikW4kGeikw5KUwOw5ZscAuc58oIq6/W1LeNTYo9TU6w6ur4dGfLtnV++EVMzpkrnUq7JCQJrzD3STYrifBSsFmMYe1"
        "nGOWw9j4ixw3y7hpTg77fXU3YVbEgB7aazzrPAvvdlZiVyNDTShDonkKGooRPUQXdtBzvD3i6UOE5N3NyfjwxYvJN/LuUdL1"
        "YnNo6LX3NyclKVyHTvX2rxOT3QRtjndPu9eJfQmonYxpJ7vc6tfk4V+n4JD/pynXauwh8AfaKl8JDNiXitSxweTnac51cq0F"
        "WOshmS0fu3S7UbpnlnliZz0kMzG2A8GG3UMiHYa7RNuYrZHrePRNPFryXQI6yFM6oa+VGb5r2OoFDZU7IPurB4euH/3lg1NO"
        "7KzptxKmadc/iQiKZaZcq/gWve9B/z7oe969BPjHS3oxKOmXK3Sig9gAt7tH2qJGAzb3fTrD932dXH2f+jTfd4wPmKat8S8N"
        "f3O/0iUAAA=="
    ,
    "24_polymorphic_probe.py":
        "H4sIAAX0jGoC/71a+3LbNrr/X0+BZaYjspVoybHTqWplxpuo3Z6TxBnb7e4ZRcOBREhiTJEMQUrWup7Zh9gn3CfZ3weAN12c"
        "tNs9HtsSQeDDd78Bz/50ksv0ZBpEJyJas2SbLePoecuyrNbpmZfE4XYVp8kymHlJGk+Fm2xbrffVKEuF380EXw1Yxu8E40yu"
        "eBgyKTIWz9mUS4xlGZ/dScYjny1EJFKeCbbi0bYlBT6yYIYV2674lAdrHoooY29/vr28/enq3Q3BEHy27LBsKSK2ElzmqWDL"
        "eMNmcSQDmWF6uKW3LV9kYpbFKZvxbLYUkom1SLdszdMAmxAkzGKSrwTLI1+k4TaIFgY5t06UXLFAqsmp4CH2AJnZEt+zAWgw"
        "K0TKNsuYJk7DGI8+82PsGcUZWwRr0coThfOWrfIM9LrsktUQjJicxSDkuxdfsRgg2Ty4Bwji2hZ/szgMeQLWbYhqIu7m8u3I"
        "7Ex7JqmQoBxLgoid9ZgfzOciJdbJuyCSLrsF9iuRpQp1noHdWSZSyZaYBQAtwjMGf0hWGq8AeEzzjL0fXXd/uHz705v/q1g8"
        "2w5YMCc0+SoAu4HBjOeLZcb6pydnvVYWrITsNJgjg1WCmSnhAL7kEHMIAIU0QiiDBNffEndo6zghvYiBok04KQLTtVBi0EDb"
        "ks3zSCHqDFqMdYEDWJQCUrwKpAZj34g34lXmqPdBFAaRABmrFbEGjBGpnnXy9dcnbCqyjSD+xnciknrJZhlkQiZ8Bk3JpxB9"
        "lusVGZ92WCQ2BLHD8jTsgi2xL3y9rlLertxGGb9ncgMJMvvquvvy11877FMeAzBrs4vuS2bpRVG+EiSheDrP5cwQ0B/2Gaac"
        "D887rHf/vD+kf2YT2pGU1q72B1V+sA78nIdstuQp6GjdkG75pLMxhKRYGPhCGxpLeLZkPElgySyXArryGvJPV0GkNd2WQhBZ"
        "buvnm8sfR8yep/GK+VwupzFP/RNnwIyXgA6nQZLJk2OO4q8pMRM6kMQppu3P+SjjyFW+BtqCOYwGiu9JMLsLRfGkxVw8ya0s"
        "voIVYTB1E5AuWgpXohBDzEx4j8dWMRssB90chpqU+wAyBvCb+K3W9dXVLRuqRbbnzYNQeJ5D0ME/89HC7i5t4mqNsnsdJrPU"
        "VktPmGXYYjmOxkesuYRoPag0WMyjmfDgn8iQNQIiWkCpROqlc28OLwMPJxljz+BMPvEBG531TjWgaR6Efn2Wtz4tgACBfIZB"
        "HtZhqJ99QJm4z7woTuGqg7+LAkQ54NF7tv/TANS6GY1eg1Nnp2enLS0dl1THpnGn9c775fL6p8t3tzc0p6dWr4yxyzIM+Axm"
        "X48SrVbrGfvXP/+B30bwGDA7gueGTxXwK1iJb9tEOGbu/8Nv68+XNyPv8vb28tX/Ek3jFrHEtjKeZ3EYL7ZWx7DJ6rfZ1TVr"
        "99vDdr8c3f2x5KcwsJyOAZNHpCJShHDFeg2B+fkdoiC7Gb0Zvbola00VExIu5SZOffbD9dVbNSy7XSzaAQnElt50S7MNFhb3"
        "YedtNfn4zw4Ycu7eFH7PL1YRZpfvXrObN6PRe/vcOQxvB4xUgcH3PuWIymo+wHzPXl9fvWe3l39+M6oR8hQYZVxexhclNhd6"
        "7CXcL4yx71ycmIE6JOseXCihBKuFF0MF0zgtWHOBMSbT2fCemTfDAmAFqAlFrglKGHO/hst6wfTYgeUHoXzk8A+aqjwNaK51"
        "wRnyjfmwejUogd1fnHAFsQllGvvbJjLWBY01kfHjWU6x0J3F8V0gFGoFnEnd9lYH4nIRfbsmOCMS/fftD2ndnLDxKNjbUkV+"
        "RmlFnkbMstyPcRDZMzdPgKftUJZifJH+wNAF67nnTIRwJjM3jDdq3pzSMEqeKF6WW+hEodzlmUkZmEoXkEXJGMmjyg5kHQ3p"
        "IsSFGLUtRgyl6fisIRIAah9RAnxHJkTTHAYM+o7CGI8KFY2krBCqspESp42E7zGQZ8s4wKux9YFchvUhov9f9b7TH5w+WIXQ"
        "xHkC5418GhOVvniU0+zKoAaoDUBtq/05KdTAxqmn06UnoJIvVaT8+ivYVhsHA6vxCqbJqkqI0QGGndLCb+nfd4pblGPR53mv"
        "1zvOqMKhz62H6BHf8b+OEDK34uVQvapwQpKi0zWPVKhELUa+jUgyUQ+VSuqX9ANGqiHr4mX7g2U73w9BL9VRh9h7Vq0zwF3k"
        "eSLy7XqO5CpR2jPHKWeTVI6unTmHLA4TauT5cT5FohQKVWTU7AeiQy57FV8jP9mGAvlDQDGd/fXyh66OTGwVUB0B3/+9Lk5U"
        "4SBULssUXEp44YjuxJZinix5dbchzoytq2uSnHLkFqISfYAr1qTBRT1ZHuXdeZN+MrJK8HebDgCMexP2jf4s3Yh67g8mHdZv"
        "Kk2rdfVeZQmF6+rUPUxnx7o7OzbWaRpHp8Ctpt6dA3rVOSAM8upGTMiabMqpjHSQdl8mVKNxwxFV8ei6vfT6HWKbeQ3ui9Sl"
        "bL1gEUFTD3eVjdXc3ZlTCitOKkAuanCI2AaHwFen4jyBjMnB7LBSEUBRzFuB1FDahS8MUMoA1ahMvfX7k3R+6iZ3IalCOoWr"
        "RW4/r7ZJ59hHVxcuQbXnztPgUEjzJ+DR698IMVqkfOWtVTsACXf6BHRMOgzcZPKRjNM59LHI4lF8c20kocxWWKoGXL2xW3Fx"
        "FyWa7U2FzFy1wNKboMAYMuqB2IkPe8DimVyXS32e8RNkA1QX+eC6l6U8iFCVuJhlOa1m2uP6KaQ7i8N8Fcnh2Ar5VITwtq4Z"
        "aog9nXcUYzvEgI6iBV9PjTJAFYMIpY5WfpvqFUzn99D6aHja6xkF4WkK9KPEBYtjaZsJHeZT7TDEOBT1+WmlpUFHu1uhjIzs"
        "hSCPB2bhpKarqvYeEiKFhzQbjoMJxnVpPtefF0PWP/1WB79+nUhMNwSpdpA3pc6V2hOGd4QDRVRDsUb+pVm02ZlObajZoao8"
        "qWOLpDgDCb6GwH6A6iEGHigYG6sJsImEfLG3+lDNakMG1o+jW+swlGgPCuhyoTGRxPSVTVMdN4vBFL61nU7rcHVS6s/c0lb0"
        "EDxaWnrGxSyE/RwqYLb9m951FkczntljvnApe8081DPi3iaVHN6muXCohj/26hAq0TFACAb8PpDDvjNen2r2g0kJeci5C2Px"
        "g1mm+h/c/pszHiB06EkjraqqTrLHuxp+mKehVIBJNwrQ9ogUJp3GUgx7jjsPqfMX2Y7RAzVfO7QGKthZc9YzCBDOHbXBxKmj"
        "Wdpn8bqjoBZhBvZfuOckpThgDS32NfvWxAI9NreOtIwY+9c//snYQ9U9eKx1De7ZA6zQrhfizmPRIbCco5sesSRiXD2otLQf"
        "oBYntSzLxIyHoWdsUslavenU0zbTmkBUg+zJt5CY6mjW3LlugBL4B1rweFjLqfSYhbmvG6AgSmbUyiwDLv3ozuqQ9coRZBRI"
        "nYhHxTaU3dRaMZT+6GWN4a/Zaa+ZABXrXe77tskdCF3H+d4A+GZofNkOVSpYlNuPB9U2k3I68WxNPCrmNTcvGV7koGvsqlhf"
        "DBCJhaBqiWwxVEx7sEgy1sAIyCLR4EmJCI/ETTwqubWOt0MKJDG1+PpolGXXCihtrHnymuYc8+YtkykrLV7q1RJynt2xLC4J"
        "0qqdpaW0iYN4STws5lQ8pIrHcGhcYT9p8MlTWRDhPwbcAf6+iSbfqzehEqM88EaFEaKz9qYESuhBKaIqROojgiEji7TN+pdD"
        "yrYdV+ZIvRsYja3IotAZNQc1FPVGf919DUZ7FKvVlBT1NSKy3vmERWUSWi5YCW58nPBrS5BCITxoJF2aA+wOLA4+vzbYW/oM"
        "OZhIlCWfMyodZNYlQatWPqVcXbHmvmqnlIakvL06gqKsrgSlcnAdKHi6QOpX7AtLO580kUW5BFlV8ie/9WCRPkKTd3QDWcsT"
        "JlBrxhniBwdIBwxF9xcBSucHYKTz3wSDjOgAlFAWUB6rvEAxrmIQPP4uC7QlZnHGQ9Jy6Odcq6SytbqlacmW6m2mlop6ZH4R"
        "+j5EV/rgbcAe9JrHkwe17WMlfgPcqrFhbtlm/oma/XW/1xu4/fnjV86HyDKOpNjkoa2P6tqD/gv5yB7a5PLag3P1XUNpD172"
        "e+qZrAdP3+KhueFDmyzBvMJDYL43Y22XYu2LXpVGK+JJN4VfeukOFfDDkK+mPmf3A3bfNN1aZl2SMB+3yW+3J4YGPCsqJpoM"
        "PBpCJoOXL3wwkRbg4eK53yTEEKMWFFu2J4p/L18oDipodc9AML91n8/1PjWzL19YpesuM4aNamDUT0FncR76sOEw4FNU2gtU"
        "1+ZUTqdvgt9tySzn+0pD1cN8x79dsL7bm+xok8qJFCQHPGhExiI5qoAu+ZrOVGhnOOMYKUVT4sYRFYroNiX9IXobG28FekQF"
        "1tZeTR+9M8LWGVj/mUKQP2u0cHaZgUgCbjQzB2T5yJPyKkfagLvzfVc47k329I2xXY0zRljXtFLLkLftqxglr8JduMwoy/Bh"
        "M25XiqP0idEYOeH2ZDw4O5v8KS1VSbcDi+LaHJta9P3w0alVrHI3dNKqK0B1purnq0RS+uPVkpcqE/usd7WUh6HciD6RKhnn"
        "NjA8+TwAc7nAq4ls0IjOJwb0F/h6q9AZHbfU10edjEfZ8NTZ9a//c3P1jhFLMnW6Dz8LHj0eKRBaLeiW55HcPY8Nh8zyPKpi"
        "PM/SuqVLmvrhyOWPP16Pbm5++mXEsgBWTmIvD0xevbm8uRndqOCtKgu6LXEXxRtChU0F28BUUVIDKgzGHHI8Uxc2VLdvRofw"
        "1akLn4KTpE9bOowITEGg+ixkpz6y7TSY6ksKKrtfCkDTGxcXYTC2oqRaXQuRogYdbgDQg2zAiqs4cFYbfWxPNg1QcruaxnRJ"
        "BLtnW0dtkgSJUFcriqs9dkLuhDA77fW6qne7gXjijUN7ggH6Qg3ALeFzMLWm0CyjgW634TjpZgK7/cvoBtUs3VX5mCvvIkV3"
        "HgZJAtJdVW+meeTxxQI5swzWoqg89w/7O41j/o7pp3X0hYMv6qb9piKypiGo81pV+mS47KlDzYE5RC5f99WhUHkxpa+YrQ4m"
        "1JtTFFd3YHE50OZttqCLScRBPFhNU7LoJhKQJtnD9jZFhFIlZXGGDMlEdFOKqTsJSO12YJh6X1+m4jN1gMWCRaTTVwIVxgu4"
        "d7hePyA13AWgMKWDlaE6XSlw792f0d2WM0WeOpcusEtzsQ9DnY8zfbWMzsghXHxX/5To1OHx7jIzVeGeLQlfLWa6U6QpgVqq"
        "SJZQQKxvW0uHK6nRSeme0KCCdO1NHdPqy1bUJ9UXyPQxqxKjREEKO8gY3/DtLqIrdXcugtbxBTjAF3S5DJaqzqERtCM/hISS"
        "OJHKSugsdxcEaSGrDow1pZiMmEDXBsA3WHV5oQzyXyhHEO0x7SYjyY7bBOAVLPkVNLs9sV/0Ov3+eee77/Bx1un3zvF5ir8X"
        "nRenzi6MLE7GbaDZ/qadItDZ/b0Z2j2M22KNOTzEnHZxxN129uWYESlxtIohZwotJU8Uw2UmeKg5LskT7CuhWE2Frzg8p9Yj"
        "WBlQpkD3/yC8GtvyNDyiBeTiPI32vhLcAp87to1zfYwXa0cO+DJP9P0lseCpSq0wKcXjjK6k6SOVnUzCusIEulMJIun21DoQ"
        "G3WdS9BVPR6AUtoIecAa8cBX2rURdKcLUAO6XLkDT+2YIDyJaCZMk2hDXaV4Bm4p0AFt9CknP0IEEPws9vl2F7dvEIU+6qoK"
        "n+r8rXnZ5anbGJ+J89ax6yBP382Y7ErrsdXMWS1gfagLadW89O1Po+tmHJ+F8JAIQkgdng67TzQedUL38Fhv2VAk2pKxSmJf"
        "hYIL81yVR1tf2OIvQFU8+M86/X9ct/8P7Pj/nq7/f7Pz/1u7/198AvB7TwF+z0nAH3QasN/nU0CPdPlgD2OYAHWiUJvAi1K1"
        "Wuiw86VlhnW8ptiBd6y2UGWu6mJVDbFiGXXCjvW5VOuz6C19rmrZUc4GauqKjIGm7x9MHvcKUpT2YNZjvU3UAPJ4qEu00ylq"
        "LNAdox51jGqHs4SnOrAopDMu2DNpFtdVoQw1/eXy9eh1VeeKA3WuqOrc8xf1OheVn2mbrasO29qg4K55iDhUqA3oqOY2WmxH"
        "FlTcuyzrAl2mxfW2m+65lSxUDKOSdJ9HyRfU5FUJUivMk6Nlea263lH6z8TII6p/ogCdkQXVSmUqeo9Wyfs1cqLkc7waVnmG"
        "1e1WtOq7YltJrei1ub61U4+1/g3k84A2YDIAAA=="
    ,
    "25_generate_codellama_corpus.py":
        "H4sIAAX0jGoC/81a63LbyLH+j6eYwJUSEJMQKdupU0gxVbJFr5WVJUWU9pyNVoUaAkMRKxDA4iKJq+Kp8xB5wjxJvp4LLiTt"
        "3eTsj7BsETOY6enu6f66e4avfndYl8XhPE4PRfrI8nW1zNI3lm3b1tG74F6kouCVCMIsEknCVxxPRV6XXr5mjDkfljyvRMHe"
        "DNhMhFWcpeyN944Nh2zOS5HEqWCaRFa4lnVZZFEdipJVS8GWIomGWV2xD6DNzog4G795z3j0KIqSFzFPWFnPS1HJ4fhfxiUr"
        "cxHGi1iUvvW/70YjNvvrWcxeM/n8P7MZS7NHkbCcr5OMR+WAxSk7egduqhDzWbZgRyOWg2NeVTx8YNU6FwPrKa6WcpEoC+uV"
        "SCsRsVLgzyPYwBdNkCTYIitYmKVVkSUJXkQx8RpXa8+yrpdgL1ayvb85+3bAFnwVJ+thWRV1WNUFxjfK8NhpRYPPL67lhKM3"
        "Q82zRWvytGKFeCpiUq4zHrcbIR55CTW32+B67DyDniEaqUg0soO8JcAeq0usTIxXBY/TOL0HZ4ITQxAyUds2YI9ZyOd1wov1"
        "gFW1GlaSjuRTVljVshDlMkuidhZtNBZdM1509rMSZcUiXnGWpQlp5v3xh2+n5yc+4ym7kFYEEgVY89hlFkPUuMJ+0Nvjn4mr"
        "by5v2Hef2XzNwiVP78EAFlpmZWU52BV2/Lebq2lwdXP+/uLiW28VuX+SKjy+PCWNxhH2Lw5hPFXGEgiV0Exw8eHi/Prq4uxs"
        "esK+mZ5Pr46vTy/OmdOxW9cH76ucFA02JiMoIsuDh8mYOSKFAkPocb62ODEZZNodkhjreUqsD7LhDvYbz4CJZx5WbEW+xCp+"
        "D4kjK4rvSV+FwI5GGA59ZFKeok7ZiqfxQkj2L26uL2+uZ8yp0wgUSb+HMIbk0PUtxlr/pB3CLnhh+ciaDw9DkZNVN8bhLIrs"
        "Z5GqzVLG5PboFOJHaKbs01GdXTqv0cdL6M+RFga+w6yW1ptnRbVFsk6DRqIfaRI+jTYOlSaU8uC50NtQeZAiWQ4s9ksfJSdP"
        "Q6gP3oINjFegyVf5AHsQPog0YtKQrJvZ8TdTqYUVdFku5xkvIqXLV8ps8KTQkJVhEedVefhLgDgcKmGad/74zVwSVHYNm87I"
        "IWHos9kn8rIUo53GSP3x+O2bt2z4Z4xUz+5vxIT2n9YfJFfligPFCnxnD8re5gK7SFu4WsVVRY5HvrjM6qL87TgxYHwko4zc"
        "gSBY1ORzQcDiFRkOdJRmFSfPLC3L9BX3OS9KYdowTfNI1mSey3VpHmn7Lb3HlaCWoW/aykR+BoKqcTmvlkk8N8Mu0bQsUPTo"
        "hRenAK7KGRE0Fg69dMB6nIBxFyMK8n5XEdqDEoZoFyxAacmP3v0xyBaSaJwHC5FSjKQdSrOfuM+mb0dHhjvpdgH8Po6kdgzN"
        "MBGcUFy9gFhpVmB7459FFDyIdesiPaLW1QUC0ITtlUR/EfJgiBx5yGwCHpseCHxseumtHqK4cNTocnJd1IKgLi6rIHuQTZdi"
        "wGwazKYA3wl7e9T12Ve91GErAv9pNwTPRZI9WSfTj8c3Z9fB++PrD5+mM1A9emfJRjA7/duU2iPL6tH2mS2zBW201PsUJ2Sf"
        "5JTzLCMVDilxAcwjd4kG7OYcMUJ1DUCMTKU/QBRFVugBiIDPPEJ6Ar2zbL6oy5Cr2AqcV7FURMOfalGsiZYIl2mMVunJxGU/"
        "V4VYJBJxyTjgmvg+ufhsFqT48YpSmSKe15VhDZaKtCEtPduCvKfBx+PPp2enUke3thYzkPzbA2Yj1COdkDOpSSKqlhmyi7q2"
        "lLqdA7mDjrzUpaUNpLT2nQUJ+3w0gqnRJBo9RdmqpdsIprvuLGv2/ex6+jm4vLr4fElW6Uju7O+zWuYglFiENdKmNTRXCl4A"
        "aLC1dZxEBGa8TVL++/jj4enJjJEV10rttqKlAEwmTJzBzKbns9Pvpg09lhcZBUHKLAHkcZklnKw1WyxkvpvwuceIH00uVzkv"
        "K8FAIv2SElSdfjZhVOZolC5R1iJjMmeRgI1UMqoiO9LkkOHwBCE3Wqt0TkQeu6irHCIVdYK0+IdUjxw3Ly7Oz74Hvb/MkO/w"
        "ouBrShYJa9J7xFnBIZRqMUokC/5k+PIaYkeUZSIdKR6i7CklKWSQgKPKhBHNtF7NRaFytd5SlAM2dN54bErr6QXYqoaoc9q4"
        "RZ1KT4XvwI1WeWJSWmievIR03PhMS/Ctx74DC6xJspEDFFmp0nDJgY8sfbEQBE4MSPiENAtSlzkPZWYLsM0i+aQpkrP2vLfM"
        "SL7qKWu3i2wthTkMm2yz5eidx05oBllImNSobcRznvBUBTN0lpgk2TPk4KhASOUf3wfvr06nH2HaL4pc32H9pqOLQ1QFgfCP"
        "Gusc6EIcLniCaiBOpeghQbJ377Hj8xM2RkZ7qJ+OoLl1Ilzt5z04wGIdANxaRBcB5FXSlpGthcgskFKGWVKv0tJQ3EEUX/V9"
        "RYCanIWViRD5IXIHWJGxjoZqF4J83dzLJ3RFJYwqSB5jLh9ooMLuXcrbaKbCxg53HbAfShuSGX8CS5Vwv4qf0WGMy9Duw6Lf"
        "dKiosMU5QHwVk9lox8RYlCdkxrQA0rVUah+WBpAzK7S46ncaMsLAeTJSOAGPAq9P19eXBGw5xBctjxKMffMk5xp/RXWCHJpc"
        "MaOQjIpccbOI4REEmQSGBfSMojgrGp22sA6yTQBTXD0DsaWMKDCz+n7JVLI0lG6CsUyWwQ2p7aDgd7qGrfKI9LzQe0+ADzCR"
        "En8+awMmk4U/ulEnQ60oNqNEqnJjWVYkFipyBIDwVV45ys4DOjLwCW9Mga8bDz5px6X0HW3fUvUSICnVUYo+C/sbnTKzl4cN"
        "kKmE6AgmL4qUh7Ip4aFwDoKDATtgB+6mRQmDMIrOtcFCn730oONWkbrbbE04bk88MKUjjFfn2ErH3Z7wtehBzHcChYkmvprv"
        "avXJbD2QkxyMloqSCkog9y0ad1pNoDTpZb803LVUenit4D2GmjpMwOKoLqYhVbH2G77hg6BF9YAn1dZSok+8QMQGBMsa0cHY"
        "geTF9XtZjt424tB5dqVdP5PLYPydHCieqc5UqxBPJ4IAYEqA0lLKeVlqET6i0CK/gwYy1CCyLiqZQJElPdtokZJcSiI8NuPJ"
        "I7+XYWJFAYhrQkAbcLOC7zwVnLYNCTMQh6LNE0eMlHHlaYkIobLlg1JWcIS3SldEvqQMLPGkvh1Xf9sD2zwqiROSGMrzyjyJ"
        "KzkPb6DAZqbSRalYjYio6mlmy0l+V/my27FvKb27s7fUTgdrcVqL3gSROonL/ox8XgJNcju6Y5MJvofju7YHVO2DH2y7T5As"
        "Ibkd+xja9Bt2PVJfGoF411PNW23ATX0LkaSvOKGu2uQWDljHjTQcxKIcmPNGFJeoWYU600i1tOYsZtCeppDmBo32+CqI0TVq"
        "dDlXO5HeC0cTdv1tHRmqUlVq1b4uJBi23q0OViYNz7dq2d9LYqbTvetOwOvXEzZuNSkk621l9xqc7vu86lZ1dKyjCjl50Elg"
        "3XqMBFrQ/BLuGsgdsLbUczvTAcDOwmbstotvmztdOb7MX483hy9ahxujhYkG340SafJCfzd2S7eHLy1e6aM/YyOONgnF9YD1"
        "ShV1qtWS1AByVafkmhI4GAci9NdpBWLsRXRZ6h7GGUt2bHufQa7V2nC4tsQMFhylfmS77tcdUB53k5PBALfAfMt3c6kQ5KUO"
        "je+TzR4G5pRw0pxQOHmP2f4MGDVBWvbg71SfO1LnXxVarbsl515Z6UMnJZOtoxMn3+GNhsWpdOr/N4N2VANfkWsK+9dySet6"
        "PIoc8NGfYlDgV63t7sj1q4CkDyYaN3dBTSPoCjWqY5CPjMQc4nnHxb3EhEtqFU4k1LkirHMSBACMIPgVB74UDOVxRxAmCLeT"
        "hvoVfzppKX4SSf7RDHU1M6RC2LPiwrH1cSV8CHzzOqkmdu/k0v7iPDpQ7U5rTlnt/RIswc3E1jch8naC3aAeaaY1aWnnlFhf"
        "CpiDZO/L3NBxILih/Z7EFKoMX82J8t5pGhX3ztw6bPsiDWUuQ4J4IrKX1rvR6IvzeV0th1X2INKOOul266t6zPWxwRwFORIi"
        "OZ+smfT13We6E5qLJRWZ0CitoCp2WXIAq5/XjS6Le8qMwJeBuntkPJZ8pw9wJ72zW4c2i0yu9OgJ0A/VqzY9DeRqgeRH9bZt"
        "k3SEbRyB9uGm0gYdOVo+6vWrEQbKLI7+OGq6xB/8Vak0ZdCi0q9M5Pgh1fUGSStPPx21SagAaA3VCLBjKlS6zPOMPsqfkjgg"
        "7JYPc07BfjchanbGZEYt73Syh6nYy945pB7S5Ej7GNEZU0+WjiRU3f0LkjyXpRSEvv89OTATYnRPMf8VKXp5H+2T0iwSJsWZ"
        "1btbm7QKf21YtnT+/4+//x/+qZK4paiPLHt3enTM4erxv/k/5RNqVfjEjbwZ2HsHqQpCCWgZQpKjJkGjT9BnKp6oRJhQ4mKO"
        "SiZ2XS2G/2W7lA8t2tAjM67y0VOX4s6iDV5Pui97cm5tXUrp42MT+DpNFQHlqbMQ0RcQ2k74XIYCu7mxp4a0hqDi99QwdzHq"
        "/sa+c3tZkwy8JuTqC+G4jZP9kNqVYM/EARsPOuq1vxwXuzbb3CshgbnTNggb27dfnbve7f3Cq/+gzaIjLWI1UGldV+kt4dIx"
        "ruRuuU33Sr138w+MEfSTBHURQAfOcSGP2v5dD5LrUv5DdZO6vnaItU7WHNLBdCixm36mUhGAv2x6RhQM6J+0g0DWgKCwVTJT"
        "3UYzQ9QhlSOHjlzgRlum6QwtVLpoxDcHzNLcWyv39xtax/T9rpVtD1B3+BiD0CY5snUPts52O8P1bXwg0ySfLeyXJo5u/Jcm"
        "hm7s3hxEZWkIflt19t5LLG7g17BqMHprZFAixZekTCnZGdH+yCdoyyYMfrE7PxFBe+SN6NaMfieC1lg/5/TsjTYdgluhwfC2"
        "HTHaCQYq5KIyjvoyQdfBw9VRSXWqOOLK5Sue6N4ml+8yYpwj6I5sPGaHATooCyhU0qbCkCOnXyUcwqSet/peb9FkY/x/26Wt"
        "HFt7RlfCjr/sCNp5p+XtyiUSntMlQymwd1HZsNvJmtgQiZTkZsf0MbGuQkwyPwzwUkCU+W2Ah3euF5eZqjecLgEUqqQb+5O5"
        "WpQ3Xuz8S7+9Mj+ukhd7Oz+p8rTBb7ZweBezt39MY2uc/iVsloeWUb3KHTMdEEMIRFdZkyONmSrpQv5oYzftic3+wP74tp+P"
        "HZtYxl76xobUTKaaL11z3YB9StteOua6ce0+ySttMppkY0GbnZ/2sBfD/e3BlqEe3P1hPBr53nix+f0W/Q8qZWGgrxKRzdaA"
        "z5ooBiiNH3xF4wf92a2WLAvlRxCkfEW/aJlMmB0EVAgHgT6gVFWx9U/1CY4OfSkAAA=="
    ,
    "26_generate_deepseek_corpus.py":
        "H4sIAAX0jGoC/81b63LbSHb+z6foYGrL4IgESVn2Tuhwt2SJHisjS44o7WRWVqFAoiliBAIwLroMw1QeIk+YJ8l3TnfjQlK+"
        "VCWpyLIINLpPn/sNzR/+oVdkaW8aRD0Z3YvkKV/E0cuWZVmt/dfurYxk6uXS9aVMMinv3FmcJkXmJE9CCPto4SW5TMXLjpjI"
        "WR7EkXjpvBLdrvB8PAnupdAQ4rTdan1MY7+YyUzkCykWMvS7cZGLY4CeAHT3YiAGB2+x9F6mmZcGXiiyYprJfCj+/WW/Lyb/"
        "chqIPXX9r5NJK/GewtjzM3EfeAzSj2fFUka59Kv984e4m8ZF5IskjfN4FodOqyXEBQ8NBI0STsKLRDydF9nMo+UzL/IDH5di"
        "nsZL4WGazGR6LxnMMslFLpdJiAlOCWxfELrLIBfSmy3UYHcgNJYijwEmjGeg6kPsg11FGuRP4uPpQARRlnsRkAgiQCt/jmWu"
        "mHoehU9iGfvyjZhLoEC0Lr18tsD10cVEpEUoxclxJnrCl9ksDRJaldVhTb3ZHaHASwEpFPM4FednY3DKD2g68FoWuUeX4NDx"
        "yeTo9HwyPhZHJxdHV6eHFyeXvwm7LuU9MZnFiWwPGagid19LTAQZCE9zyDB8asXAZxlkJJZbj4htcICxD+PbYCamElhJIe+9"
        "sFCYiMsFQOH3YfHUMup2AEwBJJVJnObiCOSI09BbehCiX6oTHmdFmGdiMv54eHF4OT79rcMTAIkwbtXWVUgTKVmeFrO8SAl3"
        "MQulF3VEFEfdWZDOitBLxVJ6WZFKUjVCULYqhMHoTIZBJAnYx9P9NwyRpMYSCOM4gean+vEAnD58e3pyOSYEj8Xbq5PT4waX"
        "OyBjLq4P+jeaz6QpvpeClWFwG0m/bj4tzC0ymZU6p40PsDogO4BWTlPp3WWk7V6Rx0tW9iRIGGWmhTbJgqzFcNiogmwWxiy8"
        "aRjk7A58MS2C0O+I83fvTk+gROdnxN0ETAHLoGalVbVKM/flXEYZm6SE8LQfER+LMMQ+Hqzm0ZvBrrxbEUdaUw//AETxtw+O"
        "OJmLp7hokfJIEJbEQZTDzyhVxmIiSDNEoyYfE7ARNrcIbhfiBc90mUFe+AKc+p2eEjHkrhgBLX4DR8F+iNO7ILoVXka2xU9I"
        "GwjwtLhlWUKIbw+PfhmfHU+GsLludxGDwB4uWD/Nz3modI3cSCpsRdvPH69AH3uwyeS9yIsokmGboWD/TM4YWK+CtOk7ZmAW"
        "bIogGhMstS2Li3Qm4XivJoc/j4XNvsz3ssU0hgb12oTtD6RRKZFD0H49fMeOYVtpbXh/cfj3q4uxe3F19vb8/Bdn6fNcT+Ox"
        "TKAlhLqKIUI5oqz3lTDySbspI8xFEeDX9YKemdxNB92a7g0HB9Pu5wcZdaGaeQD1qUAw59nJ8lWDibvGma0/9X8aKE7cBUnp"
        "zGHcAauzHcXMF+/eC0LgIdukMYmJZdrPd2M46U7dWjK4ldniu/lh+PBPsIS/4C7S8WufYzKL0HXnBfyTdF0RLJkEL4JKesrr"
        "t8xYegsXnElzP8vuzeXvWRyZ67SckD1l5hL+uhwu0jAMpk4Dlh5L5ecCxtzSipVLWmdwMvcdhvZHHEk1L/HyBRabaR9x22ph"
        "b4ceOLBwmeZ2v0Nu2KaHNsgNQhDbJhwglnZbAfLIgtyYzcqdhQEeGaDK1o54DJAW3v6r1248Z6BB4sITkehI5lH82RuK8UF/"
        "32DHrtOFR6cEgNyw4Z+KBPqBJC+QLnHzh/TdO/lUxdoG0NbF+fmlGImdlOiP1vkVTeGZPWEBumfRBYUVix46yzs/SG01Oxtd"
        "poUkBwf1d+M7vm2TD5qM3ckYYWQkDvZbx+N3h1enl+7bw8uj9+MJBgevWnzjTk7+Psb9fr+FlOrEfXf44eT0hKdcW9M4Jjpd"
        "2FvkWx1hFRF44FJY41uSpbozU+pphvqxZJrGabVmIR9dk1wBGA3B6czuAAP6kz5ZNy3kc0084KlDeGgFIEP+qK78eFnB9XJI"
        "c1rkGiGAab29OP91Mr5w348Pj8cXBGrF+FlXUKvuIQwvt4bCtj7Ef8BzeL1XTl/Yv4KO+CETZ5di0Hf6bwQGXh+8EY+vD9ri"
        "MElC+auc/hLkvVcv/+y8fC2sbZKxg/3L+8sPpx0RBnfw63J2F7fF0QI6JXuD/QOnT//ExJsjs9WArLZinnU4m8mEELNy+Zj3"
        "Fvky7HjYN1AM6z3SyN7j5ugyfPN51Hf+sfNj70e++slqAOyeetFt4d1Kggx/eTXpyEgtwcQ1dHPgTn6bXI4/gE+2WvlbXMBz"
        "IAoiUJkEDamvlyLbTFVs5XBY5e/wjb2T40kta9MMspRT00EC+jg+m5z8bVzCIx/KUTiIKH4HWRxybhHP55xBwdk6BpQuHWBZ"
        "9/CNtUwdOkAxqqwFCvK95Is526VMwyvx8UJkP/5TN08pYvoI6JRgx6kjzos8ASmUxgDVf56cn4EJqfcEXJAjPJQJlQZEXiS6"
        "zSgOIDgsvfQO+sNZIsVBygu99MkRY6oDzFJOWafE2HkRzXTGreEh9UE1IWk3irzarVKmL2eLKPhM5k6p6z3AErdMhqrLE1oT"
        "40+aOVar3brYf1aqsKogUvLL8IEtIb0uJJeR2JhdBt9NqWlUjXBKIZpEzmyRx6Gvk03K7JFrcvQ09QptA3Q1NJRV90EMHaGS"
        "j/N9I2gqTeB405hrLlNDUZpmSjGdrHvLEprWhUwuPQTvWQZ15axuSUUFmaXSCu8+DggWwrTIkCQGc5QeXIUYrABIplHWgTYR"
        "s/xgPpfkepEUzGLS/16WeDP+NJJo6hBLEXpTAlT4K73RakNa1tnSH6StoRep4oeE2VJu8Tf37cXJ+F3lz5p+elgOdNkVCh6m"
        "ihnq8rsqJ4xvqDt0rLs6Ozk/06t2zt/y+EM19vxOmm0eDCyEDRl9NwDr0WGob7+EwGb4wJpd22Ga50OaiMkInsvgEfCMvAyo"
        "ZtgZlgNdHti9fRWJhrUb6kBUUDk8Dc1V/VkVsPD4+PyDJrQ2YzOKDWtD3YpIrBAPQb4gfybvSRkXcAmhTJUrb6G+EunAVe0J"
        "W5mCmz8lcB1zbxlQcnrHGb+A7UJjI9QD1s86ERWru7XgdBqiEquGzl2r5Tfrysk2A+Dcsle1/ZwiSWRqt9ftL/tV2pJsRDvT"
        "odU2VOwbKvSGHeM9XPYohgr2LiNhvRGW8zsKQrs5SwTz5johQ1i8hVJe6t4BuL3BkI/GIemth5+ilb5cf4o+RZuU110bVc5S"
        "oTUUK/7cueaLLo5FXDo27dFQ7JLA2f1tQmNvlml3ZjzXbm9kaCo5zTm9yxKxMcEwFlNHjUyZH/KzPH0algjcYxqVEg5rRTWJ"
        "fsD8IDONLfue0qIsbw8byGu2X1Om/9jmiPNI4e3+hqfJR0pi1A6kOscSxizH5C4qOImXqUYXpSJIHNVSAhUSKPvRYULstv60"
        "Opa5rO0I1J0MmVVOsS2z2zVEQUjIwTfksp9gWteUfd5YG+TQTBnZYVv8Bcm1WnPdvxGjET67g5tqBECsF58sa7iVSIaCJg+G"
        "mN54Buoc5H4y8u2wXddZPIAwfxD/9Z//gd+t9kBZxNu6rm3rmf/Hv8CQmksNlMh20iLKKp3nICyjW0ow4CrsKUp26r60h5Q3"
        "eMrSRJADmiI/Ewf9l8zXCOaSPd8ZDSKdWWUJamTqPiNWpJluiwHeAz2E/XFDRxxuNF81Tg9xgeRGc36/31f9RDDb7Jpx6glw"
        "SDOCPIMR+aSXXuEH1H+4RUYR0TbURhBT6kMG1FciucLQnrrzVErx/vLyo9D1LFKFLAY8H4xAAaO7LE4SDpynZUgBNpiyBw+f"
        "FC8NywgD5BLkFNibBLeUb8LVpIBGO6huH08XFDY5A9yrOMa5XK2xa5rDGtQbcE5CJoBmXFegd+ceXE7JzMmxyIp07lGxDQmU"
        "TPdUZxJp14L6eIFK252We3F1OnbfH1/ACiAGohb1sp1agf8p2/tk/dX+5K9edf68buPaot6oc2J8merouEa7bAqmHVFGEMpY"
        "IIoRFXjabi3LutCOX+PdEddMf72LfqPcBAWNMo0jyXiZ+Hl8uffxfHLpUGeGIGrV65R8IIfUEe88BJ7SL2G1r9wI1pMjIRB1"
        "X9Lwsdqx8CI4El6z7TaKlBzH3FoR1eveXz+PVvW2jfO5iHNZhdPMm8vRixft9XYli9oDkJr9HedCfdoYRiyWiDX+iDFpN9ZT"
        "fN3GjdoZFUiFD244NZP2yvqMjMeE2Lajh9vfhxiRDeW1euAn7TeiPxWqisO7QDqe77tKL23rKOY4271EFsP9hXqx3X14eOhC"
        "fstuiby/AZOke9dBWIR0NxoRDnzEEoFl+DUkaH0TKnvHDbpxG8Np2LivVFt/tkk10+2NCGOyK+dW5hs81pFWb8IJuUM+gmMt"
        "gdsQqwZFFurLxpOFn3JOZlIyqOTderi6X1sN7tjS0c5AYHQFuWv+NEk3rnxvJErX4KCC9b0wtGmrLRLG/AGBDZu9uLen435/"
        "sIuI7qAe6tXgiMJKk+TKoqnd1lKwj6VfKBWR9VAD17soUIOyd8veEFxOOrUvhAHe0nuQygviocMgKZND6sWpqZ3J3KS0mi8M"
        "j5zK3Cqj2yoNfM1dXBFzAeSG2GqXBFxbZid6h9qjUoK9OLcQ9BbWDeFpyORU+fqmkWqUTpLRMClkGk+l++DNtcM1yvhT6WYb"
        "7uwZPW6uNYrc2kgUS85/UV+HX1u2qSNf0BMNQTlwRTA4yWWOPdPdZW7XI6aVtRYC3FRlAyDJS2Fu8LhSRhudUh2wFOmIf266"
        "rznmcfeOOK1eU/HVvstVX6ZjCv3vq5jiLd0Ao/0ywkxVShvdSlsjspHPguNmD05WFZYbCk9vCssRbEIBBsRdq/3+xFBooH1T"
        "n4VHsNaKg5lko6ma03tiWiXvqPxy+Aho6Ir5t75RrBOr6d5g3Vtp9NdC17CjFT7XCupoRX/XNR+8FThVJaME5ZiXLrYWWL1Y"
        "NqLriKpD3u6Isj/KAvS3vM1FEZGuPucmK/KEWMn1RrSgN3dBVFT+k0RHBw9Iersrs++V4bYcGa0BM0V6kU3bNbGK70jrPNRc"
        "mGTeddjJQCt4exMPqoniu11hTWmuqVoqEJrRxNCOGJjd2u0dAWuDQcykAMknFkVkE7oDQKAHHVP7VSatayJd+QzrxVG3rELK"
        "vPZ/oMjZ4I0yai4Sasa+Q0TGsRqCNnPZhqtIBtusqjz3cNe7iW3L2LCS/S+YSdWNGWgMyTL2n7GMzZ9kv1S2zY7Cfvv5ZfHd"
        "fke4DRXc36mCG1zAuucJ3dKfmk8l8Pv1oT0x2Ano20z/OTegXuXCAlDXAIM7KRMq1nh4MNzhJejN4mjjVaPNRGyZIk2llwMI"
        "Nd9gj5oR2ybJrLHKxMb6VsukfSmTtYFHc4nxU9+697/p5FEpW7uRhHwxNppS0MNiE0sTsM+8CncO01s+IfaR7lK7VueNXBc1"
        "tut2vqg9xk8vuc/mzkIvy0Yl9Avv4biC+F6GyTszta2R4VTf01jYln7Zb+3eFbR4RZiPrO8/E2E9uyEdgKAqycAuT0U8v4Ra"
        "pVhC0hoFlPKYxYPBwcuDL5Fmzl5854a10xk796UTG7tZtgDTUe/VjrJsnYqxyz53+fLs4+k+v77kRQgFgN9vO8/jV57JAHYe"
        "d4lGqvXv5sg1rS+hNqEzJrRnvVPXOGrzpjw99ZWjJV9AUOdNO5m3cTDgWRgqoegmMu3mqiDehvWy3392vVfki24e38moJv0z"
        "lDp6RXpLfhcLTa5zi/qvxc9082vUOMdhk86QsWUOXSEQQj3UPV116ETbwuUN1Wh1bzzIrApzYM/szmXrs3k2X+r9a/GW2yqL"
        "PE+Gvd7KTKSHhAMq2/oQoaFbKjr0j9RRMZoTxa4+x9PIDhqJeVVM1TBoP5dUXp//crPV8E3p8Cm3F71crGpg1jsOiCHtqoWb"
        "7fZNba9fDy/OboQ5CrUB+g18OEoxBDJzzrSuus5GSCt5o8sqTlD6uKeA6tAf3ZrgSIa/Ob0huKHCGBWxemQw+xTpV1i0OR8Q"
        "tpXeCiUZdeNCibloW7eF4xh8MpeS3QwkUEtMobRV3VW6QedQPocBlLlxauYbAgbDKEvCXXjpAnFnTdigt0YtVfHfQe0jU/uo"
        "qH38FmofM3Ig9ZM5//u0NupfkjiwRipGyLfqaQw/Ain8DJ/ab6gDJ/AbV+oklTlet4hDasQ7s+xe2Sc31bj1oNaA3AcQG8kH"
        "egM0snBt3h2PrCKfd3+yuK82rwyES8vs3uH+f2rPKyV/0GPxg31t3v3r80rmJWntVlW2fMxJqnNN7CeeiSJW45UmzQ69KWUQ"
        "wiqP2tONOu+ae7d0Yw61qYNw1k27UXMmz2Ziqh4Jqqyr6R7qdH4FCBdoVi17sZ7XpboSlkf37KR9o9UDOrBDxEozsl0ixqP/"
        "j/Kl0wSEs6sq4LpYqh0y26i8oX4fMS33QrYAWPEeG7N+/RAFczrDY06JsMJUajF8TgI1bRnW2b85wQ9uAR5zEEqpjWxbegS0"
        "mMNs6kxDlZq6fGaM9lZZbNcLLFapchvqVdZXbDx2wviBzhPUwFOUQSHBMRiQ59aqTAtMSFaxuLZGBV9XRhQcCR/lcrZnmOhI"
        "cRjT6rlAVdNzf5SSme31pqGQufz+gPcyQqvNphoqjUPg4lbH9DF1ZdFXS+i+SOngXt/p07HLOHHvcDfQ1wldO/11gylAjVVs"
        "WLXbas833LCR9KZ3rglRGz0jxaFvyG0n8sltHR/UyKMeYRr1WNmcquNodNmtzywVfGtvOjvgkk4QC4m7drPt1YPKP26M7W3A"
        "FAP8P6jDlqGX0GkmiDWO/KyEXUs/RBcZCS/dsiMsLPIZFpnzzcjsHmxzxNnBs7YTZLEqEhtaWybwrvkmCZI2Po1qErPygM0D"
        "fUWk/AKN2P4CjfX1SCysrdyw/KbNm/ppTf6qB0ogR1yob9R8E/Cvfe0mk8BfvZiuf6GFyqoaT5Ae8+nU9+Y0KSeM4kzSdyT4"
        "LCd/HYEObMJVd+gLEvzJ31HAVhTYsVWoNnC0wa83AsBWrCgi1zhLh86XWDo+fC0m8FkUv1gmtlmOeNeBv/KRPo1M/qKSNeSm"
        "Fr2THFniR/H6oJnHHZqAKlZNY0FKx2nsqrS0NVCnXG9VWtraZO0G2uY7+kxo10PnkLTrWW+u0RaiMSgNZm3yL/5GmjocZoi9"
        "frFhly9ufhz0+0NnMF//aQP+kcrFBOCrFKuJQMWWVgtu1XXpAIfr8jtu16UOjuvqF92qndP6b6+WAfyqOAAA"
    ,
    "27_assemble_llm_holdout.py":
        "H4sIAAX0jGoC/71YXXfbuBF956+Ysg+hEomW7TrbqlXPcRNnk7NOnEr2bluvDg9EghLX/FqAtKzqaH97L0CQImWpSR5aPyQk"
        "OBjMx507A/3+dyelFCfzKD3h6SPl62KZpeeWbdvW2Xcek5In85h7cZx4yywOsrJw8zUROW+WLC+4oPM+TblfRFlK5+4FDf5K"
        "9Zc/UJTmZdGzrI9cLLikYsmpWGV0ff1xsOApF6zgAbGiYP4DyXIueSFpFRVLYrTkcTDAaTTnabRIac4KfwmFRWYpNTJKFzGn"
        "3876Z8PhQGSrzgZ/mTDxUJ235DKSJHPuR2HE5ciyiC6GQ3qTBZyuY5Ywmv79OqJX+6v/mE4heo7Ft5znU84fKkGCaGe1Evzt"
        "Ndb2rMby4Kv/lA7tDhVZwWLL+lBQLrjk4tHEzsQsExSzOY8JIeePXKzrEKowyKwVf5+lJHieicJqOcbSYGc8DihjhH169fly"
        "cnl7df1PctRZQSTZYiH4gunUtlJsCf5rGWEfnPRZKTlloTav1nnyMQsgX4qoWJMfCb+MmXruudqnIMPWTze3SOYjT5sEF4KF"
        "YeS79CGkkK9gv/kAryQxwYk9sgiOxyqpcAxHWoNBJTQoGBBW9CkqCBYBRUtWEH+KJFxT/uYC0MEj+TFnguQSMQlZHNOKiRRY"
        "6iNwGlhVuABLP0vyTEbaawAIVcFlQQj/EqZpA3IWBDxQTn36fHc7JScXWVD62Dtf09mF91KffPbae9kbIbkBK9gJf2TxiY9c"
        "xCoVTUn58rEjESCUEqE8KmDWPfWiPpL5c9pBExyxQIFV6NmvKFQbSvPm7hbGd+1rFbsXlnFzgJ/FZZLKERX8qehrDPYr6HnF"
        "Ouf9Bp9HtSUsjULE0f1FZqll3U0vv78iJxRZgg1yOc+YCE50tCoiIumLKC/kyXEu+hZZ2sMLoWYt6/Ld7dWEbt9/mPZJWVyC"
        "lioaUmHTIAJCavUquXkGNKk1CCSoPi2Y8hW9mf54wPjTP+k8efO1jpMm0MFABfULMXdd94B7p16SBZL7Xce+Up0zzxS/xoKz"
        "AMTh+zwvUBbV/jBmiz8DGJwu/3U3ufImd5/+dnPzg5sEJAue03coYdUZdMI8KC5KwT2PokQVDeCegrlUxUjLqtfEImdC8vpd"
        "Adk8agxoVQBWXBGMrHW9ycoUJGYZbBS8iBJef6zf+6T+/TdKs5LLUZ1xNK/FPuPVsiY34JqxfnFgcwRUeD0XRoF8zH/W1Y+X"
        "15DRoidkqzja6kHF0rYsK+AhqYh5Fdylo04aaZ39HS2PECXRUy0wBmbug8gvZgoORFFICI22z61YyelVX9SfJicntInuf7qc"
        "fJrRRgumLOFbSiKpmh0yRKIEFQGBuz4QRgLFRNOHKM8h5Nq9RqngSE5K9zO9oup+XL9obGc5T7UbfYXcOEr52LZRACnYCarG"
        "dlmEgz/aPUUf4c7WEKcKULdKpPsWHk4QFi6csOWPOdBlOc4InI2t+MIekbi3c7aOMxbYsz7ZmkCwfNrv7Gz/2S16UQpcFK3T"
        "WYSeMn1Is1Vq9/6LniZk0NI8b6totcJ/80Mn+CPaxIgSfOltW01WmjCbEONzGyIVw3QQUrHNSI0v/0N4pFm3Zdbt8jgqtNz/"
        "FRbwFDXimERWCOj1aDwme2iPnuVPGXgIRvqpjaHh8dwfApJdBUqhp40MW/VIT3Gm+W4QAqsVDpQ1PfpLnc5DCZq+v5ncvru8"
        "vp5hOovXFX70vu3h5MCCPctD2zSnFUK8qZ63Lt3pNKuRpa1TTRjVyGLa0W60cbsoVRvuR5W6mcFrwqK0RhrLgYSard1LsSgT"
        "MONn9SacgFfdBww99rwg8z3vCwE3oEgQdy48P0ZHHjfaJ2z1dqfxPY/zd7VozxjjYrjymLHCsfcaNxKnMjlG3PsETxhm2DGa"
        "+dHdyKm9k1QQ1JQPij/UKQHJSpFYqPqAPm230ojKtFqsYV9Wk4aKfvdac+A+gv5rctJMgNDe6Su1VQdHRAXX5oPRVE+KxxQd"
        "miSVnnp9IE6NJgPPcYfFajX78yZUqGC4lZhXZcVEBtjzDLPs/Hy1s/TV7nbUUN9OUCFardTi+2Vm/5zeX00mN5MZfcro2TVS"
        "HxxieAhcmqBfduZw0y/bfMgiXGCma4w3ydVTVDhnxomqYeqhQTuqesAeS2IJgVzZ38aUKxUWQ5I/4VqkSLIPwwAXxehybNit"
        "Ibc+7Te8HWXNdp6s3JXStqx499m6iotTZ8b4CD2eD3PMrOWAWFuqd5y+26Z2KSs628IXm+7O7YlaaBs9276wjlFE9wwFCGyu"
        "XJ+pxnBaHVtfHXDwxtrRuhn0A68sfDB4PRm6mAicejh08a3nRjKrCMlpDQq2vmvrk7FbEWvjbEvIFBUkZJk4p19ndWu/6Sbf"
        "sH34fLspMmh5XnnPhZs2APmEPTnDQwVLg2+xqGPS2ms3TjXLOBpOB8LmtYXrDqx3VFDqpEPb5bUu356sbgbeuXeBfZsOjlo8"
        "KX+NI3y/GA7bLOk9SWlWuxsbXjT7zvW+ZrXaVi022XutXzVi8HZ2NhxuW6aDDnIwpCJHfRHQY2PrO0hNOe7Y3+/9jIMGjT4M"
        "Cvuq33Ps54VkVxMAHfuNR3K0LxQG5hGn9UNOz61H5u0esx3qjZ2ru22I70tkp2TdoExyp94OqusDaAH68vis20p/Tm30Bnts"
        "00uEujub36qgm3G8qdAtdaIR2o5BHG3q4+5fmKUXsy08Mj2u9blawdee6QqqFB769KiqQSPaBXsmRwbyzcPobCi3tHnc2s2o"
        "2Ch/XouzQ0rq32Oa36QeZXNjeGbpThlstrtBUu2k4Cl2IWHdj7uo4mYe4u6ueg1u7mrw9jw1B3qemb+rodD6DxUsyBcZFgAA"
    ,
    "azure_ollama_client.py":
        "H4sIAAb0jGoC/9VY23LjxhF9x1d0sA8EbAoSdXHFdDFlrUx7VbsrqUjuJo6iokfEUMQKN2MGkmiFrnxEvjBfktMzAAle5K0k"
        "L7FWK4KYmZ6+nO4+M6/+sF+qYv82Svdl+kD5XM+y9MhxXdcRv5SFHGdxLBIxnsSRTHWQzx3n8qJPaiYKGZJ9S9OsIC3i+yi9"
        "I52RSOnSrCIliwdZtOlxJvVMFhRpKspUUZY6cTYR8SxTmrA4S3nRKW9IP1x9oI/vA+pj5ZzuZCoLoSPMUJMiyjVFKelZpEiV"
        "kZaOd3gyrubI8SQLZaVuVuSlgrptOvxqNSGUMldS3q/GfYqSPCu0cqZFlhCUlKQy7CBJFyJVPNamQuoCokQKk7NUF3CKDPea"
        "ukmtYb2iOHqQDlSUT2Ki4zksk5THYiIDxzmbiVzDC0dtGsqJWXcUnED4z2VUSNUlLZOcJcINvYM2XJmP73udNoXZpEzgaHgc"
        "2oeq7bAmOdbAv1aBbGp0NttSwn5ARO72w+hOKh3QiD1WRUumiNdEKl7gTKMCIdCzQkpjXiHhmtCMIaxloWdwOGIqTTQQPEpE"
        "Gk1ZqOP8+c2PNHpzPqT+X86HoyEN+1eng9NR/92P9P3g8j11OivXywehoOjK812HeMIX2PKxQCjtnodHNI2eYGj/4+nw/PJi"
        "fDoanZ69HZIn6EEUkYABlcys8NkwCTlYqWDgTMbhXlZqspuQ13CzTym7jl5/ePe2TVORRPF8T+minLC7QwhphBPe/PXk4OBL"
        "/CfvDN6kdwwr37jo1yOMHPHId4DTEHDy6eLyY/8dZORiHmciVKyYAFYVCQqj6RSwguKfsts2oytiG7TCJ1LhsUa2MtIZ2xBU"
        "JditmNzLNISvR4PTi+HV5WDUNX6qEuzNaHRFp1fnvFMUYo8IWcXptEovFtrIL+fje9rbwxuAkwXxnC6jnCYzkQItAV1lCDkm"
        "2eXaTPv4vgX9whCYg18zoIGEkxcR4IfBC6k5O2g4fEO6TFMZtzmtBatxj0wJ2cqL4Q/WgWkGI1EqZKykTWfp5FEu4wjJUmvh"
        "wAmvrBl4qqDbq+w+M189VrDnLk1128SG9Dqd46Nj36y3JQUmF1JMZnCo1bypKXmM+QT5FWIcrkGK0elfPwz648GHi9eXl2+D"
        "JPT/Ox2Idai2SUTOEV+atEO5EGXAVg2jJTX9Wzv/M4p0DgL+d7zhC2fYP/swOB/9CKSO+l3kAu8gn/JM1VhCfA0MtK1+eXkb"
        "RxNOflmkEiXkXNNMKATPESUmGKhxspgc5Mpwm2X3VCqpNvxblVMjPFIWeby9Y8wWt7Hk+pOVdzNe2BAOj7AYSOTKEZimZKr0"
        "eDwtOWvH46p6YyFAZdRRjlO9g7azOLqtv35S6DrVs44SWT+XRYxZgSyKrNh4x5UZlc5uGgotJrFQbGA1bfkKBSVC7XEc59vl"
        "O8f8XYsRlzyyGUcoPYhgAzhmjMV22ekYM6EzbxXQkcjlmpnWuZ2OZpQCMWO2B3VvufLEjFbqb45+jdK1/HmFyZQgCScyijkp"
        "uZerOHukztFe5/i17SXKyEvE05g7YcS9ysqy+nGVyqbTsUIipSEGpyiCPHwSHDh2l1uJegaWkKGcWQhEU5pnJcG3aV1i6DFC"
        "vxFQHPAAMPMie+IqJQz6lcHGN5VANNVoyTOiuzQrOC7A6VsJklBhzmZYjc0J4BWLOacWBJkqCZAVgZHIr8ZGPevnv9MF9+6e"
        "+TAzYLvK4gcZjm1XreNhQu+FcirKGBmIzIvSSPe+FyhvvlOp+69//gO/9GHwTlXPv99fY9K3CA7Iip6bb7AeKFDSUzKe+rT3"
        "J/aNhbtFItI1pan7zOOBxfOiu79vvzP+F137zBmwcJ2l1PFMihBoWEkOo4leiZ4hAs/uGUCEBNsbzXPpdskVeR5X9WmfE99d"
        "LBcAd2ajRsAdavzMrt1TDGVF9ItZ795wjN3XFsDPG2sX7qaRs42QQ/8YaNuvWJkBUWlYxv9NLNnPCMnkfmx0NK5uW30NyHe4"
        "HbUYPgd7TEyiWaLP5b3MKxojK4PxLi+ZLwc0MB6yxLMWlAg9MWygdg/zT/KidBKXIb9nkmQTzrcHDdttVjyU20ItDTR9PZqo"
        "gQjfej0PBvbTq/HIyF3sizzaB2dWbntNwm/+VODsGTFLqPr+mgRT1TZUwFekTwru8XObqgJtpWzUdFAmRUV3SyXuM7CMwR0Y"
        "yukVEC7C5ubyacLFsG8+GHEQJTf8IyIU2kGZ8nZ97oHe1lZT97o/GFwObujM9FnLV+rai+K8y4vB31J3hySi86k9v0Vqdeb7"
        "+L7NvWwJpwZ7QMQN/QFBABPaJXIFulbWOHa2eAcAJWUQZWnVYl5U6zupRRSjdz3LxfqMlT9tL4TX2fkBKLzn2leo+dc3Tb/z"
        "KaxH14lBbMIkt1qL6pPYlalIpOtTr2eHbpar44xJ2Yur+SlQKG+Q0XX964MbM6EhtI3s9G/WMwzyPKsWRJod/GZF5Kiaef8L"
        "Ot6bDG49GxUXLSMUZ0h7/mhg5MUYnD4gBEwIEYVra1GLLWr5m564Wbwk47KONMe+S1RBgksQVZq9FF7UHrjJuAFeXb62ub3R"
        "/DER063T7Rvr9uUqcPeUKwzR9eXbm6b5tOS9wdJltazKXDhv07pptU3veac+193O1zcc21aZ3qc4brUWrr/ZmrDLRnNqnHl/"
        "nzSEW1d9x7DVt9rMIJNcV1/UXGmZ1F+kDA2L3WYraCiXfBJd3vQ03BS8cElDeNe81QmW0gaWMdvrJHNww3HqMSvuQQlOnp7I"
        "HDyUbRIVkf7GJp65J1vVlJkolb1ssuQWB4cYmxpWm93ZGxvkD9StbiEgxdy6gUCv5CjMSPmMKSamOWA1DNLmpmqtld5m4bxu"
        "MGGZ5Mp7XoOkLX0gW+ZzvWe61u8YtA8bozYQGLUPm6MaCcKjhkFvDGamkSmMPm+lv8sxZan42O7hbiNwmIRTcttkQiNo3d0x"
        "3yGKgw4hnTY1j1Jrou4KqDEHQPlWcreInEVAj+3RtEzGeSGZc2HOSedwfc5i9XXhBzLlLTy31NO9P7rVkcN0EoHTH/DVbVCA"
        "7XONqS58W6PZQ+aCteA7GK/TtrWvce6jL6njr/eILcr1Wdq1k15tc7E6rVFYud/2GJC7qdkLLAylQILIhz336nI4cv2ttf8Z"
        "Mds4Tr9EzKz5CmfBz9GzrerMi2xLwXMOmFe9PEBKRLm3vrIid17z/iLg+0DTp9tr9xoBzp329U51R9aiauGZJaAAi3nh72CN"
        "TXDBym14MyAxAFuAqsKTMINfwRyG3bYLXtExSqFX2XR8+DV2BS/MyoLMsZrv5cIsbWl7FR9sCQCHiVSUKi3SieT9drvFr+7w"
        "oR2zE0Ddw2ZtOjk44D+H/OeI/xz7L0R2mxGtCFDFiJ9Z/sJSFtPfK95hueUOGIqIsVLBf+0qhb6o03JrVYNgGJ/QczVzUZ2q"
        "G2m7AAHROBp70g/GY+YY4/HCfeGYg0xsUetLYM5jQ3z2rfGYua9ttRbfGI2ZVj/zQzc4mC7UDrvYRYGKpcw9ntegI5+hlSuP"
        "/rDiJ1VvE1O9PIY3Laz9pDaY09QVd4KBsUbBvLXA+IhMDedFXUC3Lzgs0/rtK46d5MxxHF6vZuLw5KtxNvW0fNKrk/VSTn2D"
        "YC8uAzvfTN6s8X4wk0+VPn4tnovEeIqZEtVGPO7YAP19IJPsQdJPP/1kg2rnm7SIUZ84qjGqfynuJOFcDn+qLJH1AUQm0erA"
        "jT2YBovHtfoEtNhXotCKK6znYjO3kU98z8+nqOvY5EhsO86jPdSYQc+vDyXxtqCbBo5YARdHgeBTFqWeWbteLevaKh6dfwOx"
        "4UCx6xwAAA=="
    ,
    "build_rf_features_v2.py":
        "H4sIAAX0jGoC/7VZa3PbuBX9rl+BMtMxmZCUlU07rTbaGa0jJ24dK5GVbVLb5cIkJHFNkVwClKx13N/ecwGSohzZ+dTMJDZB"
        "3Afv49wD5NmfuqUsutdx2hXpiuUbtcjSHzqWZXWuyziJgmIWzARXZSFksHrp55tOZ7oQbHJ8INla8BuWZ3GqmL0UXGJTxKK4"
        "EKFKNn32t8M/sxlPpMAeGat4JSTLUpYVUZzyYtNR4lY5LORLwWZFtmQzIfBmzmKFbcmGFXzNwqxMlWR2zjdJxqMgEanLfi8z"
        "JQL9yu2k5TKQuQhjngThghfS933HZ0dGMOTpgYJPUkFzGcsFs9QiloxsswWXjHekWMZhlsCzaxHyEu7GCh/HmRSpEmkoLOPe"
        "Q0HWCHZ2BH/t/cjeTMYf2HT48+mIYb2Qv1rM85ha8NqXUMUwmOKLJZOqKEOKsNtZL+JwUX+0WhTZmvE137DrDYuEjOepT+GH"
        "GzIs4lwxHkFerTM2F2kZpwJRi+LZTBRwnd3EKd5mM0aCPKHYqyynBbUQnayI5zEt93qMz+eFmHOFRFS57nc6PZ+dTyefjqaf"
        "JsNT9mE4nY4mZ+x4NMTC6JzZC55GnvFKRC5TvJgLhQLAN0I/E7c8VB3GUAJxApVIhUQBNHUSp0ysuEQYAthDUDhCjQyK8AZV"
        "5vRJ1KNIB4qXCmGeb4KcKyWKtM8+Dz4zuzfouYwPuGNiKxgMlvig84+nMWuESA9jcsFz4bI0U+y3UipmhVmqeJzqPJrqYVQ9"
        "UCEKa2vblNq1mGWFCOTvSXAjNmtUcB9y+h2bZUmSrfFB6xgFkmJ9JtZal0TajPXxpDs8e9P9dHYyPuuej05HR1PyOeQK3yu1"
        "73H6mzBF8Vz7+rzlrDZkNOWIFdUks+s22lak3iZ/ZLtvogwWzsZTbFlVSsxGFi+XaDhkHVUDvaHpPkSPVR8pnW0gwgy7UxXw"
        "GQJkwtJ/GAFUqed1n3W7z+tg/PfloQnFVtFCLROkdB5kuUAmeVqnzXpNNi1kbs7oHZxxm27RLVilUhXoiIPXB9Rauo7yIkPz"
        "2dbHH9hr7yf28ZXVclysyG0q1gSONxWUpZeX6xcDxDEVRZEVA5dABxCDX4ZnX/Dw+fPngWtMNpmgTM3iW5hN4uqx6SMyditl"
        "XSHau8oPYMRcSNXgVFHCg2oR3Z5KEZaEjwwvqD/zMqWY6HLwPOMDHOfhDRAtgSOi2NljH3iey9iB86PnOW4VD0o8wM940EbI"
        "oCAp1q1wVD/12WQ4PRmfI5zQziPyYou/LpNZ5YV2u4FGnWfECr0HjFdFllMpk1kgVEZtTiGSBPEaa7WO3l+8ptVYBewaGwGe"
        "S2hBOcrN8jpLZKfzEkj+bjgZHgF82Jn3djJ8z6bH3smbY2brb1lfu+yl94rNC76EnwRwPxweUimabWRyLeL5QrnIHE0WNp0M"
        "T87YdPR5ysZnp18gw/gqi5FTzDM+Fw5yI5Rpy2UWiYTlcXhDisq8AWskXJbXXlVPmFDCn/vMAiZZLrPGE3ZAP4HRP1mOAX6N"
        "l6SlxkwGzBW3aMQoo4CJ2zyJw5g+P8xWonB1cLOSJIvICzPdnijnQqMK4i3j60SwFS9inipMhqPJyfTkaHjap9qTWxQ2CTDz"
        "M5vbThdvNfDOi6zMRYS6TGJlO8hRB0CQc2AdaIDY5JkKgHwBOi6mfgQwM1srk5hcDpUFw4yS6AaUAKLT6/e01x0Nx0mBStp4"
        "mE0CRUZNI9UyuNPa3BVPXKCQuvfT/I8awZdCcQ9pKCDRQecnSTUjlxoqI/CO7un59D3hVRRrvNScoqky8gaRGJcqL1Wfgb1o"
        "a0RdQrlCzWMFlneeyYt6IaaGQ8zBYtwOp1qXcSR2Gx0iz/Vm+0bkyuA0JWxdxCiGVEdlk5XEPTqATQony5LIW0kvxWioR2zD"
        "lBxf861OvMyzAjO9mENEivq5aH6TG9nRVARVt0jia1atf8BjIw4+hCyh29K8XspRedR+kuWRUSBvdJD9ypcAg4L6EeH09dCo"
        "BKezOJr9Aifx7X8gIx144JNxHzAhCmUfutQQNjlgB8EsTkQQOD59caocp3L2qYqqLdXFCX37atNlIkXwBRC8RUjBhYafpuPT"
        "8dsvwWTEBgiVTwGHG3ZhXV7bQHjnUj4f4O9l7/IaY+Hjp/F0FPw8Oh5PRsE/R1/+NZ682SN7cXBpXfl3h+5f76EmK74igl/L"
        "FH59lSJBQBxoc0nm5O0ZVB0Nz0dO52j8/v3obBoMj4FWgTH1hO6Xh/e253199rV7+dyBb++m70+D6fBtMP4wOtsj+BqfccG9"
        "P4bev6+qn4fe36+eQ3T0C9l9B5JxCsv7YoF5t35BsfjW7fMPo6OT4Wkw+bTP6sV/GlOX8uoFjHU6kZi1gLDJh02106c3oGQ/"
        "MWpQTeNYCqVLfmuDves9QI6eHo2sNRElNu264s/AYHmSGBm9v56kNCiNTq0UBlFVrKD+bauEIfjKy0QNDo2C1tkBCkixrx9s"
        "6wBA/aK9cmAd7DpZS8lyafe0wZAMmo6ZaRwI/VjyBD1IaJpGzZrMeShsx+grBMKVsjszDvHH2ktzrT60K/s6yxK7Xei+RO+G"
        "CxMVx3F31TzGWNvaHumCpxXvYYBtnfuL/2mVO1ywrexhKzytZi/Fa6t72B6Pq9vD1Kx+u+pae7/lVNgK1Eoje7diugxD4VXb"
        "TIt5NTLtytyRuK86bgnItM3BCHQLdVhPCn9YzEvKzQd6KmwQDH08BFwNgiDKwiBwqVyXOjJBmHApB43whK/fbAXeiSQ/rrc6"
        "lS0f58yAV0Zsy/PiFOMVSKI2uRgQ+hOo/F5ioEWDaVHipLWAnoFVYz77xzmoFuI4tx7VaabuE0oflSQyUssh59uef/XyUZmU"
        "CKNXI9deafDISryYEzhBi44Y6ZE2cLB+5xvX/eUN/rXN9JNVHMQtDgpBdlN9Qccc4qgyZ9Yphh7NwDutRMf0nvm+X8UomsFo"
        "w9q2e5xvlIAY3REMRjPn3lAgdk5jky49KsIGZrXIoi6FShOB79A8Z+uGIVDRzGVEnOinJkzau/1EkvZodzVL3PG2UjfQ7taa"
        "4TRUmzVjAytkpNpmzDn3VhW/Z+BoXpuJ12mEWTBGtSgE6CC5ImmnliG0ThEI17wg9wHdF7alnaD81864OEnCC6v+Xr1APljN"
        "lztX/aaXjRsUi0rvhRXUt1WEL5j1PM+Tjb1nYDrVqzzyz0URC3PgNmprbUYMIykpl6m8IkNmBd6VxIG2ESEgYqlHlV2dfh49"
        "89RhqfNyXBXLtyqYTWcqU6K6aRrv75u4O5qDUwDNMKTru20BrRoCCecfUEqbg1Fv8MvAqo5ziLIxU3CA7sB+uQudj/8BGWhc"
        "G+xx12V0S1HQNdTgmK4lH3oHuqGamvwmi2a3UYlkECBczCzzfBffW7rCYioq7bimJS3dc6FqTwKqQxkANEAInCuTwS2tbbRb"
        "rUtPOkwCVXDA3tCv31x61osRzigKT4/GS+8q89wEgsSAA/qTMHf0kVUGdJSnX2m2gkHEDYV4Qu2D2w8Sbw00eqz5g8puRFov"
        "P63wW0rkfofkfEfhPg7j7uMi39Gzn3G4+/nDE7r2UAh3lx1U5fF/gy9TwBj3RXyLomtVLDSmkliD/SiyOb7KeFHwje08UKin"
        "A2DtDVf8GAvCbhtyWQVng20/ufigSNwOamO+ftzqpYsQrTLM0pAr+6Lx6kHrXPl0R6oCLW9HaBkze5/CkNrpR0SvMFgwxwe9"
        "HXcurIRfi8S62gH/aq2G59Z+BCsI5cpuUQYQvZkF/+8ot/fVJYRVh6KFUrsz9KFEn92Rfn1xfHF4ZTgAu22v9q683hawmfUg"
        "GDPL1rP2QSwxh1tj9oVhGduc4bWZFU49musbhzi8Scxts7khRE89+O4KOlsFl99Q1VoYAXQTxWbbIjXq/Khc5i1MxXDbJReX"
        "6Tlfgd9U86s1d1RWTbDG/ME+8wf3O4FBUOj2iW4xM9a0A6NLHD3m6OYW7A5QkGzovz3iVP/PSyiYipfC8fVZGQfDQCN+ELDB"
        "gFlBQDw+CCzzeYbUd/4HTE1om4gbAAA="
    ,
    "evasion_resistance_check.py":
        "H4sIAAb0jGoC/71ZaXPbRtL+zir9h17ojQnIPHXGlOlamaIcbWTKIeUkLtsFD4EhiRWuYACRXJf++z49AEhIVrK7tdnXBwnO"
        "0dPHM31h9y/tTCXtqRe2ZXhH8TpdROHBTs0wjJ2avBPKi0I7kcpTqQgdaTsL6dy24vVObad2kwgvVJQuJI0vKAr10933lPK4"
        "F87JFakgUyhKZBwlqXR71G11Oh06ez9ojq8HxZ6dmuNLEdJC+m4zylJKpUpJydRq8HRI+lBFy4XEzwRjIqVF5LuKspjEnLlI"
        "d2qxWPuRwOA08/yUll66oDfD0fvL0fDqA51fXlwMx8PRDTkRlieZk3p8vHQWofdbJlkO8CBCiLYRYC5DmYg0SihT0qVmkwQl"
        "UTZfUJxEqzXNMGMsoiW5kSbgKYplgtFgp1bwRVdXb5sFHZAoVEobZj2IAg2F8g6SKSlzLcYiTWUSUjRrGVrVTBr/Rtc3YEFl"
        "U5V6aZZKzQFvSKTwN8S1/sx1lCUbne7Urn1fBKLCyuSnK6/962SiNc2yeWldMXUReuk6VzpdksNacdJM+P6akizcqSXefJFS"
        "CLFZx2ywnDaJO+H5YupL8sJcG8CUl0RhIMO0RTdgMs0NHmbBVCY4TZHrJVLbQvgNEE15TPAqYA5iqcxPW1oFy2iLELCssFp6"
        "GhBabW6WsMlK2/V2agS00fDns8nl9cg+u7k5G/w46W3FrsCkggLWsJcboSTFlPBnAwbFynK92UwmkAsyOpHLRyuoLJDg61au"
        "l1HiNmeJlDT1vdAtSBRmxRLh84NIvTtJah2mYkXmVDi3qQekNyDoinwPS4Sv5VQwWEEDxgejiScVrsffJhRNZ5lyhEazOUlZ"
        "Ca1ZEgWDhUgGkStxVhpNcZNkEPswfEGmpA4iywWeVSwcFpnPJ/O7zov2dx1BjGApXOCQ9AKsFhVpCgxrHc7kkgDGMCJD/eZ7"
        "7ZVSpSIMwgmKVKTVCrjbe3YxVZCaARkZjM1oq6f09wwAngEYLZ7fb9EPZ+Nz+/VwdPlm1CNfzr3UCyBL04+iW211uUpzvyAo"
        "FKxUxxdKeTMP8MBRG+1BSQbuOKSRjocr40BLQLdMDAo0rGfQiWzGkfK0bbCeb73rgjkF2xZkppnyYBXcW6EtngPIAUE3cjKG"
        "e24QFXpxrLEKjMDSAjaAlF5Y0OG7HLNvJDa8TBsMT+L92I07CiTC0yhJpjEZXg0HN+wEBC8qCJQatlqFz96peYEmqNZqp7ZL"
        "5lKoHv9oAXyLFgwqk9TsNMhoL6JAtqGnzJWGlTs3OEaxJjFVkc/ehbcQgym/tbvQHy6/h6tKIsPV1zcO/sKdRivcb77xQeRm"
        "7AAgZJA7fZquqfu9rW+TjTACOV0dQUCO3ZcMQVHKxE5mdokDE0Ar4JJzTHIlnYypQavsK9fbMAOn1NLUxjKI7jjKwKmCAXYm"
        "IOxiE1jMfTSPyRWiGSA9yxETCGcBBlrWRnWA1mKnpuV2It/P3VMpEA2iLARgKqqOQV5oZxa7xT51i5CWhC0J7gPtEfOlYyyN"
        "gosIMqaDDUQfbQok38PNgSWU80tu58G0gUDk2CJzbOWAGnMDVFzaPw4//HI9Pp9Qnz4aSjLzBoydhdjKD24SxfytfCn1w1J4"
        "KczAj6xjo4T4gz8G7NTtd3kRnurder+uf6xi2wlcBb/s80+EVpgmzWmlfLHuhJ9J4/NODR73EW9O4sV6KZCeJDkHUcgeJX+a"
        "4Sap/DGIEH9h2uRJ7gwO6CIpaKXRfO5LfhY+Q52f/o7IlJ/X458v1d1cf3vBnHnjv66Ei0MGEELDuH7Q0tpUVi8/zptp4Kje"
        "9vREAqghdVqdfMxhVChIVsADm/OJEGO+DDe/i41NlQWm6VCbQov2NORafjTfL4f4Zjh8/3PCLa1I3Atry+6TF4c9YQMeBLfT"
        "Ldn3oyXcYF97yZb+YT5k5utWLqPw6jZ4Nnqac95mVRRvFPrBNJKh0DUf6y3fQIcPNiHk24XXtdnrKmxnHXS3omovXujaaXlK"
        "+NhkWvr2lmM6DEENj0m78EtpSZMX6gHTekj9m23wvjJxhJKVnXrsD3fGCBqBrS2Dfd3qlKdsBA8e9sLUzO3QKin2+2S8u57c"
        "GA+ocUDkkFnGxGIvwpR5y8fn5mNu9M8H19z6hhLi7r9JqHolH9L5LYtSuZFPw0b/MI06IsXz6kjdqD/Y6kQBhy87jW5l+CSJ"
        "ZvMxDaO9983Q7kZH9znidxGh/qw/TO2HMpcssuamStfw08jQBKdACIePssIY14vzPU9XCZXscLdSJ5hLOfVctX8ABdgopeQ6"
        "jlIbF9u++x5BDwm2Nh/qkuH47OZ6POFUtPLT+rMFfZT/suvN1bpLv2zyvqbO+3RqTGaK8N8O5dLnDOebDFBrZjCeNKfrGGGp"
        "CE5O4U+Mbh2p4/UYHwgX+Gw2+YOdLc+Is9E5PvWU2OVRDLp45h38XReILKL08rv0ukiJ4RPfrsFeU3sQhELyXE6SdH7HaOU8"
        "wGQfwdf1m/JNbdj7cno+vn5HN2evr4b0BVElUV9Om5rBL3Q9pi/dL/0v3S0DP8hVUyf4SCKUTq43mTMSdq0Czip0yqlvTbu4"
        "e6gG4BSxfnM0vR/BEFTkcZ3Vcff48Ng9fnEsG53VSee4e3KAvyfHs5P940NmqdhYZ746q4Nunz+2E7s0inS6WJ7I2a/vs31i"
        "hAKaRhGX1W2RID2HI4LSKrUI+IExUDHk7HTpYnz9tvw1uH4/ujH3rMbgejQ4uzHBn2yUkwjDfF/gg62V3gWdc8mbZye6CBKt"
        "lMtARW/G1+/f0esPtLKEtZGoa1ksk2kikbD6Zr2i70HuPlA6iTk/QO2FeNC3in2uIzgrLIywkV2XxElF3fX23l77/Yg/oXX+"
        "Avv8BRH4q9vYbxxUtfz8/aj+vI61z7EQT1j3/KG2dRXWVhJZDjxH4nLlyT6jlOq01N7ZZEi//DAckQmkW3TDj10aXmF4q+8H"
        "aNi3aDg6r54Gt9B7XN79Pr4LFl7mSc4ruDTf5LLPrH/4ZeU74U/R28ltv25ZL9vFko3g5Z4nSkfzxUmj2/ke/7uNbvcQ/48b"
        "h53G4YvGYfcpSkinSCVOf0XRSI7HUdJnTZs6D7PMrlVZKmiRyFn/E6dmz3ZfnJwW6Vmes3WtT8ar1cu2qLJ5N3/1UoS67kMR"
        "MEURGPbL5ey2E2+KAmGEgqy/quwr695XpaDlno0AL9ubNdttK+4TpRqBngYzFRlp5cwsjfTQt8ps7xVxcK+tl8OtVPTF9i2P"
        "bJZIdvju5q2AEpJQQkWnntt3g4v15Zvhi6pRi+y5v7U5d15anmtV9e3hQgWSKaFAhdqf+Wmh8mfz9LQUiUfb22Gcv6HwbHd1"
        "MCi34Hm43aRn2tWpXMRZBokW2oGm3FEqGyU7tc//k4Au4AemMvTmumbfdgkq3QEkF77LXY6Yi90FVB4tK62CP5upSttiG3iN"
        "7v4BXdcHURiibKIJ0vXzjD0zB6FJwL2UZzThgvNKLOnCSwK6uhpsDPEOTl1xw+/Ok0tSeXFKB619U1gcpdk7MnS5+kKRKHtI"
        "XDyHr/aGRLWVwElNcrd1s+xgAiSwOrK4SNZxgNpufbummbhD4Z8W8a5Hn4w0gt7z5g/0zD8+GWyCyULcIsFATSs3BIYMVt0N"
        "6dFRp0PmZag7YVAE85GQXqDbEUrCz0dzpTNWV6bC8yucOMxkn34DsPYI5W1qs5zylAsIU0++omnmzrm5+ZVmvpib1indb/ZD"
        "/8j0UsoUB86iDfNXXFtU82tkoQELpC+leXR0ZBGs1jw8Oj6p6EJKnXdAaoS6bqe9f9SDyhQ8AneLfzqgl81X9NMhO+yQ+0nc"
        "9/VFuKXwLoncDDxMfny/S2evB00cckoDYHYeJeseDVn7SRRyM+AZvREsjdrsHiMViqHGwFvRPjlZDE353PRdthFfUhUTKqj0"
        "lKawAv2VDo46F1qV+x1sqXCBe0LsrvDVDGDOmbfqMS4iH0lNyJc4jjy2kk59M+Azb/J8Mras6I7yYeuofURci+ss8ZMx1x3f"
        "OJeyAeRwX3/hxTHU9pfK9rO8v9ajw33cjLdQulzT2Z1s0Fmc0u5JA7kdUFS0xpPbbSpxDnVzwZD31yZnPw/3O9+RiJEpcEc+"
        "Ih2nFXHzgP7vqKOlrpT8AUKpWdbJABCXHlcof9mu37zK4Dp05qXa6OOLVqtlFNmGXgYwxm4LEru2o+5Q1ARh2uYMs8mb2yg1"
        "4ixV7Rm3z4BVCUGka/McaoM2anhNpYW9Jdlf06RB61SX7nqOmzfAtp8Foep/NHwxlb7xmV+T8PRmoKjtZ9j3dL/JTPQwd+NS"
        "2T/cbxCKd6W9JZKJfrfTKfsDsxYENktGdPMhz08m+i1Bj13NjF1U/gpHv3koXoHw+yLuayAh5Hc3AMKy8iqIYE9fu1ytQKkv"
        "83+pPxB5qD4MMN85cT39+xrE7CMFapG4xYbND5pt5jqnDO2AD9dz2PdEU2HqE62PvQZ1P1tVTM2MwROvuMrXYGb1lYvVo6+b"
        "k3utw9n9p9DYKL7o86Cw9rnZ5HtKW6dViLRZV0C5b8A5nnQe8GIUJWFzcvMBRVBZGJr6FVRzKmE72dSvVH7nTZllWH9wSBIt"
        "mbOPT3anYoS4N8MbI2+uxOxQHhWohe5/lTkczmHeC05dTKZrfazK/7nkQrp84tYa5q9ywyHM8mCutJQszFSYWmTc/O/rHpAm"
        "+Nh8egGbRj/ct79yZ+wR79Y9mcWC9lPTe7hYvVZ3dv+dldtUW5QV0dBSNDS/rJR/ePHj3fkSla8RVqUVGcAj6v6e8fZyMhme"
        "GxwCeTH3mzokfaQLRs6Wsd1VSkb08WtO4f4zxf2vTL7X2p/dE32NP/ZOOp/vjUewsv41xjjzoSLzMf84FZvPuSnFr12ofO3y"
        "b0DMnv4nIKskYiXApk8BzJ7+PsT0iVWQTasgezRbwmz6AGZaSu4PVpFmTx9h7eKhLgC6zb4CdxVxNOY28+3Hs/8Z5Co7G6XQ"
        "jVK+pyF3ccZl7bvryeXN5c/DB9DrltCLEn7F+/+GvYvcgMXrEw5DikyWWScx4E4HcM4lZkgp9WtHMUWC8MegYwKzBhNlbSkd"
        "xExWWhUvOiaUA5XzbUu/Fe77Ipi6gpBfNVcfESN6T+nk6+weNsfuXutgVihgpwbF2ja/brRt3Ue2bc5dbNsoaOSZzE7tn4tq"
        "MX/SIQAA"
    ,
    "payload_validation.py":
        "H4sIAAb0jGoC/51ZYXPbNhL97hn/Bxx9PUuJJNtJ72bOl6Tjpk6aaWrf2e5MZyyPApGQhZokWAC0rIT57/d2AYqSEyeZ5oNF"
        "UcBi8Xb37QOy87e92tm9qS73VHkrqqWfm/Lp9laSJNtblVzmRmaTW5nrTHptylG13N7a3jqfS6syIdNUVV6WqRKp1V5ZLcXM"
        "WPH27a/Da1UqKz1GzVWeDU3tRTTnBkJWVa7x0xTWpsbPhZ8r8dJkSrzNZSGFLDPxk1LVuVI3Iloy1o1o7ZdzWWEp8XQgzlVK"
        "Tomno38K7YS6g9VU+0ORvF6tjoWr2jux0HkuSuPFVAlvFf8m3fYW701I72V644SsvSmw01Tm+fI/Qsl0LqxKjc1EUTvagnPs"
        "rPN1thQdMOTy9laaK1nq8noFxyjhvSQXmKLuZOpFh8tM58rBupe6JGdmtKvO4vYWzcxUVtOuwiK8CWxgZs17xWuK2vFG2Cl4"
        "PzdYGYNvsUsFh+F6VbtRQshdzAFSYbI6hzNlCrhpmnFr0dOlMCUZymBlinFVLlM1Ese3yi7h6x8RcQ3kcnN9jbUXGvGT+E06"
        "/OAMezLTpcxjfqhsL0zE4NTUJYLRs+rPWlvOABqPjEpNBVSD+b5IZUn7tKoyluZ5i63NagRlFHYCn+eKI2YVQMr1lFHNl1ii"
        "dMreBgwcJqa+tnAGcGOPTgyHtCKPm2lbCLnKc8ohSTuuncbeB0iXcuitvtWYfv6/t3rv9/NzShVVVF6YGe+UdgKgKZ6y0Ozf"
        "BZnPjDg5vUDQVVp7IFqispAZI3Fi+AFlYWk2MkzRsjm7u4SlYhTLD1EuxGQyq7EBNZkIXRAaMIU05nxwhEV8a9XqsS41BRdp"
        "JGnAr29OJm+PT8Rz8T2+HP0evzzZ398Xa/92Vj6K3JTXyEWCIy8MskjmC7kMOYb8UbmwspjmGEr2d8R5h7LT16Ukf52oYCLU"
        "lfDLCkl01DJAqCUgW2GAQM3gG9KOTBF9aGRIptKcOSZMBaRIU4r1GufEiA64rMm3THkkkLG7jk2p6AgifoM5yK5KUpJgLVHV"
        "tkLmA+oJIvtmcv7m9ck5ULnc3iIwrBqlpqhQoT2bjKc9p3JYboCsKRtNCYYvFRBWDeBQ9GFN1VC0m+h6fzxNBmToTX8QjO6I"
        "C+IX1M1S7ImpMcQWorJmih2+NLeUnUcnP1E1LxQKHZ+nZ4fiHb07eH7wLnBMsBRfPnnHoNDmUTKmJOJqLU8oQhkw13bAVAFi"
        "Szm+pkSdnJ61tlbV6ecIx2KO2TGZd52Ykn1EpSB33pVm4v7M9WQV5nejzwNmbIMlgcHYPboc746Tqx/wNM4e4+/z8LRCh3ww"
        "do9cxDYfCECOdtAspPbwpamuJ+HFFDw2L6S9+RTt+xbUnbcgYBBsrWLs7oq8AQ0AeP+V+b3hsNlp9saP+gn5+7V/O2CXolAl"
        "0tLcKKrTTyxeEiYAokPqS+vv313uD/8th7OrD08GH9eQe2j9uboDpaA8ZL69dYUsB3V9McmfUaDk8P0V/cFaV4++uMqO8BKJ"
        "VKnys2ibkm1xtL/BW3VLWM2BQ67sZwz+IW+lQ5Oq/OFXoixzKk3UFCi6iRTfKAS9ITZNoVuIGvuUgb0vBvy7p2nz3VPV/GPn"
        "7odx1v/KJnZiQ83EM/ECxY3taL/8vIuZSWvKjWaB8jSL/ng0Xqy5chU49Yz7KVXrHlodJMLQzIbo1vX1nCnzRl6DNbpRzMvo"
        "rNQbpnnsquD0acgCKu9WUQ3PDsS01nnWF6rQTJ3aioWxN2RIEvs7nQVWiYw9IlunU2qspKnKwDjIce1DWyeVcSieHHxHfbHt"
        "+52Gs2ZBpAYqQWicGpC5XE7BckG9RP01APtoSK5MVSBup1xssVEFBH4n/cE0PlUpWjVvE7XsxdGPp79dgOnaroP+xXojLiEK"
        "7Vgs8a/E/Bc/vzn5ZXJx9Bo1sVkMez9QJ7xBmrxoQ0NhmZwdH52fnrw5uTcjBjrpJe0T4oy2IArV4GPXNXo3zxvNAg4f4Gr6"
        "a+o8w6ebx4dSEQmbZmVF7xbi2lBU8FYLufaN5APssorQrkGeOz9oSuAwaABaOeis5Oiv+XLQsCjjBzRZqNGiyhU/o6FxRwUR"
        "QZygbaMZd9PxTnFraG4QTpKIBRK3AaRLU7NSc0o13ghXF6Di/mpis47GgjtUQx+8WzzEfbdWEMrwTG/5myfB2osp2LQew7k+"
        "0WVrvL8eIYgRlBaKUQ0dDgoqO0QtZo68xtkEAhKS0FRRbDvwDokWpD6dSoAEGIPkuCNVCBInKaha3YCfIUa4wyLH6bAhS9lm"
        "I6EGjKCMy6iRWVocn1wcn7w8npz/fPTf4/tZxhT54eng49g9/uzz6BG4wT36++YGz9SsdhK1vof08nIIdvCel2/VWSGXobIp"
        "sAp6i2pyrZSRxq9+Oz96+0ASa9FLWWQ2+NgFTxn6i+SrSzoT9BtKS2esXVISUL3pJkxAnTnt/FoCB6lHkgiHnIaH0LEC2YJj"
        "YdXIa0IbdbLciOL2VqZmgk9TPSrtQxLyfTF8QZ+HwTgk8oXVxUCQ8kLuWYtaz6g6/qyRsSwLbYHjFCJIi1Me4FcHBf7ql5dE"
        "DtDnwVR7HOViCueDSGnB/XBGI30s5qYw1/mymgszRRjiuWwFfbAHjsvqVGWDlqIAAKlRLl3SWFgcBJfCNTD1gt6sy1fKO13g"
        "0JZEHD0CRTCMHGnmXn/t7QjkSEe0XjK24xIYJvjb3xyoZ2gYALIvXkD2sxT0l/tX4jnmXw4Prro32GQCXfIuiRivlrk8OMTA"
        "TbNApbbYexeweHhVbcly2AaRcCek5LtAehxr1SVJ1QG9u+qiehbs9touMohny/6oPWQCvt1d4EZH4NUgdD+Fd0scH8RCriRX"
        "lLaIajyvouSnCEWBzXoT+wsNcnwg5wNIiD86R2w9XSQAJeVwu78OpQjGK9QlDo4JnRCX3Zy24EZOQYDMW3z6DxvgbJrYUOrR"
        "UIlIUCDb2Z1LUB3xlPegxVniDcT7HNvpfSg/9tc2BMUSj4VfmU3KYH0y5AUzpjU5sZAd8sECAKKLowIr5GV4RXmD3EfcmAFa"
        "/D6HaZtjD0PTGZ+Q8ZUvQ8TYfoNqooEP/WttvaTLBSTLj8evTs+Ow41Pe+Thi5XDIGRCOQfC6faFhoIW1trilrogGnLIOzdb"
        "3jNH1TdVnWiKnNqqlDaFVmLl25NohcaE1cwEkj2ixSCjOyKj7uG+WrAdMsoNhFuvT+TQS56xJRJFEEnx+S95EKoiqqm/siXG"
        "fxX87m5BM3yrNhxzbxAzz0V1QNKZr6AiIfuYnK09t6kioIhSRaPUXQqRRrGeqUUQDjFG/Aw8V7g5NBV/j4N5EPPwvzju9xRC"
        "i0Nr45uAaF2dsKt808chRoKRP+v3G/BijYypASR0nE8CeXZnxI3CRNvr2bv7IWKitHeUFbxS/wvcUZrJh7V1P3bXB8nmHQSa"
        "6q02tQvFdSjo8gn0IfNqjsOC12lEub3J4ftDpgLH95CttdK0Z4CqLlNfy3hpCSFwQy0XCbKm3wabrBTjCd3O94XPSdf20pF2"
        "7EYv7DyljW8SsavQ0UmXhRkHn4xrAQ22SozpcxK073gjvf56xgQXXojv93lkt8Rzsf9wbjB6E7qrCeMnRM4u2WjbF7amoUnX"
        "vaNceq+yyY1abvTwT6XXT3QvLTDuMMipx5DwTs0MlDseTZ7LittxS9UjcbEwq/8B4KILtjI9m/GJA2FBMMkKdRCaxBfp3Ibj"
        "/baThWqv4wlcvhyPZ8YImuP6S30rx/nSfcg3X2F4dxOGnFn7D4RdOiOgjvym7Fq7Sx2tAOoltGdwYJsAo3bz69oMatbVUzrz"
        "O7rrSgT++P5DGur/I7XhPYkZAAA="
    ,
    "prepare_honeypot_for_training.py":
        "H4sIAAX0jGoC/7Va63PbOJL/rr8Cx1StqR2JfiS7M6UZ3ZUvdnZ9k9i+2Ln1lNfFoklQ4pqvIUg7Gpf/9/11A+BL8jw+nCsV"
        "kUCj0d3oN/jmP/YbVe3fJ/m+zB9FuanXRf524jjOpKxkGVTSx4DclEXtx0Xl11WQ5Em+8srNZHLdVLkSgbAQ4n+uLs4/irRY"
        "CbdeS6HCtcwCUVZF1IQyEvebyZO8TyJ19Naviw4xFviP74ByJopKbIqmEpUMUlEWqvaDJkpqj3DO50IFmZzEiUwjNRVJXheC"
        "9qmfCryUTS1AYhbUSuMIizxOqgwbB1W4TmoZ1k0lRS5lpCbu+3VQ1rISb2fiOrhPpTh8O11MJkIceuLj1fUncV8FebheiHAd"
        "VEEI0LmSPzcyD6XebCYUUEjhZsFXvww2aRFEfirzmZjOgAZ/RRUleZDOsaaIZDQjrqINESmCXJxm9zICwEqkwQaEzP+T93X/"
        "+m6KZ43hpCrKoqndA+/ttAV4e8TP47kTmSvpqmSVFUk09YiVI098/tAyIvNVAuYrCCRvMlkloYhlQDJRwgXhq3o9A1ANvDiJ"
        "B7l5AgOajjgNVgqTdehNRUBnjpFa1CQ47PTfRb022wAXlEaoMk1wHFWR8QldHX86FU0eySrdEMdV8aSE6k7vPlByksqgymWF"
        "CQkRR6AkCaEFrHL7j0G6X0tVi0xCbpVaJyUpRIUjSYgjEiqwTTJZB3ODCVwVq0QBDeBWYFMlRT4V0Jp7qdFiYZELqAwIj8Qa"
        "ejWHSEn1oySsAa00D/dgEHx+PD3+8fhvp+Ly8+n/nZ5fn12cLyZzccW8JkpEUGgoOejnnfwkEu7fqqIpr9ZNHKeSAaczkcNU"
        "AAYheOJTk9ZJmUrImXgBh6oTG6m7UJscz2Bin43CIIcaP4JDmUeiKbW4gSIIq0KpscggJ9Avq6dEScabQR1T2EeTRoJlZbFC"
        "co8yJURVncTQemiG9FYeDlyVMkxiiFLBtkLpQ/4QeY/VdaDWZJSqhpaLIgYWYxVC1VXDxjeD4kdCPga8GU4EpxOQRUEh6VTF"
        "E9N0TyZGGiYjjwT86eL6VLhJDE5UVtSShF0GSsloSo8xxE94gxJiwDkat3D9+fjsvBVNkafQ6iAmm2fZ2lOLyUaTX8g+5Vew"
        "nG5Iw6MihI3koACksD8BjtZnQAvhKKMC+rWBXny5glYsiGHtP8Vv+U7xT21XYEg7rtYvtise3/n3QUrCidj7dSugoVFS2T0i"
        "PwrqoDcNdzQ3gsdx5uLo4KA3y/KbTC6+XF9+uRbuU5XUNYAgMot4Soykqs40wV5e/iL29QA0qv9KysXvgvzPDU4or98eidvz"
        "mSCfiM3voO0bHv8Ow3dT0vLY4A3VIxbu0wDh7b0yXv2uUfc8V+uxvoHbvGdd5ZP0syBPYlr4LwXr+NU/YITtkf7nNbyatSj7"
        "HqZQLWGEL0qct9YhCoqTJCuLCupWrSB8OC3zTpvaZ7iTtX1WGzVhcw6LNJXGpZi597SfrPR8iUVpcm/nLgmHRQLjKFkp89IO"
        "ldB38sOwg0gjUA9syR7btq+k2c0i3PJDk8nV/3488388/ekfF59PrsRS3Dp6lTMTTpNjLT1EiAb0q1Ip+eEpSGpoMz3KrzJ0"
        "TKwb/DnwDIfLQ4LB097h3nKPX76WfphFCn42pVcEaPiwWqOqKchCERrp3E1urq5GlIVVUjIk7KOq9P4waSi5fophr0o/ZkWj"
        "ZAHMu2jDPFxOZVDVxWqVSnoOUlDi0tO/Argn3m5Brz+oxxX/JtkKlE0uj0/8s5MbUHUw+XR844PEs/Pjj3g/PPpWiDewS6g7"
        "ZxTHV+/PzgTC4kp+DwcF30CxL7gHbVCSUpHVfTn/cTKZRDKmbCLP4UpNBHbVgvwmh/YYfNYLZgZOkMKHWrSsVRIGkYsD74CH"
        "tBaDHKNerpryeI4hWKR9NavmqslcN4TdITD+mVWXvM2RHaKwGpIH1Gg9PiDlTqeGaJv31DhAF0a1EBQ4mWgQr4mE3XyWIcmd"
        "ogA736Cug/ABKRWGwS1MA14bKBYCIbDa0FoS1TcIvNHGI8MjRE2VgAkKmyuJs8IrHYyjGSLI/qyJpj6NazCKVwaT3mVJGD02"
        "btf5LwAdTm8P70jEeCOmaUOZImiaZUZosfPMCF7EM2F/cTyit3StUKy78uHLrL/qhDMznNrDpTEtqLR4grNZ8rzHL+7gsJ7b"
        "M3d6+aaz4HOlRdNO4R2jRpiF5eeRO1YvDS/e9dfA0fgc54PUp7RXYTUpyGGnB7TMKmHoJSpIscidcvy1Y6oMQgkdGWGOklVS"
        "W5QExwPudIh8vKop4YFD5Ie9hTz2awvhm4PMZ51lAYw1xuMp1/kTtOIbAf66M98G1ipw0MefKJ+KE+CGrbvtEp0VmC0skcul"
        "cC4vrq6dPgIkS776OU18k2YbTHAS7gNRoVWB+OPXgauejhF9Vep34un71QGanxukBa3AWAGNiPZYRL2RPWevvzIsMsqTUNA9"
        "yHwnhvl8jMLZ//PW0BsrnxdjRaZ68nX15LZG0yYXzCqbUF56CIdVFWy0IeERZoTRXyTyYdfAz0RUb0q5xDgnKtq2SDbJTKuR"
        "5LoIaSfvdrtos5jO2xItwA3i3HDajmLD2+QO4zwNZeLfH5aiHyFYjQ4pRBwCEo5/Jg5g2kpWj6aAQWzp2zuwGlloWy9WLqUJ"
        "C84OmPEy8k6Q/X2AtsuFWRqCNvL/t3c88JSgNitKeAhaC9Wk0MkyhYddOk0dz79zuKiLOy6JmBQ+zM+LGT8MpROTq1wMgisD"
        "Lfmn9Yb9eeMyaH6xFZUpCiR5IwcTdbXZhqS8bcnZlkcyUS4hHO4kv4ay1BmZRw2JE0mHcUpJwzY+DtZu7Nw+IXe6QxKVlCVF"
        "nixIqZeAc2HGno0wXiC7OEnlEnkd2EQ9W01/JzfkN5db8XK4moIFHd1rAUSHjh1rbh2/j9m5M1FkJ2RXszFc671647PeqC73"
        "Gh6NHYx3wpjuJIXzckZtnGM7tBNc5wJ+HGRJuhlSNJzitPQhL55yZ4jI6LyH+g/1sMt4B4GzbyYeZcu+WeKaXxu5rQhWlC2j"
        "tNK5QRQvBiioZJCRdj9tivPtwf7hX/DPFJaDNsCM/FaAipqqU1jaPRKTuYxjSswfk0Dwdnphm+zwGClDFI/OzCRhGkopciZb"
        "2b2ba9rV8pCUBnmQQom7PPDeHswoI40gA+TBNTQZrGhhcVmGHb7SiqykJ/KipKm0jWeFMTO0LfUP5YJaw7EmipliL0mL8NZi"
        "ubOkHv0hUv/yOqmQACHGMaZmjXnrEXxkCDZ0tVSb991CnQ7wsxUxNCJGJL/e9va9M0zrvbcg+0TdTfrKCBAWjpX2HSIg8jHk"
        "YmbGbLI1blHaCaOz1PXY+FzX68rap1aHqx+3Vdf4E0TqVG0r8sDpcl2ZZPe6sKSKyldBBqHCQZqyklszk57bPePxkbfVXtY6"
        "WWA0jQ3dqOPAQH2jgOsAt0xK+7oNO5/fVzJ4mMMBQ+BzZJoPwQrn9r1wRo7YaZ05E/lrntucjBWZPi4q8ezIbV9qAwPc9KGs"
        "n+sDIPBx2QV/tplOxQ/iiOqQLMldW6Btpl1ZRfOvyY3EJPOiWa1peVEl9UbweUjFAZvZ/L6NYX+YYZWBF0bivmJ2N9BmKM2G"
        "fgCrMi9OanphKtwbTFm/q4ciCnk99XMNBoixyXK17It1tLIfR3hDnn7TtUN1E3kdoJjOC3NhYPqNHGz328ZOElELNAuqByo+"
        "M6gqdDhM6nQz3nI7iraVXwczip8Oi8y/+un8+u+n12fvtxZsBzhHW2vLyqC6bFca+85wRq4xz6DEatt78o6rFfcoL+mtciOp"
        "2xagben7URH6/szciEDNfG5rLdvFn4Onk27B32VafrCgU7OXF0SRH5hNKI/nXiX0itPoS04nbQN+eV018tWFurM4XAnegiat"
        "+c11Bt1Mm1jswDTqblqUsJMO49HBwavrSZl3Lnp39PoaOi0sCrijtnRUXVTkaxsaXENyS4edsGlUm/bz5w/avkxCQE7Z5C3A"
        "TeaDnfgoaC/lmihKz54Wl5c94H+XxJIjOpKAqUWdIAoUD0bek34S+xFCIX/3zEj4sF6E53lmW47NbSXRwUy3kMBun8lpRfH0"
        "hc1sJp7h3vZ6qr935+VNnvzcoNB/sZYGuL4XjrUJ21bqQiPhsT3jI3XBCO69uvCpEwJszoAth5OEmvgaXa3YjIsuVLomLl2A"
        "0CWUzcOo+RFU8LGtHKzXm3GYp1+OrCye1xLAmT4Y7Qq1tbapOXsv7lL2uzKzrgUz29VZmQ2aIru6qDubILNhc2PW9SJmO7oK"
        "v4J23DiYDZsAs92VvUllbLMdXD+/tGV0rnNjbsdDmlD9W9dhaZPFGakjn3EdCN6x4ucBOgGnPYjpXRcD7Va3hJ2c5/OAJypI"
        "lGmB2Z2nQ7ZtYaNMc8SCjTPAVp/H6yO6Q8zDehAddmAbBo/X8bEF2NsdoHlGdeA+TDW+R93WeoB4hHUe27lFZzcJ0iD8voz2"
        "GAQdA/27txpFrN/e8mViovN7Mr+5Nj3rpqkehh7RTSPlaiJcy/DBE1fmypFNjcybPKRBs2qg41gFQ27vQa9Or67OLs7Nxedr"
        "dk6RPqkNmqgAxPnFdYevQ3d5/NPHi+MTcX16c82A+V5NuTSyTO4oPhUWSRLHklxw6+WAJAA71PIEZG+eyxhPnMV6G66cDRa6"
        "woPJUUOuTVOiBJwoSuOAJsg3+uZp1rv7tqILg9zigRBfueaFXHLd7gno1iqWfJep+mKWkc2iuNYU7+koevkQ6i26KCYGQZFS"
        "Dd0/j88ppEpAQcpez52ySbADrd0uHx6ZRFdc9cG1G3gFmD3CALl2Ea+AE2kpZ0m9Zrmmh3yOaZX3KP5TR06/q2lBuMe7vaaj"
        "qr9IY2qXdHzuWPAy8KNd6DAn7htj8dlYOGk0rNmiglq8ZqgrH8alQ+zc/uP48/mduOxnxmaZsnqk9RbOwUy8kBGNy6nY6dTS"
        "3K206qnNRpEaZ+Z7htYYrteJ2oHsKaHqTl/y0216lYS6inlaJ+CZTZzvYeluPyZdVlZdt5HtNKjOBNn49AcisFtG622Fxp21"
        "ErVst8ux4uFOnBdb/s3q3m7rNPKwyQ2dIGcUlFtubXFM6WRbulJGuSOb7DIavv+S4FCaS7422E66biqNIGnhNOc3egazcXYz"
        "SH/GKiY64igYQ480LS/UoX7uyOHNOb3biu0mS7i7dQBESHz+UENTyNo/RrPjeIYsDmvbN1Dp+fZ3Xd0nXThBVUCFXS1xqrV7"
        "n5NYMAU0Bh/9Mb7+sTTKRCxU5yu6vaDcNJ+zwzQFMXE3JWr+f9OmG30BoiiQu7fjuxRznqMv13Q6UA9zgZGb7fVwqfXxSnqi"
        "vEBRieXqu5bvukVEE0r2X6BWWUmfZMnI7dU8qNpRNdDHJc8klhf6vAQM3iypt7Dc7FK+EfRC3DzfePxp3ovYPG/Mo7U7rQi7"
        "v4ozGq8/a1N/9Iy0Xv7BYwLXtnwYFxTfiFaoLXgrbrvujuqmUD1uyRDYjEwwi725K7n8EMBidglxAL7QVtum07oGtBLsjJY/"
        "sFw5w6TcGSkVguEuXeuFTS7JF52H6U+xhzKGSN+M2q+HWngC6IfgIgzuMekYjTffYBzMD4++nYmD5eXxyUwcLr+c/9gLAM5I"
        "9HRlPPJ/vZjdXagNZe7s+A6JP5jZul3jq6moyUrXAs9ErI8or5dHo6r+n/lJkUtPXDQ1inUOss+9nV/2nX4B73yypVnv065d"
        "X0jBCji3EINqpPvcyQaU9qs1Ov8Jwpbvk6L4Pl9s+z61p3zfWRjdoF7V5N8qAZX5ViwAAA=="
    ,
    "redteam_cases.json":
        "H4sIAAb0jGoC/2VU0W4iNxR9z1fc0mqBNDBMtBu1CYNKSLqJShLEZreVMqliZszgjccmtgeGLX3e535Av6FSn/NP+YXeOwPq"
        "7hAp2Jh7zz33+Pje7cGdfwA1+ySFt5wJx+2cRbyGR+/Oh+eDW2//mzcd/Nv35szapTaxt7/v/Ty+uaI1s9xY2vx6cT4+p42I"
        "A792f/AlrNKtp0wjMqGyOBWqDjdj6ORHfkAf0GpBq5JjI8GVE1MRUZJP8T7vBPhfBefW8bgM6l+fQaOkvV38JlHdfAO/mTeb"
        "O/wyJSIdFz2/PP/18vwZXp7/eXn+F5fP5cHL89/lvlqcucwwSZl2ppeQcnAzDoUq4NhEcljOuClPt/oBf8qYtMDUys2ESiqY"
        "EbOl/HzIBw4+8LG91NeNJkzH+grOsr6sJmgVMUcp9fV62ykVA2p9Q2Z4eXVJ/a/X9f/Tc2s9u0gotYurp5XULA4QcCpM+uA/"
        "9L6OpRtxKwp/9e1R58RGRswdbg9PmOTGodjFuffFD18DKJ0y85jNCeIjW7Ay8Njbb7V6XQ/BJcfVulWxOp47ZjijoyKwRyxh"
        "w3JbssIxZo4VDYmpYXgf1kQBnR0TmjdzqTyZoMJHrw9GF9cfo1Qu47M3s8nbD6t4kOS/3C4X0eH1iv122hklQVABn7Do0Yno"
        "sSyQJgV6joy4MdoEDwWnMMwPf/Dp88eqgIovpVDF9XYZzAyfBiEJEbqtFNuuwlov73qskk/eSEVe5L+LxmLkev0hH7tGrKMs"
        "xetpx6iwUE1UbGAu57eV/LnRpbmEQlc4YFCWRa8m6FHmYK7n5EwoeMBE52DZCk1KPZZYHUyfcCUS5aH9Wo7JQo5bdPjGe9Yx"
        "x4kNGO6M4AtuweilPSEvQjEpwGmYCum4oaeRtnegtw9STKFBDm4bjY8pCKC+GSDrdeHstojp1G/CH5AYplw/iri1jeYJ/LkD"
        "anjCC/FGzGFpdQy/37HWp/vvfyqXMGw3Ip2utUma30HKXDRD6qm2DjjKKu0uTzJUC9Un1Pe2fOjd4uX3UMYY96ZH6lrqeJIJ"
        "GaPokq105kAouLi9Gu6CFtOymAGIZhm2GNb8wAeBVyOXbIVgJuNhDeLM0N1QUakTEYHkEU4kvgs5Z25GiNdsIRK8HqIzOA7D"
        "9zQdwrBPooZh6Qb8XnA3mQLLXTZvT3C+VBHTDeJbseAK8joEEOde7A7AarngMNUGp3arKFaSK0dhn2b5KUb7uywVvljCHGjl"
        "WOTgpn6K/lHwCndXTEq+guFwhLY6BhxSzlBMNzZs6mBx2EO8vfu9/wCx2d3W0gYAAA=="
    ,
    "redteam_round2_cases.json":
        "H4sIAAb0jGoC/3VV3VLjNhS+36c4DQUnTBz/ALPT4KSzhews7YK7hmkvCBeyrTgmsmQkGZKBfZNOn2TfaV+hRzKBJZne6M/n"
        "+yKd830n1+/gOuhDR92x0suIom4mqopy3cHD5qKMube/76kJm2RXZhX0w/5B56b/IywXTcqo20hmQLvhUfjeDH6c2CnA8eA0"
        "2ABlcyJNeABxAiefPiTdw196o/ViI5g3jKUrTQ2A5FXJnV3fN8BgtMmrSepy+sBKbqODqY6TKd+OY+XCBjiGhpOKwuezPybg"
        "7DquuxGqmlRRRjObk2DUvZx8npxcQQAfk/gc8oYw+PvTJJmY22zefE6XM7y8QfrL8H3oH348CkP/IDg4Pdi8EmduLVjZPjMT"
        "DcthJRpYlDxnK5BUN5IDYQwaRSXuMyFzBWXBhSx5AXpOoSZKPeAxZHOaLbb5lRZyZehNsM0kELWguQXnBJOHEgAtkP2e4rtw"
        "lCswKIxZk7/SLpXyqnJJc6scem8zFJXnBVzKk9ESYj6RiUhGhFGpu0Fv/Baq7gusldI0tzjcjiOVybLWY4vY2zn0jwMcg+PI"
        "e/4QGdQmj14xm7QoL+/B7kbTTkqyRSFFw/MhSrN7S+5JyzFcX6c37Www5VSTkqmWq12DqCkHwbUoCuT9n6dURN41tL3E8xox"
        "ShOpXyHLaB23gW54mYmcupoUhuH7t3/am37/9u8ajGfey+FbNNY1p+Zban//QaKEYK4rhlUlGu6akmoUEBME5UKwtJXAgJYM"
        "ZlJUQFAGWmO+UFfKCPBtUlAX6O7SPk6kt+gEqxXMsZmGmi61Z36v/6Z6eOmXqr0k2keOlHIUrXerBDeUj9OOEfS0M4Rp6+4p"
        "Hk87mCe5ak+fHbffOk6338XCfNSyoV+3uFHvrm5QtiWx3rsScCtQ7ChwRlUfbI6G8MxLBmXeUhP4PT67gBTii/Z0BClO2/xz"
        "2pq6kLQGF33pXMWnsQMDT8kMnoA8LMB5rNGYGn4OvjowRlflQg30Um+xPXfOv0rMPRZO12roeXRJqprRAfZjT1Eis/mvd6OZ"
        "EHs1Kego3JFUNUwrmAkJa9luEi8rSxxxLPg40mL8gZUZjTxcRabw499EGnl2FaUiX41PTHupMMLuIs8Ct2gzZR0ySDWHR8gE"
        "E3IIO7PZ7Bh+MB0Y1zleWRXeclDzwukdw3alSC2UlqKe09fmdKYdBZdEkjnOuSiOIXYSWjK2wn0qxAIUWSmUQC64g2rAsjGx"
        "ndaK6Hlo+MoZdAns7UHag6cn6GZm/VPeM12PQ5tIrHQAlGH3e9n7W4xMWH9eT5IkTm4g9MND1w9c/wiCcOj7Qz+AC+z2f6LU"
        "NFoJzWf+h+AwBNc1DVyusFFvsaKJrHHPKdWmkQsOfuAhqUSJfgkgcsfwJYS0yQuqMcUv3X4PzVqYB9RC4vPf3bz7D3Y5gHrP"
        "BwAA"
    ,
    "text_normalize.py":
        "H4sIAAb0jGoC/5VVzY4bRRC+z1NUZg8eB69XFhyQkZE2gbCrJOsoWRQhhEbtmfK65ZnuSXePN2a1UsRlOfKTAwTCGyCkcELy"
        "iePeeYG8wL4CVd32jDcRivDBLndXV1d99dXXOzf2amv2JlLtoVpAtXQzrd6P4jiOHD51qdKmFIX8GvvVEiA5LCujF1iicvCg"
        "EApeP3sOx8LO4YNuFI2PPgU7EwZz4MO7WYFCSXUC01plTmrVA1lW2jhymCzh1vj4ANwMwRkhFZooGXyYeju1TmRzzOnSLgiV"
        "e69CLhBEVUFCX7zTh7uIFceXDqQCrRCqQmQI0kaZkU5mohiCnIYL2HFdjkXrM4RJ7cCiWfBermlVaQeJNrCQGe4u0FjR7UUV"
        "VSR9ARasLKj4YgkTg2Lej6LHsyXfj0+ldXboM/1cyUznlKzjMiC+Wn1/tbqAq9UfV6tX9HMRFq5WL4MdAy5ETqDk6NDfAxPM"
        "RG0x4nAlxSrAilOo1VSUspDCEKRFcSpzN9ub6VKfFMtqBhlhLzJHWffh6M7d2025wsec6iK3FFFbAlPr+S5tzanmCSfpNGNH"
        "KO4/un14SPdPtaEK1BKmKFxtGFTIdFnV1L0eWM2VRusKCSBLqFiO0mbMlhUlwqlgiOw6tDulWxhsAq+hV8rtSLp8CR2VEzTC"
        "IaGcEebUHypggUPGWSta9ZW05UY3Da5T8JXam5A0+JCzKW0PWpie1Nqh3cuFnSFtcFF0bEK4umXUYkj8OnSBFUfjY7DOyKpH"
        "wJ2iyYTFHhBNDGqTo2Gc3IxJxPMQED7VdZFzOHWCHgpiUwOm9Suh3T5MIWlNq76fvDAk1G1Po1w4EUU7cLApwIbmNnwVBVEx"
        "D6hwBzzMfbgvKn8L1VdqPx0WBDzev0OxiG+WKeGTBCIao7QGxg9cAKcfpQfj++PP7n3x4OARjOAsAvrEr5/9FA8h7sQ9tn/e"
        "sl9s2b9sbNiBrDbUNksIFbi+ZhPqVXPk8q/WXDXmPy+24lRGEp/2mGByKhn4SlNjdDVrA/5K/p2443N4uWX/tmX/ubGb3HJd"
        "T97K7Ue+ejeU87y1v/1ha/27jU2hAmqcn1T1OsrfvzcO/++zQ0M2dTCjjqMKsa5W3zRFkHQ0wLwrUDsM6xYnEyzcexNiOtpu"
        "dB6lxw/3j3yHic/JvDuEhWfEvEcGjesWDfrSYWmT7nkURTlO4Y0R5q8hD0sXdj/m32FIPY7vMD030ritP0E0pFnrAz6p5UKw"
        "yrJC8GFPdxqkvCDvtpZ7NLUKEhJQvmrQY4Fla8zWBVudLg2sPAkj1/PELokvPui1sQ8i8ZGfl1YonGBK+FfMQl1dF46QFe0t"
        "1nwRBY0YCa8OSkBKZrCkxzLfEo18WzXofdtg43/l1I+zpDGh909l6MHseTADivzx79aIF/12N9pa3FKMftOXJOZMiSZvufNP"
        "n95GZQtS2ySQIHgYJMyU96A+U2ZpqkjK0xRGI4jTtORXOo1DWjtAjaDxxmK6SwAFfJk+ogcTps+XTfbJux9D4vagA+OH0Bl0"
        "Rp1BTA9we1rkNFqXK96+XA0uVyP65hN+/T9PDUhmeG8wGrTh+c81r/D+eWwI/6Vl17fW1ie+ahtyohnLN8ZAdJt90izlkml8"
        "1tFz6HCX/YkRQYMFKW/n1v4nnXOAM3HDnDNrz2ifzLgb/QsYGZnMngkAAA=="
    ,
    "webids23_to_honeypot_log_v9.py":
        "H4sIAAb0jGoC/+19a3fbtrbg966V/4DLTGqx1dt2kipV5zi2krh1bNeym3ZcLy5KhCTWFKnwYVv19f3tsx8ACUqUk/T03Fln"
        "ZtzGlkBgA9hvAHuDT/+jlSVxa+SHLRneiMUynUXh9pOvLMt68tWtHPle0t120siBYrlcRKkTRFPnZre5WD756slXN7ti4t/J"
        "REyiLBaxdAPhJ0nGBaEnRkvhZp6f+uFU3OxsJSLK0kWW9rBtpykOLk6PDvf3zg9PjkUtnUkx8qcCOrJ7UFvMpZtksfTEi27z"
        "u2cimgg3CEQc3SaiJXZ2uezJV0IIN03d8TU/chMh79xxKrLY/zaWH2EwqTOKvKXwskXgj91UJnVx66czkURzKRbuMohcLyFA"
        "tXmUpAFUjebOyE2g71+HQxsmtpAuTSJbiDQS29+J1J/LpCnOoigVYzdLZI/AEZSpDGXsplEsJu7cD3xAx8z1hAuNwqUYR/MF"
        "zDFMRbJwx1LUZHPaNHqEqgQlCmEg/9XpikWUJP4okMLzE4AAU2MsJrYYSRxU4s4XAbTcfn7N46L2fujBsOFXiDNqNKDjIPAT"
        "PwoTcStjKULpxo1p5sZumErp1UUIc3HFKJtCWwHkIDAEHHsBwvtj4aeJDCZN8QbI7on0NhK37jLpUVUghC328+ktoihAYixc"
        "GIQnkmyUpNCVD1QEthjHMC0hb2S8ZDQtFQwmA4zvxyHUSnG8dUEFqTtNWtAiTHXJ8OcjKAXcJC2YXTYPk1YBxZMBDK31MYtS"
        "2QCsz3FQ8HcUQXOYNKAcqO95OLmcKIDUJMonTz9MpoVU5FwCiwOqbhWSRCoBocCcIDlZAlCTpmo6ssVF6AMDhhJmqhoNjt+c"
        "nO0PDhjbs2gBmJlEcU9IdzzTnCO9onfFn9h+PJPja6jvTl0/TJBWcRY2bn1PikTKsJHI9BWwTUFooFYBKJZpjKxYYx7u7qLY"
        "yPkCp0zi4IpJLJOZODt+K7zYvX0l/IkAjguCAgiBJoy5kxQwks7cVNQSmHoA0i4BrUCVqQ80EjXXtusAFB7GqcJ3ASiMQkAq"
        "TMpdEJN6OKacGUWWIw5AgCKhfmJEelLAmLsxosMC/I2l53AbhyBbPZHGmURaLqMMJDRkRcS6AWg4i24LQBHMhYk5AeYcoS6Z"
        "+Kh5asC9EgTuv9p2E3VWFxj/aO+tGA7e7x2fH+4Pe6zWrGQZQvPUHzu5lrHEJHCRs+JYUpdp7CL5qNtJEN02AuBlUGgStAfg"
        "0AVl9GHwunF4MOxui/3hLzRfLdvprT9GtHgyZu0AgqNFE/RAloIcJmLuJ6CEPVSC1BsSSjGQ6k0PD7ijToBuZz4wHhIC9Mtk"
        "AooBpGQuxzM39JO5gG+AiWDZFKDrr6VcMP6rJ0xsBBpCRLE/9UM3qOvpozIPiW2AmCBzCkoV5VAciBZ64AUvFORxR4CAOpCX"
        "4JAYgioaI4AYxBG5MIxAZYVTYFIoBlKAVDH+0MIAfWXgEU23m+JiODhr7L0dHJ+Lg8NfBmfDw/PfyASh2n4uLvaAC7ERSDmj"
        "C3Ed+CAAqMsUQj4Gc3fR6jRfNDsdqykOJ9AO8YoqDqcsiSwgZGBKwKyhaMwj0FEwPBAeAhKARg6FdbHX7zM40fgBuBzQ64Nu"
        "sQRKPZIXxqHVQgJsPmZ4NDSmKYljEEXXYgr8kKJSSGcwFs9NwcakRIWJ6wek9wKZStTISquAvDCLzQBDDTA14ujofSPXTDAf"
        "l3QLwCGmAO5NwE69RDQxB9xKfzrTSiy3HyCKUQyqwwWFP0JTDWjBJqAWoDCVLOdgM6i0xghgHmW3pKGsORAjhEc2qQ4XcJuS"
        "54HGcQyTvkU1hgwzXpKSJQjAQ8BWc5xmKjqt5zBKVNmxO5n4Y2KDnaYY7r8DuRbHJ4fDgbZoDdAUgBRsDOoS56EUuRug4ROW"
        "v+gtQMFZovayTaw7S9NFXezsbOO3XMdgKShag4LoLpC1hsJxkJFSJbZHdYm4KCroWeCPlg7SF4enzXyclnZ2fM+i9lYClBxL"
        "J8OCmmJcNPw++gQgsgGaMLLAOfRcmBuARIn2xSZhAg8oACvoMYFR1nJnhCQp9wBQqIEGqlNUQjB+EtmUzGquGtZ1XT0HAQJL"
        "roL2Elxwm3KFB4MoPDk2vEBT7NTGaRfjgFoOOkF3MJB2jrdCNXW+pUIT8DhagIm0wVvKwYAaAUsAigPHk2OuAXKHBCvUyADx"
        "CEIGhTIghQ4KC+UEaAs0x9p18WHvTUPLTzSaZInSxCCm6C9DryDOgODAHQEQ0J6BmySNkRu44ZhkKEUZnAJjwycJ3kg4bYHN"
        "lvFN5Me5qAEikTqxD2YBewLiAU/ZyLlZiIp9ik5HDO7PDXI90JmMmBeN9awYV0++mmTgcQMXziIvIv8P6AbMKFGhKA3UOjxF"
        "eqVoNtgddRFSht4WqoDcUBCiwSMTNKISAyDxsBsX9VNTrT6efOXPWRbi6cKNE5kXzNxkFvij/LuPXhygIclL/kiiMP+iXY80"
        "Xiou1Q8kS4gQT9FTu2ugy/0KFRoKX4BKMbdF0r1unA2HCpdPvpJ3Y7lIxSFBGsQxunAI5wNwHK9Q9iOgIoBsqCLuOu+zL47B"
        "pOSDTOQYHLRiBsmy+IxKAGiBBEP/S7JTrGexD5yToldAFRZuirjRD0/hq4FKdMVdWiMtPCzef3dx/JMzPPxfAxhPt+202+0n"
        "Xw0HwyGsyJwPh8cHJx+c4WD/5PhgCBU6XXh6gd+P8OulhUJeF1aa4G/fa6JsOTP1BWa64C/In+C94EdeqTnpciGtqydfvd/7"
        "1bk4Pvz5YnAMnTpng/OzwwHC7u7i8J6CIP5dPwgN1iYjtDUwTuA/bULz5VpC/h6vVsblZYz9dw8G0Pzz0aHz88XJ+YCRuQX4"
        "2bK2rtST/ZP378Eh4WeNBiLvKf6ChRx9aX1z901LFajFDRdDqYZxvvf6SEEHpo6JTO54jAzDnz3QIPhhDCoKrA3XmMv5SH2E"
        "5UEQLaWEL4W3rPUrWHNVbRFHHnghCROb1B59Bm0VJmoNR/XcJY6TPoM6SaxiskfDAc+V++EBh2DjuF2S3EJ31lUdnjDTyTn4"
        "L1ygq7EMOddyCeUmHA3DQc3BbcZu7DlhhjMlBNzccHmSEEK8aGTASKNrGRqTQxNLtWPQEgRcxuB687yxmZ7XweBo7zeaVqcu"
        "unWxDY5BXezWxfO6eFEXL+viO7BDbfgHDztQ3m1j03eDoyNn//0Bk+52FoGmZ6nC3yE4cHpWns/DBwvYkum4RZiiWv4C3V5/"
        "uk45C1bljcClcYMuyO6IVCCRYr/3O/yMoiht+iH1mCFuRcNFSr0+OTlyftkj2UdvqxajJanBzLbbto0sDXZHTlCenNTNUrQY"
        "y1oM6xOldj+CQYO2UNIczyLQCDVDBmCtuPpAi4DNzW9AZkutL60OcTGzWkaUoMn8ib+WwLVXqmmSLgO52lh+xGpR7MiPziKW"
        "E5/aAk+AsR7nTfmJaoumxA9TnPN38KOq0DKZOugLhNorLbozcOkn1v3HB3FyJu5hEg99+n0/frAq2uv+K4HwYB40sPKQYDwA"
        "eq0MwevCaF6zv283t9lF2do7/m2LxiF+LzMJNHnjYhWqV3S8cQqbhrhF9bb66i83MBkFHHOQJ9r5+htZZWXK4nvRbu4aGKU9"
        "owr4rDDt0n5HAvWsutX8I/LD2nq/pLhso0nOMOBM1aqYxq4c3g7j2rI+QXcwlyfHYjg4Guyfi3sc34N4c3byXtzTpAyahCtM"
        "CypGc2wIfl1pXpfW8cXRkXUlvhGhvUrStV6p+cM6MdFRYVo6I3BE/06KehVtWbuuyumXoX27jPZPMU4lSfaOD8TwaDA4rd17"
        "D3ZBgFldzGFAnmi1xHNQ8p54Bn8fEZgPe4fnb0BqaGJiq927nz307ue9dterkh1Q1IF0w78d1yGgLOyWAeTKv9y8KP5S/IWd"
        "HHf4uU8FFfpkvV63gvMkeuB/uxqpNBvk9oyWZHajENY0KVvghP7KO1zzpqDt2B5lC1jTyLt58JcNiu7v0wbhYHAmXv+2ahbA"
        "OG+yNXr8mwwWIX3FoICNgfXAL4Oz8xqU1GtKJ/zjH7iMBl1u25s6Qwx9aU97w/Paehdibyig0qaeCpw/1t3F6cHe+eDX90fQ"
        "F8xof++81r57IfMZ0T7gyXHNtutYbtc7dhV/KmiDX8/P9vbPQRQuBrX1mVT2YODMXmfpJKXtYudjJuO/0436PP77Mi34Shyc"
        "nZwKMp9Mxwqr+lCNvleAu8G+uFs447mXzGQQ1LZKIHJv2H7YqsATeJozZ7REz9fEUhWKVjCBTvSqdOdLohi84HyN5NPGS0S+"
        "dpKBt6/971Si2K+tjQpfe+6G7lTqhrRTiB+XSd5RtODVZ+GsZpOCQPnArHq1ZbPtqzWDjaN7uGc4puEgJLyFtfbZ3vnJGfrx"
        "99zUyv11q1f239XcLMNPU1WMEl1p1f5rYCvFunrJfqm6pTJd0dDuqppRoiuV5EVVK5XpigbDqGpGCVR6QFT9OHRAnEH+jUWp"
        "5QagLWsdm/iCPm/dJcmW8V3vfDXHUXTtS1tZiYkfz6mdAgQL5vmiDClvCSLn+qHxpGv2t7gNtwo4GnQxDAV5dVy3tAfVDCLe"
        "cCxBwK0ueDKtHDw3D90bf4pM2kTW2gOcpQUIroKggyFUAXZvBjKcpjMDQGd7+0XVZFM/DSSBggXmr8Mh6Iq3zuAXvfHBPdQs"
        "fz4lUQmJ7rhRIpJ43L/bAnVXs5Ib9RS3c/KHHXio2+P5f7kKt/TDRZbygwkMKaHGyPr0zQAwd2PgIclVgatIjBWQG9+T0abh"
        "aQCeTF0/SLhaGk2ntHGwJUD6QzULiTt7nzMYsG0wAuk+Vheq4YHnJ8flT2K1e7KCPncUZWlvFLjhNYPDIxA1/2w09zUCiHKD"
        "X98cHjkHJ+/3Dllg1EafjJvyjvbWyTe68QPz+7hbfKtSoZba7YzKUO4mZTC4F4FbeHlZPqDTvfN3PBwFiXpVu0/4ZyRdEAD8"
        "NI3dEWlmP7w2AZztvVcQaLvFTQnrvO2pd4XG1MK0SWAVsbuyM1rpTSbj2F/QuFJ36lBwA4lJCDXYatxMHVXpqso9VM+qrPP3"
        "/OwH05oWes1++L6lKlQ5UcVwSuvmaV1QaR2jB1asZ1l+7aoRwSJ1+nCPTR/EPQFSWxYVw6scloGO6infTH/47HnDB6i+4pNs"
        "fe+KGRCwb/3h3rhcs7cJlFXqBLgk8IEb6tZMxsCbwJohfrvx5S1QDzp2f9ha9fKiuMwmrP/rtJOPv0E+y2guCVvZ3zPYvvoB"
        "svNjixvW/A7JGO0izkGdF19ZRzhanU5kOp6pp5XMWQL3GIvmBkFbqP4Wnpj2Wq17xsdD6x4R8vA/7wkjD/2tb1fM1aPcbM7j"
        "sXGE8lYcYt2a3UQt+PeOwsTXY6OgerUv7tte7fwviOtfF1WTq/Notk/qP9yRdwol+EeSxbTxTOXK7HMdMPKVPGZCqELq039K"
        "DfKAquB+jnbYNFycWPVY4QnbaKGs9ifVoznRGwyfRBH9JKWQ+JUrgdx0Kd84/1742aixcgcbv+hHOdnV0/y79qpfD44P3x47"
        "sBIevD1RR32XFjk+cRT6Yz6VWgCDS9I3s4h9kzRa0iNYHVzzKROupJLNKy+07FlKSmrqxh7bafSP5lHq37C/g6Efkm1/NMY1"
        "Ax3kPLKcs8Aj8iI2yxPAuY9xN/jl2gd5VSdEKakvBHeVz3fv4EdY7x/+ouZ7ixFVMuEZqX0iRCm573LuZ3NeNswXLrssOFnc"
        "3H1kaLduKmNw/qMJ+UjxNAqjuT/mFewfWaLaY8hV4H0CVozRX/FU6iYBhvNwUA85P1msnyBb+8nsEVjuBPxGT9eHAU74CM0l"
        "+lJ8A48SY6Hi8BFIchw1JrEvQy8gUs2ke7NseIrKc3VgFeDAaWhzctI3gvP8qZ/yKG5gMe1yKwzyYZ4IoiQhyGHkJ7IxBo9M"
        "Bhhd8QjMiZukDcId1UNhD72567HfGsipy95dnE2nTO8kkPL6EYgUm5L6jL1R5k0lk0AG2R2Vwbouw6gNngliYU6hUI/AxOVD"
        "Vua3Bri/njQ49vjkQjny8wiWfKSa0Z/n3QuMhlB7GrNIJsytk4mUjbl+4MnkuhGAP141EGsZTV2oy8yNoeWAX+6F+LgxitJU"
        "zRl6BTG4VhT3qHal4FvXcjmKQNiZm0A6WVKZl2MWNljN8BxS5WiP/PFyXC0NOJbxjDd35zLmA9PYx2AL3iDK1Kcbd5yxzAKb"
        "Y4RhJbQ0chPVIJuPQAPwEeytCysTRmsWTlEeGKEYZoPqrhLWwg+C6FaxEh9/+ovIM4jTwAAqT3HyDYNMxm48qYSXhNSIKZlR"
        "nFsjx+UicPWc+ZMOGZj5UaXAWlEMnO7/CW0Kjnp9tnesDrTdMWv1PyUQacaTveHVbDyihSXyPbApWwP+G6Cns2mdGEvefQfu"
        "H9Fi2Ip9T+mBbM6LVVAgM1oOL7KAec2L/ckGQQHJvWGFDupLAVr4rO0/ZqBZ/uQYBB0+ACjGuT4V1jvAmghB1MnOiBHMcRqK"
        "xSx2MU6y0TBiFRPpxuNZS4emo1xSMBvCAWXhpyDLFJqJ8VUYnCmQl8EmSAxVcRcRCH60mGE+AwgaaorQS2jJIEPcyHpKgSEu"
        "Fsl4TBkHMBb4mi2wAAZkc/Acx4tGKaxabkHZiZPjo990dKUfIiCVYaGTJZrig4/x7jhemVAsK6vyiY/xtm4ogCFvONI/keiv"
        "ppLGw/yNeSH5VAQFpAYYX8upEFY+XVdQ8L4lorgopeGExRwAVSmyZym4kiJpKRhtnGZuoMePwWjuHVJB45mipnWiiedzqHaP"
        "+3UoUoZiKRWNHFrzq3KKpWw3KWbqKUWKddptylNRRKd8FJUlMPHjJBVqv5+jFv2kCD2qc+jiU5UQsTo5jgelPIQCb0wgGDaR"
        "C/rhSSIUZKWmOEfaovkRZ29aR8Pz9xhjF4OXxaHB0MUEQ7Nrexf7jbOTfdHBudhF9B9gB7yyKXJsEoUID2ids7K8Ay6GQckx"
        "UH1MpE4joHAEdYirpu6imYv/u72zA+d48HYP3SDn9N3Z3nBgbq2ebL1G276VCBSgW5dF7RzNUix+jOQWSfVrGYqvxY/gGC+5"
        "YOgGKZScSkSYeBuDXsz3JV8Dar4Wr2H012iYRe0gug1T+Ecbke/dMTzdn0nJyuAsAv74WpxFQSB+8cNlAec9DessC4mhhmjx"
        "RG0I6g3oTaA+gJeKVX4kC1kX9Ox93r5LTDERQ9BEMEgYMnX4OluKTl28Be7viDcxby7+BF0ApH2ypuId214F5w14WjRkf5FQ"
        "KFskakdo22gQx+BmoT6YQ/OTLFU2Ze/863OBu7JSHOHKOsptk3X69VvuYAYeoXidgV9FQzhMt1DyhnPMtvoQxfDwPFqKYeEZ"
        "W69BuAAJX4vjCOMo3oIaFfvKWgBegTbidYDy9gbUsEte1Luv34uffA8HriMYNbinguInW+g6ZyiNSQLeGOnKBLyTcRRQCpD+"
        "68ECShZqDgN0hz8ftWBBI8ADwDAxFWppEVho58me2G23Re2QrBeog6GMMSqfKtjckcQEp4SYX+3VNvP5fmBl0qPocljywMxq"
        "2yAWuCzH/ItX8JmOaeEvawpPekXzIU2sJ8BgLjDfB9WPzs54uvOy22n8+uMrMTjfE9uNXQEGzKdkB8CcMYZjUEg9Af4Hps1Q"
        "wDigCJTZKwZLSUxuMM440QHEj7KFQEUb41ARyXgeg7obVEa3126/EgtQlwmGlN9ICv8XHCRs9P4684EPnnbaO11KHODcGD8l"
        "7IoOYAAxuH+ISKzG4Qnh5+nuLsy3J7rCT+UcqPc/Xu40d9ugOcAXf4VB5UrnLhaBb+AQYzc92Qj8aylaoN7Gs5CCkSexO6WQ"
        "QkTqiKLN6yWucWPUwnURLZL8xAN0QV98TJfiG8wLSB1CKmU71ejhD4I9bVvcUw5PDab3UD5zAV64VwlVPYzqUyiFz+2HV0KG"
        "N30VE2nwOXuMPdFyF37rptvi+ElM8cBlQiDjV7iDPxMYx++Tf67a4VmgK74Vo1diCR/HoiG8V3rdfwezWL7K66YzzOFCif6h"
        "D8bppfj6a8p/wch14Jzv+6jl89pIqZoPMIEJfPG9COHPt9/ivMFtEt/2EX2X/pUx/aegKZg/W2SQx8pvAfRX+CBikYVogFMj"
        "EF7TYV81B5EHblWHo/+gVW+4bMJftPpoV0Rtd3fXFp3udmNn9/mLpqGYJWVkoKucgDJudXd7xEJZgiHcP2+L7wG5P+8IikP1"
        "aRDozBYQTplIYvjTxVOx93q/AZ28AlWWymkUgxswKHYmQNu9dZEr8sWHdUaiBvzb2hV4JkRM+LvFKTeK/nVBtkfL6H/8bhWd"
        "s9TFEreHMQCcBrjd7NZcmz0ESZ4fmvs6VGPpByDF+A/DG9zhEU+77e5Oo93+rgMylEngQtIe4IemrwRqBIHmpNPcfdaaR011"
        "3KYs89ngLUZ7758cqO2JiyEq7P09/P32NVmRC/x9MMDfb87w94+nZH3f4u/Td2Q2jg1nf//i7GxwvP9bCeoBVhtcnDHcUwbz"
        "G/d1wN0cMLxTA9Tw5EwFQZOYOm4yVgsx+AI6cMwhsbdqBRBTni7vqATyBlftvIMCXr8b++mSj2qeUmQ8yt678/NTleKACZWY"
        "mDAG1RNqr5gdSHBuMEuPXSOgLmj4RQRLQYRUQx9ygWQEawoVdWopLMJUSk69yLK7xdajGNxIoRNEeEEwt/Mp4/69835w/u7k"
        "wNiYa/FyweoB9gbnFOuteMwsAhvv4uGWLtKNUeeQysUnpydDrg1ahlxeEwLnZq0BmPgBetlGRXQis0UBj/b4hue/HQ34oMEY"
        "Ow/dAQBzrP/RCBrHWOqeDm0mlegUgOl7oa1oCA6l8OJTNSQK7sacHlK2PfrGsVcJ1OY48Z4OGM+3KkEoHI7S6vE33j9Jxg6q"
        "fAKPf4mRphjpMM7iGJPK8BEddktLb2uCB4PZuzhHSYHUlE4JqibjxG2aBqgK3L9v8VZwwilqzFK49gFvwKN89qecHUfpPswh"
        "TbGXM8vCH1+Dg3c8MHrDxQGWJDNaDnCSUrok3qQ0MQp0ofnXOW9M4V7QZgYxKBWzD+N7NvkPqMYoOQmzLJ9iumIWpL7maZN9"
        "PUopwVmMoxlnkN74oHDNFdkxwshCkErSmkpIMM84YZGaUSYwriFoHrjGo/Q2vYd6tv9ucP7bqblquKzlUlEvMxmdWOfyUeY2"
        "ffj9WA18pkWpvsJ5dp4YUMulZa0OQSgeGgxqdr+hwiMjM/suZHpVdBhELt1rj00oZi1DYhhEqYvS02ISGyhg9vEYkQopXsFg"
        "0YMadr0ktn8NTavCzFA2je+TjKKSPfA/OnV1fr4YnP1G5soAarY22ao8NIMS68k2BqsYGNOYudLdH528PTzm7jFfL+ReOOXH"
        "LMHgM/7eXMwW+VjQ/6zsHTeOVFs6jAgUsLxjFVrHXZv5SMpL5IFL6Y3UZi71p9d6GzbhWuwbMXaUh9hKfdqHpp7ByjjHe++V"
        "i8E25GNBRt6tU0c7Jp/rjKFYTngzGKit/IIacbk7pUNTPvSgFaJOagYFdYtbfGu5zCqZmDLkzhzKJi/FML2P/vSDwG3tNmEl"
        "qtMEAWOddhO8byh4vvNK3D3fscUeLH7kBzn6yU9bu9svmtvPRe2nd+fvj+qCFkBvcWFni/0ZrPZlq9PdabbxPzF0J+DiqCYW"
        "JhMVcT7/ut63N/Te6W7o/dcOeKlHfpjdibuXz52/0mf3S/t874LjnEbJ7JXATYBA4PbPyVD8CghwOrvOi/IYnrd3m51mZ7dy"
        "FL/wJl6r86IYgW6AY9j58jFsO7tfjoTOJiT8BbLDcvKmR5xkcz+tbrvThv874g1IzSS6Yz57DDzR9WIEmiYr01cB334U+DYB"
        "f74Btn+K5z6wNDu9EPwZMdd54ewwbjQq/xoZd8T7aOQHgNTdQWfnZUHUnSZmmL3cMCia4yuxF3px5HtA+Ffi1L8Dwr786zLM"
        "A1mj6fPPIGnnL0rywJs+okR2i55XrmFodZvbtJkgtos6YFiD1ssms4ohjaXrMZ7ycbSoqQgXftiM4iluZXY2zJW2BlJ0qF+J"
        "Y7wcY0jRF+goD0IwRlBMtywAwFCBa+EJWitMZHOWzoMcOKj6iz1n/93J4T6bjozc8KwuHNwUMxQ41/wwOHz7jrX5LdV0wDSs"
        "13zy1dngzeBscMbmCC0LHVuqQd3e3janUTQNJO5utFYfjXBlrx80Vi1iXhX8iGv8N424svI+/u5U6QOJRpEi3/0xXgWyoJ2b"
        "vzsJGgOG4nDqAFJr3yzcOMVt0hT3ZH9QS4TmGf3RMUTgQQByMfpd3QLQhNVCd/d5zfpPlcpGUOwmeHeRJ2u23ZzJO88HJyOt"
        "gbPXeW6LZ6LW/eab7W455KnUXQ07UjmtztkAyPzLAM8vTJOeXzvQ9BdOKNPbKL6uhXx+EiJ3XBYktNosXq2XSN3Oyrd28zmJ"
        "X6vTpoLui+K5AaPz/Ltmd1fVfE41X3Sbnedc0KWC75RpbHV3iu9d9d2E9R22fGnC+u5ls6MKdnXBLqjPdnu9ebeNZr/T2c57"
        "6moN0uKvO+3S193dpvGvtd3V4K5yBxp5wfETZ5GNAn9c8xc9A8WHpzc7e/yZmAPzBHol+tHtZeES2iHuMUmaCCFpn7tMQyNZ"
        "ubjHiLt1/AXRHrPIC1aEv7oz0Db9nGUtH31nXV/x0+0MNfh5nEkjxGuMORu4ekHurZoVZXZMJW5VeiM/TWrAn0YIrV/CTQ7N"
        "7pXVhMIFZooUdYrZ6jtHpAMq16+p41yaJ4WL9spSsDL3BYxdNfm8PCFsYCmxHDfpULNWnYa5y2mY42YQ3VIlpN0YKbd4JCdp"
        "e7WzRVNdVFOzRH4bwiMAuo8AwPsYrGfdF5a9AvRZt22VVceiCsN3QNN/ewRvRo+OqkSM4Gf9ncr+Isq/J/Ru75so/4HLBtUo"
        "/7vN3ll0K2ixjF5FS6zfrfOvMYHUpYMLyRruEfZod5qYAyydwhcdkeFRDX8l+s2y8JpoiNetuJ4zTm5qHDcOq1jMDe/zJS1X"
        "da6b+H/KfnHpi6k9GPy3fRFI4CWs3AS3dhG6NbuMeapYMHyOIocvUTImUBcpHk+nPZwFq0n6aJd1aZXdVVflKOSDyb0qctq/"
        "DAHqvppPI4Bh9UVp7nhppUz7l5tut7kyVDQOKOZ7IRlGGjmeP05rGFCJx9LWqrYOEd+dtasXkAL55FFqFBrX90jyWk2+RrEG"
        "/dvlaij2FS3/WEkAbddhNA1wj9erwoj+eGwQpYFc/nGFkPXtitq30o//NVI7CFPciMEz6n+Zl+onkQNWGRFc9juA9ZDQoPLx"
        "EBgrXOKNSFd2ExT7hMqsZ781ns0bz7zzZ+96z973ng3/l2X6IOpWGXAhyKsw+pCLaDxTDu/mjuiWuBTEL5fVUYabZNCQAbRa"
        "ovpWp5XUl3sCWvD61YNzz6AejLwVDOl2+No3HAUZtno+C9xhhZkv63hNGyiFIGU/Kvfg88niZQ0VUxff5i2L+HmjFTp6frjW"
        "n8GZq48ujeZXxbFUEU6pXT+rV+kQrmIFRmhAtFeXapkLcIr4+6RWLDb1liLopXxZWRfX/Y592b5ahWPcNtQTxnKnUzN6r17o"
        "XPY63Qp4HwFQ2yh+KGhiEKMSbUXNSwJ0ZeguulsMPJGXbYtsPqJLX/p1xVkPsHq11FUbOzvb+oIMDsUAvnsM67mqfejdY0/I"
        "iqavS5fEIiiDQ1EVOFnsU6qSdsMUo2KRg0YCmoFLzVFzPVpRPJqqku8kU/pTZS6Kyj02b+9NZ2U4xjGBeWnMepqXsbu9nkqn"
        "DmExA7+UE3SvpvpAmw+KOmtjKDBQTKLAhGHRUMaRsPllXFizaI3nnOa5A9NX7/4b8qgz2q2v9VVe/bvPgvU19tqf4n6Tgrdi"
        "z2iIdaOHj8bMNbLUeTc7BoAzamQgS6fKl7OY6L5TfTeturhOtcDFmkPLPtptqBt3yNWNJcAkfORqAPOnaLKIoxG5SqHuC3dE"
        "ZO4zWZZ1pi5ypqs4+cpmr7Rboy9wxuseUz8QLqyCbzCqXoKKlQ0Erjkjv1uaT6jwFnC6thyvB9K3drqCr+lVF/Hy1c1NSyMZ"
        "3R51kzSqZb4UrPpiP9P/KS+kv1lF6MRSMO/V3wfLdLX4Iu4SU9OVZAUdmqC0EtCK9vpV2n2DXJcM6ooS1YwboTVBsDZdhVVa"
        "iq+sasrkW3GRil5NxijppBWnq9Q57iOU+tbwcjNoskoZUOlZ0/U83atduWuQD4lxYnAylBESaLQoIhwK9kZdx9xjDjFu7dY3"
        "ipuXf6s7v2t4R7hdtZuyxgSWvu9Z0/7L6f6ZNOeh9fU1nE2O8waDWttZA0RKVymOrUZjC4li3CEJwrR6zwbfttb6ZushvKee"
        "dGbgpwi0iTCKGMgbOUlYb8kQQ9kdjkB3tAHH0FpJXgMGyI3Tuum/lNwzEOsj909EMobzT0OK6ihCTb6l7yuBJnQ3vgJYL+7O"
        "55B39E052F0HjMDHNMoAotcUdGsuvRqA4mFSoaNLjDcQcGyKbu16FNSFQSNxlE1nFCY3BCtpjDJL9GXK+ECZODXYuhmUAtow"
        "jiiriwPv+f7e0psTCkUHZLYUXvOuLO2JrmVzVm4Xlslilchgmx7WWj9XK9dxrQbFFFfaODO62lJloBjLvGIvaBeF97+2OVlB"
        "X9gJPGxepE2B/W5A6WwU4ql2IfDCDodSE/pVG3I5qtZGYu41gLFiQVqfUmWigP0gnq7dbEgXFj1UuwWP9VBkYwLYiueU+1b9"
        "iJOY8k7VVAucrOhfNQr8wzed1KqIrDnTWlmaWDp0bUL+14b5/0PdeoGnQaUt+rGKYi2tRWprGbjm6sVSAXr4xyzm4CFahkys"
        "k7OD8lDa9KMGhBQxm+pgGbxFF5pfTqzhTxeNCgAFQfmErfAlXtpXJYgY3dervL3KrMaBPpWTNyNeS22MEL8KlJViWnW7hwqC"
        "gnpZEDHbxoqkeMTxSyVtXLEzXOjjPdZ4rHmhvaH5Cq2KORulgEDl0BVaVO2LcairuliiLmgsvBvsp3VDcxZwWduXtSje5s9v"
        "xcFYPnUgjEHQS2w8d1OVEYTXC9H1LGlTuw1DrW/U2xnSmUu63o+NGXjQM4YExl7jNnYXuS9K81cbgDXOMtLvDMFr4tWrf3hk"
        "9MYgKHLjwKfUDNOykFEp7UhgkA+/AYdeYqKvTRBjGWPeWbq0m+KQTQe+DcJEbSnwU97NXAx09+oUve2HKvw9xxJCR/c85DvY"
        "XX27LTj1eGk/vnAlDwG16PppGMZNxK9dUVSkJacNRAQbK9wA9z6XuQXweI6Gyb3FqEwz169Iz1O4cjG4HJiA+KGwecXM+hst"
        "0wpn9DdpN13vbrUKy8tmQeKNBmN63L4JyyIQLTcL0tyy4mNqlpBxTfMNMVzhQs/f0+ZqPnh7Zalez1f+eZVLaHZVaWNM6pJ+"
        "K46G8xcZPR5h+nh8aAHjsfDJjVGq62A2Rlo+EmO5BuQT0ZRF/StjuUaLVMAO51dRUuQ69nAnRa9qkI5XlACyWm0TwQyFTd0V"
        "DpFHjnVJcNRDFfbfF+uB95dY/Uq7/rw5Y8S2XxKcK82V5SuKzGD3YriEUfZJtExcbmGlrSvwNYoibbehOPdtSvDNwPl18AYe"
        "cphlK6x3rlbAlmPv1wEX0HJ34KoSUDlM/xMIMGbbqHC4KPvDflhxGOq77ba9CTtGKoDhpRWoQPfhauXgpRgaPjaGh183EsLM"
        "L3gMX+qq+koYZh7COoyacmDzK5zbnZf1brv7fB0lna5N9wSvlndfcrlVtRM1sc7vy6dA3W2u3lsp3/1OgalmnrUciU+QnesD"
        "ZlsmL6jWVPwl3r7Rhblju4WXwm/Vtya4Wt66WsdZCWa+1t7kJW3eHFTGR+IxFJ+IVB+G0Nto+CBU7fER4MRw9vZVAkWDVs91"
        "TGdz5yMfvJxU4stE8ne74QvOliueWohvlnKVigwxLQ9NIWU2ctYSeUe1LPZ5F9zGUwXtPhT55G64ZIfBV6tYem8OvXOQd2/Q"
        "VSo8QUpLB2eDrl7BV77RUWS++cNvaVN3mxWzwcNSF28cYAcw8QN+USEevOo0EnObUR8JEZnxpWP3OUYfaCs5/1q+0fuvnDdh"
        "UujKyj2m6zboBlY8lqBDdZsL6MiaP+ZjsOlqFj81IiH+/Y6wNB7+LzzH+tQ22eadGfPs6//U6Vf+PgzQPf88nypXrzgsqlZ/"
        "0Ev1IrZejMTWY+MTg/LeeeVBwbbpggP2tEvW5yOulQ2VXHPpbdjS2RcNwzz52hD/YELJz4RWgBQtWHFAGxAZ6z8tzMOOvGVp"
        "0CWRzrV61WY8PyGHtGAs/TPCqyqKoqfiDHWtofHx/YsqLVin2RFl8PWPALJS5TKksuI130Kn9W7xZk/eug/pOpg8BXH1zZ1V"
        "W+XbdqVJZow2yjvfJSYpzjhWqFWNsw3HvwXb5gQ2WLOuuiusON5V9WWm2whkKtnv0gndJwZkDKU4lKj2JIwt5S/3KZR2LB8k"
        "mLuLuDGPSr6IazG3wtS7A9mekOIrLExpOw4nimod52sU88zhgUKB8YguVCSUmIV51lXeIdigq/KOXvEyZ6vH6Cw9xpeXxuW9"
        "Ox2QX5pb/oLC3pqiNGoVrw/sGYg2UWSaNW0cjPdDVVRly2UYErMSvfRvxbDpF6bxwQ+mHjOHlPZYi2M6qMBHQ2bfFW9qNWYk"
        "fih3WP0yXS4t9jxLMqTG+KgMffkBuRmmsXo+viKRG2EYUlF5mv7oiaeaszrD+URAgPHy6i+2wSbbbcTT4zECZWn/Z3zluvZL"
        "vjTKq7SnUi8Z2S8Px+mjSVhV9f9fe/0baK/OY9pLidkm3WVI33+HAmMVdhv7IFMUe81XBeFNDHVc7zqTmTasq8LHwbAoyC6m"
        "LKl3cFYoos26Z23xR2dBuLR70HxPB/pzP2X5KxRG2WWmmyPQYVamYcXvJLUMYL/c3alX6U/bdK1V7016I0KCjiNl+q0FOpfG"
        "8DnmYiWC4jPDqEqRVJjhUjdDytao+NlAvwAja4uNL556+WbmvzLzO9TipbT8/4aps7w0SZpq+P7dppfNF0mN5lrXC243Gft+"
        "n3wU3Hexfg/NsC6SpktWJQ6utFDBEQBVCEpudetWtWHNQo1Ug7LuudrQrlApVAH3BVX7KrVztT5YpW5Yz6zCqNJFFTAopWJl"
        "V8HIJkC1ZHBVoRHahoXEu+g8ep8R6ybR4DwCaGt4BXz61jET1YqmoEh7lekMKwPQQXB5w+/7pZYb1rPlsavRrEWiaZiNErX0"
        "2DWG/v4cgnPGmhvoNxD8a9IIFHxZmwDuEwdVJl8nmmgn1GH62Wbaj6N2dnHZnc1rY52E7NQ5h2sNmI1nZwp/xZQSI5MmVvdZ"
        "lvJpQGGqczI2gqpbm14UT28mrRz6ikHy0Roh81UMqyE6K5xCw0NP1Zw91FMDfFS56rb09vRaGcI3evR4YaCJxFWWU4jA2+4Q"
        "XPHUQJzOr6nCDizz3bsaVUaRXEmXMoCYyyY/rGmsuQs6b+aXnDf34im90+EUv8EqQXJeHb4Sw3G8aOw4tDKhi8pjhy7U7eeN"
        "z9zbg6LBOxks3uiqtu4MN1IcV/VSsxoNNJQN2qIEv2a5kH1KPvt09eTz6oM1+gLouvZnAter4c8aOL7DTU4xhF9taPcv8cL3"
        "8cxRr2JTX9y74p3YmDqnggz6RuXNvXAzxhIxvR4cJr59qhXO/vMb8eQbeOVdpBvA6sBNiyF3mu1HECLpEnjdUdFqp7u5Ue5I"
        "NNCR2NBtu7mzEULELy3n8ySTckVz/Faz8JaP5SICtyKO5s6tHPle0t3GN5VZWsYALOo06IT4H7tJarlRTNtoDWGhSAlSRTnS"
        "xiHdhPqwtgB55ggo/JQfepV/LmsIu0lNkT8xFkN/pgiK8uPEfJ5Y4LVg5LZSvUDlL+xe9Y4NVef6Y9G3LkmMp2bP+UEU7lQb"
        "KMBgMSwqRqVLCCwzmbknu0yaePVrzTqOBL1ATewPf0nE1L+RYVMMJR4P4oURTSvHOF2bX7P2dWat1syxTrdNms2mFioaG5XS"
        "FiljqF7KkbVLCDNmY2D4iyDkk78yxvAJo2uO0y76/UQrY2i2iZyJJXDV4+e4IZvVE/fmUPLj96IN3nu80kTcG+N4KKgAtGcu"
        "VYqQFo2FTjNpjH0qO9qnIedfoGbNHFLdnDTlnpceEzOVatCOE1rMzVCMEIONAwYl/ZkD/qyuyscOJWg0BFbQTkHzHJsVTziM"
        "6RgDNmm2ZvdFL6VRmqByFlntw3iw2oUxJ01utdbPu2AnyRBqhywHOEo1c8LfGgOzV9iNnWS6sxh5VXMnV34Q/4nc2GP2K8pe"
        "swYR96UBPYhSRMrEqmkq9+9LRH+oC3O46qlZ9GBbZd2u3cLc2TZFtW6Slxi2pHAIm5eGHlkDZkhw3cQVgjIVTw5J5xTCWg/g"
        "qL2iIhQcX6NEexX08Y4Pn/FVqPwWgT7HMdZLf9QCDk/q8A2STFU2qfhOEQsX3OMIj/P6VpZOGi8tm67ozNKVFV6lP6s4Qes4"
        "mvn6+k/V2rD4U4F7csXZxsuq+tWXBtRJt+idNVOjaF1CGNUVmEmK6wL0z+b9PKON+vjotl69oM1KF1ocLqPrK3GP8B+AvTVf"
        "49of8/Nq9/kC/EGlRgncvE1w46M6GuselRXDaeSN621oH9GbgBBXns1xw3T63cT8SEPFfyZZc77+f5aqgIF/K6JqS6CUAlrV"
        "sn7/YY1e6nnhjxj+jwHL/jxaGi2YpKXu6yu92V9AyfxI8/OoudKvSdTKCZXoOtpgi/5Z8pbAfZrKBjJXiS0Dd8Ex7sbKRTRg"
        "MaNMsXSvnfkIg0OBRnTORddGZXiRalF0djHceztwhoOjN3YzztBfipNEtESn3d1Zmwjt5nHLddeC1nY1K3RDWneVPYLfwz44"
        "ZQe4d+2PMpXowBFR/f6qt3pOd8woXKPzQNufW+S2YpTtSnXa8u13YH1DW7l20YJ3iDsYOQ0OhuCKbaHIsFaxbQCnsFWzWzMF"
        "QHWMr1Ab423uPf0mH3KU+hX999Z7WmcUcG7WWrbIL11pW+/YvWZ38tDr2GVklHezeWfVi27DXp4ZS2nb8zrfPJcAAaTH4JsU"
        "rV+zVzbpoHLp2EZvoK8e3RQrjXuocvmid9X7ofMcVWO4RjCUFGD+GxmIWonr8x10g+p52RrKDHQZlUyEKdIBur7ptNu9Zmfy"
        "QLmMVG6vDctMX29w/JLOLMYsVbyWSHdobufjwOhqlY3DK9d+bITdR0foYc5lOE5X3rOF5x/gwgHS0Nvuk24qfJIHWvwZpWjT"
        "HjaC1m8iU6B15FLRXAnPGoTfwxNyLHtKb7GbuVZtwIoLaikVRmRJQD5PgVnFGa0MlPbqNduTB/H+tb4FB2+3c1AROg6t8RwH"
        "d0gdRy/xeL/0yVf/G8DTKh2vngAA"
    ,
}

def materialize_scripts(target_dir, verbose=True):
    """Write the embedded scripts to `target_dir`. Returns a summary dict."""
    target = Path(target_dir)
    target.mkdir(parents=True, exist_ok=True)
    written, identical, updated = [], [], []
    for name, blob in sorted(_EMBEDDED.items()):
        data = gzip.decompress(base64.b64decode(blob))
        dest = target / name
        if dest.exists():
            if dest.read_bytes() == data:
                identical.append(name); continue
            updated.append(name)
        else:
            written.append(name)
        dest.write_bytes(data)
    if verbose:
        print("Embedded scripts -> %s" % target)
        print("   new .......... %d" % len(written))
        print("   updated ...... %d%s" % (len(updated),
              ("  " + ", ".join(updated)) if updated else ""))
        print("   already ok ... %d" % len(identical))
        if updated:
            print("   NOTE: the files above differed from the embedded copy and were")
            print("         overwritten. If you had edited them locally, re-apply your")
            print("         change and re-embed, or the notebook will keep resetting it.")
    return {"new": written, "updated": updated, "identical": identical}

In [ ]:
# Write them next to the repo's own scripts/ directory.
_res = materialize_scripts(SCRIPTS)

---
## Stage 2 — Synthetic honeypot corpus generation ⟨optional⟩

**Script:** [`scripts/webids23_to_honeypot_log_v9.py`](../scripts/webids23_to_honeypot_log_v9.py)

This is the answer to "do we have the honeypot synthetic dataset generator" — yes, this is it.
It takes **WEB-IDS23 CSVs** as flow-level seeds (timing, IPs, method/URI shape) and synthesises a
honeypot-style JSONL log with attack payloads injected into realistic request envelopes.

### What it generates

| Family | Generators |
|---|---|
| **SQLi** | `tautology`, `union_based`, `time_based_blind`, `boolean_blind`, `error_based`, `stacked_query`, `auth_bypass` |
| **XSS** | `reflected`, `stored`, `dom_based` |
| **Benign** | stateful multi-step sessions (browse → search → submit) with coherent per-session values |

### Four v5+ correctness properties worth knowing when debugging

1. **Enforced payload uniqueness.** v4 had 72.9 % exact duplicate rows (dom_based XSS repeated up
   to 39×) because some families had a ~12-output component space sampled 36 k times. v9 expands
   the component pools *and* enforces uniqueness against a run-wide seen-set (25 retries, then a
   nonce). Rows that needed the nonce are flagged `forced_unique_nonce` — **expect ~0**; a
   non-zero count means a family's component space is too small again.
2. **Two distinct duplicate flags.** `synthetic_duplicate` = the same WEB-IDS23 flow row was
   sampled twice under oversampling (flow-level). `forced_unique_nonce` = payload-level collision
   fallback. v4 conflated these; don't re-conflate them when reading the log.
3. **User-agent realism.** v4 had 6 UAs, one literally `sqlmap/1.7.11` — a model given UA as a
   feature would learn `UA==sqlmap → malicious` and collapse on held-out LLM payloads. v9 uses
   ~18 weighted UAs so tool UAs appear at realistic low frequency.
4. **Schema noise removed.** `host` is now always `ip:port`.

### The source CSVs and why this stage still defaults to off

All five WEB-IDS23 flow CSVs **are** present, in **`data/prepared/`** (not `data/`):

| Flag | File | Rows |
|---|---|---|
| `--sqli-http` | `web-ids23_sql_injection_http (3).csv` | 74,300 |
| `--sqli-https` | `web-ids23_sql_injection_https (2).csv` | 102,584 |
| `--xss-http` | `web-ids23_xss_http (1).csv` | 4,558 |
| `--xss-https` | `web-ids23_xss_https (1).csv` | 4,533 |
| `--benign` | `web-ids23_benign (1).csv` | 825,187 |

All carry the 6 columns the script reads (`uid, ts, id.orig_h, id.resp_h, service, attack_type`)
and lose 0 rows to `dropna`. So this stage *can* run — it defaults to off only because the
committed log already exists and regenerating invalidates the committed models.

**Two things the printed arithmetic will show you:**

- **XSS is the binding constraint.** `match_min` sets both classes to `min(176884, 9091)` =
  **9,091**, so ~168k SQLi flows go unused. Total would be 36,364 — note the committed log is
  36,594, a 230-row gap, so these exact files are not a byte-identical rerun of it.
- **Only 7.2% of benign flows are `service=http`.** The rest are dns/ssl/smtp/ntp/ftp/ssh plus
  51,382 with `service=NaN`. The generator never filters on service — it only tests
  `service == "http"` to choose port 80 vs 443 — so a DNS flow becomes an HTTPS-looking web
  request. Benign payloads are synthesised regardless, so this costs **envelope realism, not
  label correctness**. Pre-filter to `service == "http"` if you want genuinely web-shaped
  benign traffic.

> ⚠️ **`import resource` (line 66) is Unix-only.** The script will not import on Windows. Run it
> under WSL or Colab, or make that import conditional first.

### Where the benign data comes from — and why it is synthesised

This is the question an examiner will ask, so it is documented here rather than buried.

**The CSV supplies the envelope; the payload text is synthesised.** The generator reads only
`uid, ts, id.orig_h, id.resp_h, service` — timing, addresses, protocol. Benign *content* is built
by `ensure_benign_session_state()` and `benign_step_value()` from archetype pools: a session picks
one browsing archetype and one identity, then advances through it coherently (search → browse →
product → checkout), so requests within a session stay thematically consistent instead of being
re-rolled independently.

**~55% of benign sessions draw from `BENIGN_HARD_NEGATIVE_PHRASES`** — text engineered to *look*
attack-shaped without being an attack:

```
O'Brien's Hardware          Ben & Jerry's           Macy's Black Friday
Error code: 500 (Internal Server Error) -- see logs for details.
Status: shipped -- tracking #4821-XJ; ETA 3-5 business days.
x = a + b; y = c - d; return x * y;
config: {retries: 3, timeout: 30}; env=production
threshold >= 0.8 && confidence <= 1.0
```

Apostrophes, `--`, semicolons, braces, `&&`, parentheses — every character class an injection
uses, with none of the intent. **This is the false-positive surface**, and it is the single most
important part of the benign data.

### Why not a public benign HTTP dataset?

Three were evaluated and rejected on inspection:

| Candidate | Rows | Why rejected |
|---|---|---|
| **CSIC 2010** | 36,000 | One Spanish e-commerce app, 23 paths, structured form fields (`nombre=Vino Rioja`). Almost no attack-shaped punctuation. |
| **HttpParams** | 19,304 benign | **CSIC-derived** — same data recycled. Only **4.1%** of rows contain any special character; most are bare values (`ruiloba`, `48112`). |
| **FWAF `goodqueries`** | 1.29 M | **Mislabeled.** The "good" file contains real RFI attacks (`/config.php?xcart_dir=http://192.168.202.96:8080/...`). Only 2.4% have query params. Training on it would poison the corpus. |

All three are *weaker* hard negatives than the phrase pool above. Real benign HTTP payload
corpora are scarce because privacy rules prevent publishing genuine user traffic — which is
precisely why this field synthesises.

**Precedent in the literature** (for Chapter 2):

- **Yu et al. (2026)**, *Multi-Agent Honeypot-Based Request-Response Context Dataset for Improved
  SQL Injection Detection*, arXiv:2603.02963 — a Request Generator Agent "produces diverse
  malicious and benign HTTP requests"; 140,973 labeled pairs, no real benign corpus. Nearest
  neighbour to this architecture.
- **Cordero et al. (2021)**, *On Generating Network Traffic Datasets with Synthetic Attacks for
  Intrusion Detection*, ACM TOPS — the ID2T toolkit injects synthetic attacks into background
  traffic, mimicking background properties. Same real-envelope/synthetic-payload pattern used here.
- **Ferriyan et al. (2021)**, *HIKARI-2021*, Applied Sciences 11(17):7868 — hybrid synthetic
  attacks + benign traffic in a real environment.

**If you want to strengthen this**, the highest-value change is expanding
`BENIGN_HARD_NEGATIVE_PHRASES` (~40 entries today), not sourcing another dataset. Note that doing
so changes training data and therefore requires a Stage 5 retrain plus full re-evaluation.

> ⚠️ **Regenerating invalidates the committed models.** A new log means new session-grouped
> splits, so `models/*.pkl` no longer correspond to it. If you set `RUN_STAGE2 = True` you must
> also run Stage 5, and every downstream number changes. That is a legitimate thing to do — just
> never mix a regenerated log with the committed models.

In [ ]:
# ==========================================================================
# STAGE 2 - SYNTHETIC HONEYPOT GENERATION (optional; skipped by default)
# ==========================================================================
RUN_STAGE2 = False          # <-- set True ONLY if you have the WEB-IDS23 CSVs

# The WEB-IDS23 flow CSVs actually live in data/prepared/ in this repo
# (NOT data/). Verified row counts, all 6 required columns present, 0 rows
# lost to dropna on id.orig_h/id.resp_h:
#     sqli_http   74,300     xss_http   4,558
#     sqli_https 102,584     xss_https  4,533
WEBIDS_SQLI_HTTP  = PREP / "web-ids23_sql_injection_http (3).csv"
WEBIDS_SQLI_HTTPS = PREP / "web-ids23_sql_injection_https (2).csv"
WEBIDS_XSS_HTTP   = PREP / "web-ids23_xss_http (1).csv"
WEBIDS_XSS_HTTPS  = PREP / "web-ids23_xss_https (1).csv"
# 825,187 rows - far more than the ~18k needed (match_min uses ~2.2% of it).
# NOTE: only 59,393 of these are service=http; the rest are dns/ssl/smtp/ntp/
# ftp/ssh and 51,382 with service=NaN. The generator does not filter on
# service - it only tests `service == "http"` to pick port 80 vs 443, so a
# DNS or NTP flow becomes an HTTPS-looking web request envelope. That is a
# real modelling wrinkle, not a crash: see the note printed by this stage.
WEBIDS_BENIGN     = PREP / "web-ids23_benign (1).csv"

HONEYPOT_LOG   = DATA / "honeypot_final.log"
GEN_SEED       = 42
OBFUSCATE_PROB = 0.4        # fraction of attack payloads passed through obfuscation
GEN_STRATEGY   = "match_min"
BENIGN_RATIO   = 1.0

with stage(
    "Synthetic honeypot corpus generation",
    "Synthesise the labelled honeypot JSONL training corpus from WEB-IDS23 flow "
    "seeds, with enforced payload uniqueness and realistic request envelopes.",
    inputs={"generator script": SCRIPTS / "webids23_to_honeypot_log_v9.py",
            "WEB-IDS23 sqli http":  WEBIDS_SQLI_HTTP,
            "WEB-IDS23 sqli https": WEBIDS_SQLI_HTTPS,
            "WEB-IDS23 xss http":   WEBIDS_XSS_HTTP,
            "WEB-IDS23 xss https":  WEBIDS_XSS_HTTPS,
            "WEB-IDS23 benign":     WEBIDS_BENIGN},
    outputs={"honeypot log": HONEYPOT_LOG},
    params={"RUN_STAGE2": RUN_STAGE2, "seed": GEN_SEED,
            "obfuscate_prob": OBFUSCATE_PROB, "strategy": GEN_STRATEGY,
            "benign_ratio": BENIGN_RATIO},
    optional=True,
) as rec:

    missing = [str(p) for p in (WEBIDS_SQLI_HTTP, WEBIDS_SQLI_HTTPS, WEBIDS_XSS_HTTP,
                                WEBIDS_XSS_HTTPS, WEBIDS_BENIGN) if not p.exists()]

    # Show the target arithmetic BEFORE running - match_min makes the smaller
    # class the binding constraint, which is easy to not notice.
    present = [(lbl, p) for lbl, p in
               (("sqli_http", WEBIDS_SQLI_HTTP), ("sqli_https", WEBIDS_SQLI_HTTPS),
                ("xss_http", WEBIDS_XSS_HTTP), ("xss_https", WEBIDS_XSS_HTTPS))
               if p.exists()]
    if present:
        import pandas as _pd
        print("   SOURCE FLOW COUNTS + TARGET ARITHMETIC")
        counts = {}
        for lbl, p in present:
            counts[lbl] = sum(len(c) for c in
                              _pd.read_csv(p, usecols=["uid"], chunksize=200000))
            print("      %-12s %8d rows" % (lbl, counts[lbl]))
        sqli_nat = counts.get("sqli_http", 0) + counts.get("sqli_https", 0)
        xss_nat  = counts.get("xss_http", 0) + counts.get("xss_https", 0)
        print("      SQLi natural %d | XSS natural %d" % (sqli_nat, xss_nat))
        if GEN_STRATEGY == "match_min" and sqli_nat and xss_nat:
            tgt = min(sqli_nat, xss_nat)
            ben = round(BENIGN_RATIO * (tgt + tgt))
            print("      match_min -> SQLi=%d XSS=%d benign=%d  TOTAL=%d"
                  % (tgt, tgt, ben, tgt * 2 + ben))
            binding = "XSS" if xss_nat < sqli_nat else "SQLi"
            unused  = max(sqli_nat, xss_nat) - tgt
            note("%s is the binding constraint; %d %s flows go UNUSED under "
                 "match_min. Use --strategy custom if you want more."
                 % (binding, unused, "SQLi" if binding == "XSS" else "XSS"))
            if WEBIDS_BENIGN.exists():
                bser = _pd.read_csv(WEBIDS_BENIGN, usecols=["service"])
                bn = len(bser)
                http_n = int((bser["service"] == "http").sum())
                print("      benign available %d rows -> need %d (%.1f%%)"
                      % (bn, ben, 100.0 * ben / max(bn, 1)))
                check(bn >= ben,
                      "benign pool (%d) covers the target (%d)" % (bn, ben))
                print("      benign service mix: http=%d (%.1f%%), other/NaN=%d"
                      % (http_n, 100.0 * http_n / max(bn, 1), bn - http_n))
                note("Only %.1f%% of benign flows are service=http. The generator "
                     "does NOT filter on service - it just maps http->port 80 and "
                     "everything else->443, so dns/ssl/smtp/ntp flows become "
                     "HTTPS-looking web requests. Benign payloads are synthesised "
                     "regardless, so this affects envelope realism (port/host), "
                     "not label correctness. Pre-filter to service==http if you "
                     "want the benign envelopes to be genuinely web traffic."
                     % (100.0 * http_n / max(bn, 1)))

    if not RUN_STAGE2:
        skip("RUN_STAGE2 is False - using the committed data/honeypot_final.log. "
             "Regenerating would invalidate the committed models.")
        note("All five source CSVs ARE present in data/prepared/, so setting "
             "RUN_STAGE2=True will work - but you must then re-run Stage 5, "
             "and every downstream number changes.")
    elif not WEBIDS_BENIGN.exists():
        skip("RUN_STAGE2 is True but the BENIGN CSV is missing - refusing to run.",
             produced_by="the WEB-IDS23 benign subset")
        print("          missing: %s" % WEBIDS_BENIGN)
        note("Without --benign the generator writes an ATTACK-ONLY log: no "
             "label=0 rows at all. Training on that produces a model that "
             "cannot discriminate. This is a hard stop, not a warning.")
    elif missing:
        skip("RUN_STAGE2 is True but %d source CSV(s) are missing." % len(missing),
             produced_by="the WEB-IDS23 dataset (not distributed in this repo)")
        for m in missing:
            print("          missing: %s" % m)
    else:
        if HONEYPOT_LOG.exists():
            backup = HONEYPOT_LOG.with_suffix(".log.bak_%s" % RUN_ID)
            shutil.copy2(HONEYPOT_LOG, backup)
            note("Backed up the existing log to %s (regeneration overwrites it)."
                 % backup.name)
        rc, _ = sh([sys.executable, str(SCRIPTS / "webids23_to_honeypot_log_v9.py"),
                    "--sqli-http",  WEBIDS_SQLI_HTTP,
                    "--sqli-https", WEBIDS_SQLI_HTTPS,
                    "--xss-http",   WEBIDS_XSS_HTTP,
                    "--xss-https",  WEBIDS_XSS_HTTPS,
                    "--benign",     WEBIDS_BENIGN,
                    "--strategy",   GEN_STRATEGY,
                    "--benign-ratio", str(BENIGN_RATIO),
                    "--seed",       str(GEN_SEED),
                    "--obfuscate-prob", str(OBFUSCATE_PROB),
                    "-o", str(HONEYPOT_LOG)], cwd=ROOT)
        check(rc == 0, "generator exited 0")
        check(HONEYPOT_LOG.exists() and HONEYPOT_LOG.stat().st_size > 0,
              "honeypot log was written and is non-empty")
        note("Stage 3 audits this log. Pay attention to the forced_unique_nonce "
             "count there - it should be ~0.")

---
## Stage 3 — Corpus audit

Reads the honeypot log and reports what is actually in it. **This stage is not optional even
when Stage 2 is skipped** — it is the ground truth for every claim made downstream, and it
catches corpus problems before you spend GPU time training on them.

### What it checks and why

| Check | Failure it catches |
|---|---|
| Row count and label balance | Silent truncation; a corpus that drifted from the reported 36,594 rows |
| JSON parse failures per line | Partially-written or newline-corrupted log |
| Schema (all 17 expected keys) | Generator version mismatch — a `v4` log fed to a `v9` pipeline |
| `attack_family` distribution | A family that collapsed to zero, or one that dominates |
| **`forced_unique_nonce` count** | The v4 duplication bug reappearing (component space too small) |
| **Exact duplicate payload rate** | The primary v5 fix regressing; inflated metrics from repeats |
| `synthetic_duplicate` rate | Flow-level oversampling — expected, but should be reported not hidden |
| Session count vs row count | Whether the grouped split will actually have enough groups |
| `obfuscated` fraction | Should land near `OBFUSCATE_PROB` on attack rows |
| Payload length distribution | Truncation at `MAXLEN=200` affects LSTM input; you want to know the tail |

**Note on `attack_type`:** the log has no `attack_type` field — family is carried in
`attack_family` (with `benign` for negatives). Scripts that want a coarse type derive it. Don't
go looking for `attack_type` in the JSONL; it isn't there.

In [ ]:
# ==========================================================================
# STAGE 3 - CORPUS AUDIT (never skipped; ground truth for everything below)
# ==========================================================================
import collections

EXPECTED_KEYS = {"time", "source_ip", "host", "method", "uri", "user_agent",
                 "request_body", "referer", "flow_uid", "dup_index", "session_id",
                 "session_seq", "label", "attack_family", "obfuscated",
                 "synthetic_duplicate", "forced_unique_nonce"}

with stage(
    "Corpus audit",
    "Establish exactly what is in the training corpus - counts, balance, schema, "
    "family mix, duplication rates - before any training consumes it.",
    inputs={"honeypot log": HONEYPOT_LOG},
    params={"expected_schema_keys": len(EXPECTED_KEYS)},
) as rec:

    if not HONEYPOT_LOG.exists():
        skip("data/honeypot_final.log is absent.",
             produced_by="Stage 2 (webids23_to_honeypot_log_v9.py)")
        raise FileNotFoundError("honeypot log required; cannot continue past Stage 3")

    rows, bad_lines = [], []
    with open(HONEYPOT_LOG, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                bad_lines.append((i, str(e)))

    n = len(rows)
    labels   = collections.Counter(r.get("label") for r in rows)
    families = collections.Counter(r.get("attack_family") for r in rows)
    sessions = {r.get("session_id") for r in rows}
    keys_seen = set().union(*(set(r.keys()) for r in rows)) if rows else set()

    print("   ROWS")
    print("      parsed ................ %d" % n)
    print("      malformed lines ....... %d" % len(bad_lines))
    for ln, err in bad_lines[:5]:
        print("         line %d: %s" % (ln, err))
    print("      unique session_id ..... %d  (avg %.1f rows/session)"
          % (len(sessions), n / max(len(sessions), 1)))

    print("\n   LABEL BALANCE")
    for lab in sorted(labels, key=lambda x: (x is None, x)):
        print("      label=%-5s %7d  (%5.1f%%)"
              % (lab, labels[lab], 100.0 * labels[lab] / max(n, 1)))

    print("\n   ATTACK FAMILY DISTRIBUTION")
    for fam, c in families.most_common():
        print("      %-20s %7d  (%5.1f%%)" % (fam, c, 100.0 * c / max(n, 1)))

    # ---- duplication metrics: the v5 fixes, verified ----------------------
    def payload_of(r):
        """Same text the trainer sees: URI query + body."""
        return "%s %s" % (r.get("uri", ""), r.get("request_body", "") or "")

    payloads   = [payload_of(r) for r in rows]
    uniq       = len(set(payloads))
    dup_rate   = 100.0 * (n - uniq) / max(n, 1)
    nonce_n    = sum(1 for r in rows if r.get("forced_unique_nonce"))
    syndup_n   = sum(1 for r in rows if r.get("synthetic_duplicate"))
    atk        = [r for r in rows if r.get("label") == 1]
    obf_n      = sum(1 for r in atk if r.get("obfuscated"))

    print("\n   DUPLICATION / UNIQUENESS  (the v5 correctness fixes)")
    print("      unique uri+body ....... %d / %d   (exact-duplicate rate %.2f%%)"
          % (uniq, n, dup_rate))
    print("      forced_unique_nonce ... %d   (payload-level collision fallback)" % nonce_n)
    print("      synthetic_duplicate ... %d   (flow-level oversampling; different thing)" % syndup_n)
    print("      obfuscated attacks .... %d / %d  (%.1f%% of attack rows)"
          % (obf_n, len(atk), 100.0 * obf_n / max(len(atk), 1)))

    lens = sorted(len(p) for p in payloads)
    def pct(q):
        return lens[min(int(q * len(lens)), len(lens) - 1)] if lens else 0
    over = sum(1 for L in lens if L > 200)
    print("\n   PAYLOAD LENGTH (chars; LSTM truncates at MAXLEN=200)")
    print("      min/p50/p90/p99/max ... %d / %d / %d / %d / %d"
          % (lens[0], pct(.50), pct(.90), pct(.99), lens[-1]))
    print("      longer than 200 ....... %d  (%.1f%% - truncated for the LSTM branch)"
          % (over, 100.0 * over / max(n, 1)))

    print()
    check(n > 0, "corpus is non-empty")
    check(not bad_lines, "every line parsed as JSON (%d malformed)" % len(bad_lines))
    missing_keys = EXPECTED_KEYS - keys_seen
    check(not missing_keys,
          "schema has all %d expected keys%s"
          % (len(EXPECTED_KEYS),
             "" if not missing_keys else " - MISSING: %s" % sorted(missing_keys)))
    check(set(labels) <= {0, 1}, "labels are strictly 0/1")
    bal = min(labels.get(0, 0), labels.get(1, 0)) / max(max(labels.get(0, 0), labels.get(1, 1)), 1)
    check(bal > 0.8, "classes are roughly balanced (minority/majority = %.2f)" % bal)
    check(dup_rate < 5.0,
          "exact-duplicate payload rate %.2f%% is below 5%% (v4 was 72.9%%)" % dup_rate)
    # Tiered, because the generator's docstring says "expect ~0" without
    # quantifying it. Observed baseline on the committed corpus is 1.11%
    # (407/36594) with a 0.00% duplicate rate - i.e. the fallback doing its job.
    # Only a LARGE share indicates a family's component space actually shrank.
    nonce_pct = 100.0 * nonce_n / max(n, 1)
    print("      forced_unique_nonce rate %.2f%% (committed-corpus baseline: 1.11%%)"
          % nonce_pct)
    check(nonce_pct < 5.0,
          "forced_unique_nonce rate %.2f%% is below 5%% - above that, a family's "
          "component space has shrunk and payload diversity is degraded" % nonce_pct)
    if nonce_pct > 2.0:
        note("nonce rate %.2f%% is above the 1.11%% baseline; worth checking which "
             "family is colliding if you regenerated the corpus." % nonce_pct)
    check(len(sessions) > 100,
          "enough distinct sessions (%d) for a group-wise split" % len(sessions))

    CORPUS_STATS = {"rows": n, "malformed": len(bad_lines), "labels": dict(labels),
                    "families": dict(families), "sessions": len(sessions),
                    "unique_payloads": uniq, "dup_rate_pct": round(dup_rate, 3),
                    "forced_unique_nonce": nonce_n,
                    "synthetic_duplicate": syndup_n,
                    "obfuscated_attacks": obf_n,
                    "len_p50": pct(.50), "len_p99": pct(.99), "len_max": lens[-1]}
    rec["corpus_stats"] = CORPUS_STATS
    note("CORPUS_STATS captured into the run log for the Stage 13 report.")

---
## Stage 4 — Feature/split preparation ⟨diagnostic⟩

**Script:** [`scripts/prepare_honeypot_for_training.py`](../scripts/prepare_honeypot_for_training.py)

### Read this before you assume this stage is required — it is not

`18_train_stacked.py` (Stage 5) **does its own loading, splitting and featurisation** by importing
`load_log()` and `session_grouped_split()` from this module, and it **rewrites**
`data/prepared/*.csv` and `*.npz` itself at the end of training. Its own comment says why:

> *Rewrite the prepared splits so downstream eval scripts (13, 14) read the SAME rows/features
> these models were trained and tested on. Without this they load stale .npz/.csv from a
> different split size and produce garbage.*

So running this script standalone **before** Stage 5 is not necessary, and its output will be
overwritten. It is included here for one genuinely useful reason: **it shows you the split
geometry and leakage properties before you commit GPU time to training.**

### The two leakage protections, and how to verify them here

1. **Session-grouped split (`GroupShuffleSplit` on `session_id`, 70/15/15).** Rows from one
   session never straddle train/val/test. Without this the model can learn session artifacts
   (a specific `source_ip`, a `session_id` hash) instead of payload structure, and every
   evasion-resistance number is inflated. The cell below **explicitly verifies zero session
   overlap** between the three splits.
2. **SMOTE is train-only**, fit *after* the split. Oversampling before splitting leaks synthetic
   neighbours of test rows into training.

The meta-learner is trained on **val** probabilities, so the base learners never see the
meta-learner's fitting rows either.

In [ ]:
# ==========================================================================
# STAGE 4 - SPLIT GEOMETRY + LEAKAGE VERIFICATION (diagnostic)
# Stage 5 redoes this internally and overwrites data/prepared/*, so this stage
# exists to SHOW the split, not to produce artifacts training depends on.
# ==========================================================================
RUN_STAGE4 = True     # cheap (CPU, ~1-2 min); set False to jump straight to training

SPLIT_SEED = 42       # must match the seed used in Stage 5 for these to describe it

with stage(
    "Split geometry + leakage verification",
    "Show the 70/15/15 session-grouped split and PROVE no session_id straddles "
    "train/val/test, before spending GPU time on training.",
    inputs={"honeypot log": HONEYPOT_LOG,
            "prep module": SCRIPTS / "prepare_honeypot_for_training.py"},
    params={"RUN_STAGE4": RUN_STAGE4, "seed": SPLIT_SEED,
            "split": "70/15/15 GroupShuffleSplit on session_id"},
    optional=True,
) as rec:

    if not RUN_STAGE4:
        skip("RUN_STAGE4 is False - Stage 5 performs its own identical split anyway.")
    else:
        import importlib.util
        def _load_module(path, name):
            spec = importlib.util.spec_from_file_location(name, path)
            m = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(m)
            return m

        note("Importing prepare_honeypot_for_training.py (same code path as Stage 5).")
        _prep = _load_module(SCRIPTS / "prepare_honeypot_for_training.py", "prep")
        sys.path.insert(0, str(SCRIPTS))
        from text_normalize import normalize_text

        t0 = time.time()
        df = _prep.load_log(HONEYPOT_LOG)
        note("load_log() -> %d rows x %d engineered columns in %.1fs"
             % (len(df), df.shape[1], time.time() - t0))

        # Stage 5 normalises BEFORE splitting; mirror that exactly.
        df["_payload_text"] = df["_payload_text"].astype(str).map(normalize_text)

        tr, va, te = _prep.session_grouped_split(df, SPLIT_SEED)

        print("\n   SPLIT SIZES")
        tot = len(df)
        for nm, part in (("train", tr), ("val", va), ("test", te)):
            pos = int(part["label"].sum())
            print("      %-6s %7d rows (%4.1f%%)   attack=%6d (%4.1f%%)   sessions=%d"
                  % (nm, len(part), 100.0 * len(part) / tot, pos,
                     100.0 * pos / max(len(part), 1), part["_session_id"].nunique()))

        s_tr = set(tr["_session_id"]); s_va = set(va["_session_id"]); s_te = set(te["_session_id"])
        print("\n   LEAKAGE CHECK (session_id overlap between splits)")
        print("      train n val ........... %d" % len(s_tr & s_va))
        print("      train n test .......... %d" % len(s_tr & s_te))
        print("      val   n test .......... %d" % len(s_va & s_te))

        print("\n   FAMILY MIX PER SPLIT (a family absent from test = untestable claim)")
        fams = sorted(df["attack_family"].unique())
        print("      %-20s %8s %8s %8s" % ("family", "train", "val", "test"))
        for fam in fams:
            print("      %-20s %8d %8d %8d"
                  % (fam, int((tr["attack_family"] == fam).sum()),
                     int((va["attack_family"] == fam).sum()),
                     int((te["attack_family"] == fam).sum())))

        print()
        check(not (s_tr & s_va), "no session_id shared between train and val")
        check(not (s_tr & s_te), "no session_id shared between train and test")
        check(not (s_va & s_te), "no session_id shared between val and test")
        check(len(tr) + len(va) + len(te) == tot, "splits partition the corpus exactly")
        check(abs(len(tr) / tot - 0.70) < 0.05, "train share is ~70%% (got %.1f%%)"
              % (100.0 * len(tr) / tot))
        for fam in fams:
            if (df["attack_family"] == fam).sum() > 50:
                check(int((te["attack_family"] == fam).sum()) > 0,
                      "family '%s' is represented in the test split" % fam)

        SPLIT_STATS = {"train": len(tr), "val": len(va), "test": len(te),
                       "sessions": {"train": len(s_tr), "val": len(s_va), "test": len(s_te)},
                       "overlap": {"train_val": len(s_tr & s_va),
                                   "train_test": len(s_tr & s_te),
                                   "val_test": len(s_va & s_te)}}
        rec["split_stats"] = SPLIT_STATS
        del df, tr, va, te      # free memory before the LSTM stage

---
## Stage 5 — Train the stacked ensemble

**Script:** [`scripts/18_train_stacked.py`](../scripts/18_train_stacked.py)

Trains all four artifacts from the honeypot log in one pass. Architecture, as reconstructed to
match the committed models exactly:

| Component | Specification |
|---|---|
| **RF v2** | `RandomForest(n_estimators=200, max_depth=20, class_weight="balanced")` on **319 features** = 19 structural + 300 char n-gram TF-IDF |
| **TF-IDF** | `char_wb`, `ngram_range=(2,4)`, `max_features=300`, `lowercase=False`, **fit on train text only** |
| **LSTM** | `Embedding(128,32, mask_zero=True)` → `LSTM(64, seq)` → `Dropout(.3)` → `LSTM(32)` → `Dropout(.3)` → `Dense(1, sigmoid)`; ordinal-encoded, `MAXLEN=200`, `vocab=128` |
| **Meta** | `LogisticRegression` on `[rf_proba, lstm_proba]` from the **val** split |

### Three details that cause silent wrongness if forgotten

1. **`normalize_text()` is applied up front**, before every feature, identically here and in the
   live app. It exists because the Unicode attack `１＇ ＯＲ ＇１＇＝＇１` evaded detection. If you
   ever score text without it, the app and the trainer disagree.
2. **`mask_zero=True` is load-bearing.** `0` is the PAD id and the first LSTM must propagate the
   mask to the second (`return_sequences=True`) so padding never reaches the final state.
3. **The meta-learner fits on val, not train** — otherwise it sees base-learner predictions on
   rows those learners memorised, and the stack's advantage is fictitious.

### On the "two-layer LSTM doesn't learn" note

An earlier comment claimed this architecture trained to AUC 0.5. It does not reproduce: seeded,
it reaches val_auc 0.9996 in epoch 1. The original failure was an unseeded bad initialisation —
hence the explicit `GlorotUniform(seed=...)`. If you *do* see AUC ≈ 0.5, suspect the seed and the
mask, not the architecture.

> ⚠️ **This stage overwrites `models/`.** The cell backs up the existing models first, and
> fingerprints them before *and* after so you can prove which artifacts changed.

In [ ]:
# ==========================================================================
# STAGE 5 - TRAIN THE STACKED ENSEMBLE (RF + LSTM + meta)
# Overwrites models/*.pkl, models/lstm_best.keras, and data/prepared/*.
# ==========================================================================
RUN_STAGE5   = True
TRAIN_EPOCHS = 6        # early stopping on val_auc (patience 3) usually halts sooner
TRAIN_SEED   = 42       # keep equal to SPLIT_SEED so Stage 4's geometry describes this run
BACKUP_MODELS = True

MODEL_FILES = {"RF":         MODELS / "rf2.pkl",
               "LSTM":       MODELS / "lstm_best.keras",
               "meta":       MODELS / "meta.pkl",
               "vectorizer": MODELS / "ngram_vectorizer.pkl"}

with stage(
    "Train stacked ensemble",
    "Fit RF v2 + 2-layer LSTM + logistic meta-learner from the honeypot log, and "
    "rewrite data/prepared/* so downstream eval reads the SAME split.",
    inputs=dict({"honeypot log": HONEYPOT_LOG,
                 "trainer": SCRIPTS / "18_train_stacked.py"},
                **{"model BEFORE: " + k: v for k, v in MODEL_FILES.items()}),
    outputs=dict({"prepared splits": PREP},
                 **{"model AFTER: " + k: v for k, v in MODEL_FILES.items()}),
    params={"RUN_STAGE5": RUN_STAGE5, "epochs": TRAIN_EPOCHS, "seed": TRAIN_SEED,
            "gpu": GPU_AVAILABLE, "backup": BACKUP_MODELS},
) as rec:

    if not RUN_STAGE5:
        skip("RUN_STAGE5 is False - keeping the committed models. Valid ONLY if "
             "scikit-learn matches the %s pin (see Stage 1)." % SKLEARN_PIN)
    else:
        # Record pre-training hashes so 'did the model actually change?' is answerable.
        before = {k: (sha256_file(p) if p.exists() else None)
                  for k, p in MODEL_FILES.items()}

        if BACKUP_MODELS:
            bdir = MODELS / ("_backup_%s" % RUN_ID)
            bdir.mkdir(exist_ok=True)
            for k, p in MODEL_FILES.items():
                if p.exists():
                    shutil.copy2(p, bdir / p.name)
            note("Backed up existing models to %s" % bdir)

        if not GPU_AVAILABLE:
            note("No GPU: expect ~165 s/epoch (vs ~10-20 s on a T4). "
                 "%d epochs ~ %d min." % (TRAIN_EPOCHS, TRAIN_EPOCHS * 165 // 60))

        rc, lines = sh([sys.executable, str(SCRIPTS / "18_train_stacked.py"),
                        "--log", str(HONEYPOT_LOG),
                        "--epochs", str(TRAIN_EPOCHS),
                        "--seed", str(TRAIN_SEED)], cwd=ROOT)
        check(rc == 0, "18_train_stacked.py exited 0")

        # Surface the trainer's own val_auc / test lines rather than making you scroll.
        interesting = [l for l in lines
                       if any(t in l.lower() for t in
                              ("val_auc", "auc:", "f1", "test", "train=", "epoch"))]
        if interesting:
            print("\n   TRAINER HIGHLIGHTS (val_auc ~0.5 => suspect seed/mask, not architecture)")
            for l in interesting[-25:]:
                print("      %s" % l)

        print()
        after = {}
        for k, p in MODEL_FILES.items():
            ok = p.exists()
            check(ok, "model artifact written: %s" % p.name)
            if ok:
                after[k] = sha256_file(p)
                changed = before[k] != after[k]
                # An UNCHANGED hash is not a failure here: training is seeded, so
                # a rerun at the same seed on the same corpus reproduces RF and
                # the TF-IDF vectorizer byte-for-byte. That is determinism working.
                # (The LSTM and meta still differ - float non-determinism in the
                # CPU/GPU kernels.) What actually matters is that the file was
                # REWRITTEN, so verify mtime, not content.
                fresh = p.stat().st_mtime >= (RUN_STARTED - 5)
                print("        %-11s %s, %s  sha256=%s..."
                      % (k, "rewritten" if fresh else "NOT rewritten",
                         "content changed" if changed else "byte-identical (seeded determinism)",
                         after[k][:16]))
                check(fresh,
                      "%s was rewritten by this training run (checks mtime, not "
                      "content - a seeded rerun legitimately reproduces identical "
                      "bytes)" % k)

        for f in ("rf_train_v2.csv", "rf_val_v2.csv", "rf_test_v2.csv",
                  "lstm_train.npz", "lstm_val.npz", "lstm_test.npz"):
            check((PREP / f).exists(),
                  "prepared split rewritten: %s (Stages 9's scripts read these; "
                  "stale files here produce garbage)" % f)
        rec["model_hashes"] = {"before": before, "after": after}

---
## Stage 6 — Ollama + LLM payload generation ⟨optional⟩

**Scripts:** [`25_generate_codellama_corpus.py`](../scripts/25_generate_codellama_corpus.py),
[`26_generate_deepseek_corpus.py`](../scripts/26_generate_deepseek_corpus.py)

Generates the **held-out** attack corpus with local LLMs via Ollama. These payloads are the
evasion test set — the whole point is that the detector has never seen their *expression*.

| Model | Tag | Payloads kept | Notes |
|---|---|---|---|
| Code Llama 13B | `codellama:13b` | 500 SQLi + 500 XSS = **1,000** | straightforward generation |
| DeepSeek-R1 14B | `huihui_ai/deepseek-r1-abliterated:14b-qwen-distill` | 300 SQLi + 300 XSS = **600** | abliterated build — the aligned release refuses payload generation |

**Total kept: 1,600 payloads.**

### Ask for more than you keep — the headroom rule

Payloads get thrown away for three reasons: **validation failure** (refusal, reasoning prose, no
attack signature), **duplicates** (deduped on a normalised key, so near-copies count as one), and
**malformed output** (the model ignored the JSON format).

So the number requested must exceed the number kept. The notebook now computes this instead of
hard-coding it:

```python
raw_needed = target / accept_rate * SAFETY_x     # SAFETY_x = 1.5
batches    = ceil(raw_needed / BATCH_SIZE)       # BATCH_SIZE = 20
```

| Model | Keep | Observed accept rate | Raw requested | Batches |
|---|---|---|---|---|
| Code Llama | 500/type | 78.4 % | 960 | **48** |
| DeepSeek-R1 | 300/type | 10.1 % | 4,460 | **223** |

The previous hard-coded values (25 and 15 batches) were **both too small** — Code Llama needed 32
minimum, DeepSeek ~149. That is the mechanical reason the committed smoke test came up short: the
loop ran out of batches before it ran out of target.

**Over-provisioning is free.** The generator stops the moment a target is reached, so a generous
batch count costs nothing when yield is good and rescues the run when it is not. Under-provisioning
silently returns fewer payloads than you asked for.

> DeepSeek's acceptance will now fall *below* 10.1 %, because reasoning prose that was previously
> accepted is correctly rejected. The 1.5× safety margin absorbs that.

### Checkpoint output

At the end of the stage a compact **CHECKPOINT** block is printed — counts, rates, targets met,
and any stale rows, with no payload text. Copy the block between the `+---+` lines to share the
state of a run for review.

### The methodological rule that makes these numbers meaningful

**Section 3.5: generated payloads are test data only, never training data.** They are never fed
back into Stage 5. Generation runs under controlled decoding (`temperature=0, top_k=1`) and
outputs are **not** cleaned to taste — Section 3.5 is explicit: *"Generated outputs will not be
treated as cleaning criteria."* [`payload_validation.py`](../scripts/payload_validation.py)
applies fixed structural validation, and everything rejected is written to `*_rejects.csv` so the
accept/reject ratio is auditable rather than hidden.

### Three operational traps

1. **`zstd` must be installed before Ollama.** Both the installer bundle and the model blobs are
   zstd-compressed and Colab's base image lacks it. Skipping this fails with a confusing
   extraction error.
2. **The namespace underscore is not a typo.** `huihui_ai` (underscore) is the *Ollama registry*
   namespace. The HuggingFace repo is `huihui-ai` (hyphen) and is **not** pullable by Ollama —
   there is no GGUF, so you get a 400.
3. **DeepSeek Round-2 needs Docker.** The Round-2 ModSecurity-PL1 feedback loop cannot run on
   Colab; `--no-round2` is forced there. Round-1 output is still valid, just not feedback-refined.

### Where these files live, and why the notebook moves them

This trips people up, so it is worth stating plainly:

| Location | What is there | Who reads it |
|---|---|---|
| `colab/*.csv` | The **committed** generated corpora — the artifacts a Colab run downloaded | nothing, directly |
| `data/eval/*.csv` | Where **`27_assemble_llm_holdout.py` looks** | Stage 7 |

The generator scripts (`25_`, `26_`) write to `data/eval/`. But a Colab run produces a zip you
unpack into `colab/`, so the committed copies ended up there. **Stage 6 stages `colab/` →
`data/eval/`** when generation is skipped, so Stage 7 finds them either way. It never overwrites
a fresher file already in `data/eval/`, and it reports whether an existing copy is identical.

Each generator emits three files, and all three matter:

- `*_holdout.csv` — accepted payloads, the frozen test corpus
- `*_rejects.csv` — rejected payloads **with the reason**; this is the audit trail for the
  acceptance rate, and Section 3.5 wants it kept, not discarded
- `*_run_manifest.json` — model tag + **digest**, decoding params, per-family counts, elapsed time

### ⚠️ The committed corpora are smoke tests, not full runs

Read the manifests before you quote any number from these files:

| | Code Llama | DeepSeek-R1 |
|---|---|---|
| `target_per_type` | **20** (not 500) | **20** (not 300) |
| accepted | 40 total | 38 total |
| acceptance rate | 78.4 % | **10.1 %** |
| `target_met` | sqli ✓ xss ✓ | sqli ✓ **xss ✗ (18/20)** |
| Round 2 | n/a | **disabled** (needs Docker) |

So the committed set is **78 payloads**, not the ~1,600 the full protocol calls for. That is
enough to exercise the pipeline end to end, and **not** enough to report a detection rate from.
Stage 6 prints these numbers and fails a check when a target was not met, so this cannot pass
unnoticed. For thesis figures, set `RUN_STAGE6 = True` with the full batch counts.

DeepSeek's 10.1 % acceptance is itself a finding worth keeping: the abliterated reasoning model
emits far more malformed output than Code Llama, and `deepseek_rejects.csv` (339 rows) is the
evidence.

### ✅ Reasoning-leakage filter (fixed)

The committed DeepSeek corpus had a second, subtler problem: **8 of its 38 accepted rows (21 %)
were the model's leaked chain-of-thought, not payloads** — `<think>`, *"Let me start by recalling
the basic structure…"* — all labelled `label=1`. Prose *about* an attack was being counted as a
missed attack, which depresses the reported detection rate.

Three separate holes let them through, all now closed in
[`payload_validation.py`](../scripts/payload_validation.py):

| Hole | Fix | Reject reason |
|---|---|---|
| `<think>` alone matched the XSS "tag open" signature | explicit think-tag check | `reasoning_think_tag` |
| Prose *quoting* a payload satisfied the signature test | reasoning-marker regex, applied **before** the signature check | `reasoning_prose` |
| Nothing detected first-person reasoning at all | sentence-shape check | `reasoning_sentence_shape` |

Re-validating the committed corpus: **Code Llama 40 → 40 (untouched)**, DeepSeek **38 → 26**
(12 prose rows rejected: 9 `reasoning_prose`, 2 `reasoning_sentence_shape`, 1 `reasoning_think_tag`).
Verified against real attacker payloads (PayloadsAllTheThings polyglots, `' or ''-'`,
`<img src=x onerror=alert(1)>`) — all still accepted.

### ✅ boolean_blind signature gap (fixed)

Separately, the tautology signature matched only `OR`:

```python
re.compile(r"\bor\b\s*['\"]?\s*\d+\s*=\s*\d+", re.I)         # before
re.compile(r"\b(or|and)\b\s*['\"]?\s*\d+\s*=\s*\d+", re.I)   # after
```

So `AND 1=1` / `AND 1=2` — the canonical boolean-blind pair, and one of the seven families the
generator produces — were rejected as `no_sqli_signature`. That biased the accepted corpus
against its own family. Both bare forms now pass.

In [ ]:
# ==========================================================================
# STAGE 6 - OLLAMA + LLM PAYLOAD GENERATION (optional; committed CSVs by default)
# ==========================================================================
RUN_STAGE6 = False          # True => pull ~8-9GB models and regenerate

CODELLAMA_TAG = "codellama:13b"
DEEPSEEK_TAG  = "huihui_ai/deepseek-r1-abliterated:14b-qwen-distill"
OLLAMA_HOST, OLLAMA_PORT = "localhost", 11434

# ---- HOW MANY PAYLOADS DO WE WANT TO KEEP? -------------------------------
# These are the numbers that end up in the frozen test corpus.
CL_TARGET = 500      # Code Llama: 500 SQLi + 500 XSS  = 1000 kept
DS_TARGET = 300      # DeepSeek  : 300 SQLi + 300 XSS  =  600 kept
#                                              TOTAL   = 1600 kept

# ---- HOW MANY DO WE ASK THE MODEL FOR? -----------------------------------
# More than we keep, because payloads are thrown away for three reasons:
#   1. validation failure  (not a real payload / refusal / reasoning prose)
#   2. duplicates          (deduped on a normalised key, so near-copies count)
#   3. malformed output    (model ignored the JSON format)
#
# The generator asks in BATCHES of 20 and STOPS as soon as it has `target`
# accepted, so over-provisioning costs nothing when yield is good - it just
# stops early. Under-provisioning silently returns fewer payloads than asked,
# which is exactly what happened to the committed smoke test.
#
# Observed acceptance: Code Llama 78.4%, DeepSeek 10.1% (and DeepSeek's will
# fall further now that reasoning-prose is correctly rejected).
BATCH_SIZE   = 20    # payloads requested per batch (matches the scripts)
SAFETY_x     = 1.5   # extra margin on top of the observed rate

CL_ACCEPT_OBS = 0.784
DS_ACCEPT_OBS = 0.101

def _batches_for(target, accept_rate):
    """Batches needed to land `target` accepted payloads, with margin."""
    import math
    raw_needed = target / max(accept_rate, 0.01) * SAFETY_x
    return int(math.ceil(raw_needed / BATCH_SIZE))

CL_BATCHES = _batches_for(CL_TARGET, CL_ACCEPT_OBS)   # ~48
DS_BATCHES = _batches_for(DS_TARGET, DS_ACCEPT_OBS)   # ~223

# Smoke test first? Uncomment to prove the plumbing in ~5 minutes:
# CL_TARGET, DS_TARGET, CL_BATCHES, DS_BATCHES = 20, 20, 3, 6

GEN_FILES = {
    "codellama": {"holdout":  EVAL / "codellama_holdout.csv",
                  "rejects":  EVAL / "codellama_rejects.csv",
                  "manifest": EVAL / "codellama_run_manifest.json"},
    "deepseek":  {"holdout":  EVAL / "deepseek_holdout.csv",
                  "rejects":  EVAL / "deepseek_rejects.csv",
                  "manifest": EVAL / "deepseek_run_manifest.json"},
}

# ---- plain-language summary of what this run will produce ----------------
print("=" * 68)
print("PAYLOAD PLAN - what this stage will produce")
print("=" * 68)
_plan = [("Code Llama", CL_TARGET, CL_BATCHES, CL_ACCEPT_OBS),
         ("DeepSeek-R1", DS_TARGET, DS_BATCHES, DS_ACCEPT_OBS)]
for _nm, _t, _b, _a in _plan:
    print("%-12s keep %d SQLi + %d XSS = %d payloads"
          % (_nm, _t, _t, _t * 2))
    print("%-12s ask for up to %d x %d = %d raw (%.0f%% typically accepted)"
          % ("", _b, BATCH_SIZE, _b * BATCH_SIZE, _a * 100))
print("-" * 68)
print("TOTAL KEPT (if all targets met): %d payloads" % ((CL_TARGET + DS_TARGET) * 2))
print("Duplicates are dropped on a normalised key, so near-copies do not count")
print("twice. The generator stops as soon as a target is reached.")
print("=" * 68)
print()

with stage(
    "LLM payload generation (held-out corpus)",
    "Produce the LLM-authored SQLi/XSS evasion payloads used ONLY as test data "
    "(Section 3.5), or stage the committed corpora if not regenerating.",
    outputs={"%s %s" % (m, k): v
             for m, d in GEN_FILES.items() for k, v in d.items()},
    params={"RUN_STAGE6": RUN_STAGE6, "codellama": CODELLAMA_TAG,
            "deepseek": DEEPSEEK_TAG,
            "cl_batches/target": "%d/%d" % (CL_BATCHES, CL_TARGET),
            "ds_batches/target": "%d/%d" % (DS_BATCHES, DS_TARGET),
            "decoding": "temperature=0, top_k=1 (controlled generation)"},
    optional=True,
) as rec:

    if not RUN_STAGE6:
        skip("RUN_STAGE6 is False - staging the committed corpora instead of "
             "regenerating (~8-9 GB of model pulls avoided).")
        # The committed CSVs live in colab/; Stage 7 reads data/eval/. Copy them
        # across, but NEVER overwrite a fresher file already in data/eval/.
        staged = 0
        for model, files in GEN_FILES.items():
            for kind, dest in files.items():
                src = ROOT / "colab" / dest.name
                if src.exists() and not dest.exists():
                    shutil.copy2(src, dest); staged += 1
                    print("      staged colab/%s -> data/eval/%s" % (src.name, dest.name))
                elif src.exists() and dest.exists():
                    same = sha256_file(src) == sha256_file(dest)
                    print("      data/eval/%s already present (%s colab/ copy)"
                          % (dest.name, "identical to" if same else "DIFFERS from"))
        note("Staged %d file(s) from colab/ into data/eval/." % staged)
    else:
        # -- 6a. zstd BEFORE ollama (both the installer and the blobs need it) --
        if IS_COLAB:
            sh("apt-get -qq update && apt-get -qq install -y zstd")
            sh("curl -fsSL https://ollama.com/install.sh | sh")
            subprocess.Popen("ollama serve > /content/ollama.log 2>&1", shell=True)
            note("Started `ollama serve` in the background (log: /content/ollama.log).")

        # -- 6b. wait for the daemon, explicitly -------------------------------
        import urllib.request
        up = False
        for attempt in range(30):
            try:
                urllib.request.urlopen("http://%s:%d/api/tags" % (OLLAMA_HOST, OLLAMA_PORT),
                                       timeout=3)
                up = True; break
            except Exception:
                time.sleep(2)
        check(up, "Ollama daemon reachable at %s:%d after %d attempts"
                  % (OLLAMA_HOST, OLLAMA_PORT, attempt + 1))
        if not up:
            note("If this failed on Colab, read /content/ollama.log - the usual "
                 "cause is zstd missing when the installer unpacked.")
            raise RuntimeError("Ollama not reachable; cannot generate.")

        # -- 6c. pull the tags -------------------------------------------------
        for tag in (CODELLAMA_TAG, DEEPSEEK_TAG):
            rc, _ = sh(["ollama", "pull", tag], timeout=3600)
            check(rc == 0, "pulled %s" % tag)
        sh(["ollama", "list"])

        # -- 6d. generate ------------------------------------------------------
        rc, _ = sh([sys.executable, str(SCRIPTS / "25_generate_codellama_corpus.py"),
                    "--model", CODELLAMA_TAG, "--host", OLLAMA_HOST,
                    "--port", str(OLLAMA_PORT), "--batches", str(CL_BATCHES),
                    "--target-per-type", str(CL_TARGET)], cwd=ROOT, timeout=14400)
        check(rc == 0, "Code Llama generation exited 0")

        ds_cmd = [sys.executable, str(SCRIPTS / "26_generate_deepseek_corpus.py"),
                  "--model", DEEPSEEK_TAG, "--host", OLLAMA_HOST,
                  "--port", str(OLLAMA_PORT), "--batches", str(DS_BATCHES),
                  "--target-per-type", str(DS_TARGET)]
        if IS_COLAB:
            ds_cmd.append("--no-round2")
            note("--no-round2: the ModSecurity-PL1 feedback loop needs Docker, "
                 "which Colab does not provide. Round-1 payloads only.")
        rc, _ = sh(ds_cmd, cwd=ROOT, timeout=14400)
        check(rc == 0, "DeepSeek generation exited 0")

    # -- 6e. audit whatever we ended up with (generated OR staged) -------------
    # Imported locally: Stage 6 must run standalone, not only after Stage 3.
    import csv as _csv, collections
    print("\n   GENERATED CORPUS AUDIT")
    GEN_STATS = {}
    for model, files in GEN_FILES.items():
        hp, rp = files["holdout"], files["rejects"]
        if not hp.exists():
            print("      %-10s [MISSING] %s" % (model, hp.name))
            check(False, "%s hold-out CSV exists" % model)
            continue
        with open(hp, newline="", encoding="utf-8") as f:
            rows = list(_csv.DictReader(f))
        n_rej = 0
        if rp.exists():
            with open(rp, newline="", encoding="utf-8") as f:
                n_rej = sum(1 for _ in _csv.DictReader(f))
        by_type = collections.Counter(r.get("attack_type") for r in rows)
        by_fam  = collections.Counter(r.get("attack_family") for r in rows)
        uniq    = len({r.get("payload") for r in rows})
        total   = len(rows) + n_rej
        print("      %-10s accepted=%-5d rejected=%-5d  accept-rate=%.1f%%  unique=%d"
              % (model, len(rows), n_rej,
                 100.0 * len(rows) / max(total, 1), uniq))
        print("                 by type: %s" % dict(by_type))
        print("                 by family: %s" % dict(by_fam))
        GEN_STATS[model] = {"accepted": len(rows), "rejected": n_rej,
                            "unique": uniq, "by_type": dict(by_type)}
        check(len(rows) > 0, "%s produced at least one accepted payload" % model)
        check(uniq == len(rows),
              "%s payloads are all unique (%d unique / %d rows)" % (model, uniq, len(rows)))
        # ---- the manifest is where smoke-test-vs-full-run is decided --------
        if files["manifest"].exists():
            man = json.loads(files["manifest"].read_text(encoding="utf-8"))
            tgt = man.get("target_per_type")
            acc = man.get("acceptance_rate")
            met = man.get("target_met", {})
            print("                 model_tag  : %s" % man.get("model_tag"))
            print("                 digest     : %s" % str(man.get("model_digest"))[:16])
            print("                 decoding   : %s" % man.get("controlled_generation"))
            print("                 target/type: %s   acceptance: %s" % (tgt, acc))
            print("                 target_met : %s" % met)
            GEN_STATS[model]["manifest"] = {"target_per_type": tgt,
                                            "acceptance_rate": acc,
                                            "target_met": met}
            # A run that did not hit its per-type target is not reportable.
            for k, v in (met or {}).items():
                check(bool(v), "%s: %s target was met" % (model, k))
            # Distinguish a smoke test from a full protocol run, loudly.
            full_target = CL_TARGET if model == "codellama" else DS_TARGET
            if tgt is not None and tgt < full_target:
                note("%s ran with target_per_type=%s but the full protocol is %d. "
                     "This is a SMOKE TEST (%d accepted payloads). Enough to "
                     "exercise the pipeline; NOT enough to report a detection "
                     "rate from. Set RUN_STAGE6=True with the full batch counts "
                     "for thesis figures." % (model, tgt, full_target, len(rows)))
            if acc is not None and acc < 0.5:
                note("%s acceptance rate is %.1f%% - low yield is itself a "
                     "finding; %s documents every rejection reason."
                     % (model, 100 * acc, rp.name))
            if man.get("round2_enabled") is False:
                note("%s Round-2 (ModSecurity PL1 feedback) was DISABLED, so "
                     "these are Round-1 payloads only." % model)

        # ---- re-validate against the CURRENT validator ---------------------
        # A corpus generated before the reasoning-prose filter existed still
        # lists its prose rows as accepted. Re-checking here means a stale
        # corpus reports itself rather than silently inflating the attack count.
        try:
            sys.path.insert(0, str(SCRIPTS))
            import importlib, payload_validation
            importlib.reload(payload_validation)
            stale = {}
            for r in rows:
                ok, why = payload_validation.validate(str(r.get("payload", "")),
                                                      r.get("attack_type", ""))
                if not ok:
                    stale[why] = stale.get(why, 0) + 1
            n_stale = sum(stale.values())
            if n_stale:
                print("      re-validation: %d/%d stored rows FAIL the current "
                      "validator" % (n_stale, len(rows)))
                for w, c in sorted(stale.items()):
                    print("                     %-28s %d" % (w, c))
            check(n_stale == 0,
                  "%s: every stored row still passes the current validator "
                  "(%d would now be rejected - regenerate with RUN_STAGE6=True "
                  "to clear them)" % (model, n_stale))
            GEN_STATS[model]["stale_rejects"] = stale
        except Exception as e:
            note("re-validation skipped for %s: %s" % (model, e))
    # ---------------------------------------------------------------------
    # CHECKPOINT - a compact block you can copy and paste back to Claude to
    # confirm this stage is on track. Deliberately small and self-describing:
    # counts, rates, targets met, and any stale rows, with no payload text
    # (the payloads themselves are attack strings - keep them in the files).
    # ---------------------------------------------------------------------
    print()
    print("+" + "-" * 66 + "+")
    print("| STAGE 6 CHECKPOINT - copy everything between the +---+ lines     |")
    print("+" + "-" * 66 + "+")
    print("RUN_ID        : %s" % RUN_ID)
    print("regenerated   : %s" % RUN_STAGE6)
    print("targets       : codellama %d/type, deepseek %d/type" % (CL_TARGET, DS_TARGET))
    print("batches       : codellama %d, deepseek %d (x%d per batch)"
          % (CL_BATCHES, DS_BATCHES, BATCH_SIZE))
    _tot_keep = 0
    for _m in ("codellama", "deepseek"):
        _g = GEN_STATS.get(_m)
        if not _g:
            print("%-13s : MISSING" % _m); continue
        _tot_keep += _g["accepted"]
        _man = _g.get("manifest", {})
        _stale = _g.get("stale_rejects", {})
        print("%-13s : accepted=%d rejected=%d rate=%.1f%% unique=%d"
              % (_m, _g["accepted"], _g["rejected"],
                 100.0 * _g["accepted"] / max(_g["accepted"] + _g["rejected"], 1),
                 _g["unique"]))
        print("%-13s   by_type=%s" % ("", _g["by_type"]))
        print("%-13s   target_per_type=%s target_met=%s"
              % ("", _man.get("target_per_type"), _man.get("target_met")))
        if _stale:
            print("%-13s   STALE (fail current validator)=%s" % ("", _stale))
    print("total kept    : %d payloads" % _tot_keep)
    _fails = [c["msg"] for c in rec["checks"] if not c["ok"]]
    print("failed checks : %d" % len(_fails))
    for _f in _fails:
        print("   - %s" % _f[:80])
    print("+" + "-" * 66 + "+")

    rec["generation_stats"] = GEN_STATS

---
## Stage 7 — Assemble the LLM hold-out benchmark

**Script:** [`scripts/27_assemble_llm_holdout.py`](../scripts/27_assemble_llm_holdout.py)

Merges both LLM attack subsets with held-out **benign** rows into one labelled CSV:
`data/eval/llm_holdout_full.csv` with columns `text,label,attack_type,generator`.

### Why the `generator` column must survive

Section 3.5 requires results stay **disaggregated by generator**. Code Llama and DeepSeek produce
qualitatively different payloads (the abliterated reasoning model writes more polymorphic
expressions), and a single pooled detection rate hides the case where the detector handles one
model well and the other badly. Stage 8's `19_eval_by_type.py` segments on this column — if it is
dropped, that analysis silently collapses to an average.

### Benign rows and the shortfall warning

Benign negatives come from the committed `data/eval/holdout_eval.csv` (`label==0`; 198 rows
available). The script targets 600 by default and **prints a `[SHORTFALL]`** when it cannot reach
it, using all available rows instead. This matters: **FPR resolution is bounded by the benign
count.** With 198 benign rows the smallest non-zero FPR you can measure is 1/198 ≈ 0.51 % — an
FPR reported as "0.0 %" means "no false positives among 198 rows", not "zero false positive rate".
The cell makes that resolution limit explicit.

In [ ]:
# ==========================================================================
# STAGE 7 - ASSEMBLE THE LLM HOLD-OUT BENCHMARK
# ==========================================================================
BENIGN_TARGET   = 600
LLM_HOLDOUT_CSV = EVAL / "llm_holdout_full.csv"
LLM_MANIFEST    = EVAL / "llm_holdout_manifest.json"

with stage(
    "Assemble LLM hold-out benchmark",
    "Merge both LLM attack subsets with held-out benign rows into one labelled "
    "CSV, preserving the generator column so results stay disaggregated.",
    inputs={"codellama holdout": GEN_FILES["codellama"]["holdout"],
            "deepseek holdout":  GEN_FILES["deepseek"]["holdout"],
            "benign source":     EVAL / "holdout_eval.csv"},
    outputs={"assembled benchmark": LLM_HOLDOUT_CSV, "manifest": LLM_MANIFEST},
    params={"benign_target": BENIGN_TARGET},
) as rec:

    rc, _ = sh([sys.executable, str(SCRIPTS / "27_assemble_llm_holdout.py"),
                "--benign-target", str(BENIGN_TARGET),
                "--out", str(LLM_HOLDOUT_CSV)], cwd=ROOT)
    check(rc == 0, "27_assemble_llm_holdout.py exited 0")

    import pandas as pd
    check(LLM_HOLDOUT_CSV.exists(), "assembled CSV was written", fatal=True)
    hold = pd.read_csv(LLM_HOLDOUT_CSV)

    print("\n   ASSEMBLED BENCHMARK")
    print("      rows ................. %d" % len(hold))
    print("      columns .............. %s" % hold.columns.tolist())
    print("      label counts ......... %s" % hold["label"].value_counts().to_dict())
    if "generator" in hold:
        print("\n      rows by generator:")
        for g, c in hold["generator"].value_counts().items():
            print("         %-18s %5d" % (g, c))
    if "attack_type" in hold:
        print("\n      rows by attack_type:")
        for t, c in hold["attack_type"].value_counts().items():
            print("         %-18s %5d" % (t, c))

    n_benign = int((hold["label"] == 0).sum())
    n_attack = int((hold["label"] == 1).sum())
    fpr_res  = 100.0 / max(n_benign, 1)
    print("\n   MEASUREMENT RESOLUTION (read this before quoting an FPR)")
    print("      benign rows .......... %d" % n_benign)
    print("      smallest non-zero FPR  %.3f%%  (= 1/%d)" % (fpr_res, n_benign))
    print("      => a reported FPR of 0.0%% means 'no false positives among %d rows',"
          % n_benign)
    print("         NOT 'a zero false-positive rate'.")
    if n_benign < BENIGN_TARGET:
        note("SHORTFALL: %d benign rows available vs target %d. This is expected - "
             "holdout_eval.csv holds 198 negatives - but it bounds FPR resolution "
             "as printed above." % (n_benign, BENIGN_TARGET))

    print()
    check(len(hold) > 0, "benchmark is non-empty")
    check({"text", "label"} <= set(hold.columns), "required columns text,label present")
    check("generator" in hold.columns,
          "generator column preserved (Section 3.5 requires disaggregated results)")
    check(n_attack > 0 and n_benign > 0, "both classes present (attack=%d benign=%d)"
          % (n_attack, n_benign))
    check(hold["text"].isna().sum() == 0, "no null payload text")
    dupes = int(hold.duplicated(subset=["text"]).sum())
    check(dupes == 0, "no duplicate payload text across the benchmark (%d dupes)" % dupes)
    rec["holdout_stats"] = {"rows": len(hold), "attack": n_attack, "benign": n_benign,
                            "fpr_resolution_pct": round(fpr_res, 4),
                            "by_generator": hold["generator"].value_counts().to_dict()
                            if "generator" in hold else {}}
    HOLDOUT_DF = hold

---
## Stage 8 — Evaluation

Three complementary views, each answering a different question. Running only the first is the
most common way to end up with a number you cannot defend.

| Script | Question it answers | Output |
|---|---|---|
| [`17_evaluate_csv.py`](../scripts/17_evaluate_csv.py) | **Headline:** overall detection rate and FPR on the LLM hold-out | `reports/llm_holdout_results.json` |
| [`19_eval_by_type.py`](../scripts/19_eval_by_type.py) | **Segmented:** per attack type *and* per generator — where does it fail? | `reports/eval_by_type.json`, `roc_by_type.png` |
| [`20_significance_holdout.py`](../scripts/20_significance_holdout.py) | **Is the stack better than its parts,** or is the gap noise? | `reports/significance_holdout.json` |

### On `19_eval_by_type.py`'s type derivation

It classifies rows as SQL-syntax, `nl_intent` (plain-language attacks), or `other`. The
`nl_intent` category matters: **plain-language attack rows were deliberately excluded from
training** (Chapter 3 edits, item 7). They are in the corpus as `nl_injection` — only **2 rows**,
per Stage 3's family table. Any per-type metric computed over 2 rows is descriptive, not
statistical; do not report a confidence interval on it.

### Interpreting the headline honestly

Both baselines are also evaluated on the *same* rows so the comparison is like-for-like. The
detection rate here is on **payloads the detector has never seen the expression of** — this is the
evasion-resistance number, and it is expected to be lower than the test-split number from Stage 5.
If they are equal, suspect that generated payloads leaked into training.

In [ ]:
# ==========================================================================
# STAGE 8 - EVALUATION (headline, segmented, significance)
# ==========================================================================
RES_HEADLINE = REPORTS / "llm_holdout_results.json"
RES_BY_TYPE  = REPORTS / "eval_by_type.json"
RES_SIGNIF   = REPORTS / "significance_holdout.json"

with stage(
    "Evaluate on the LLM-generated hold-out",
    "Headline detection/FPR, per-type + per-generator segmentation, and a "
    "significance test of the stack against its individual base learners.",
    inputs={"benchmark": LLM_HOLDOUT_CSV,
            "RF": MODELS / "rf2.pkl", "LSTM": MODELS / "lstm_best.keras",
            "meta": MODELS / "meta.pkl", "vectorizer": MODELS / "ngram_vectorizer.pkl"},
    outputs={"headline": RES_HEADLINE, "by type": RES_BY_TYPE,
             "significance": RES_SIGNIF, "ROC plot": REPORTS / "roc_by_type.png"},
    params={"csv": str(LLM_HOLDOUT_CSV)},
) as rec:

    for lbl, p in (("RF", MODELS / "rf2.pkl"), ("LSTM", MODELS / "lstm_best.keras"),
                   ("meta", MODELS / "meta.pkl"),
                   ("vectorizer", MODELS / "ngram_vectorizer.pkl")):
        if not check(p.exists(), "model present: %s" % p.name):
            skip("models are missing", produced_by="Stage 5 (18_train_stacked.py)")
            raise FileNotFoundError("cannot evaluate without %s" % p)

    # ---- 8a. headline ------------------------------------------------------
    print("\n   [8a] HEADLINE - overall detection rate + FPR")
    rc, _ = sh([sys.executable, str(SCRIPTS / "17_evaluate_csv.py"),
                "--csv", str(LLM_HOLDOUT_CSV), "--out", str(RES_HEADLINE)], cwd=ROOT)
    check(rc == 0, "17_evaluate_csv.py exited 0")
    HEADLINE = preview_json(RES_HEADLINE) if RES_HEADLINE.exists() else None
    check(HEADLINE is not None, "headline results JSON written")

    # ---- 8b. segmented -----------------------------------------------------
    print("\n   [8b] SEGMENTED - per attack type and per generator")
    rc, _ = sh([sys.executable, str(SCRIPTS / "19_eval_by_type.py"),
                "--csv", str(LLM_HOLDOUT_CSV)], cwd=ROOT)
    check(rc == 0, "19_eval_by_type.py exited 0")
    BY_TYPE = preview_json(RES_BY_TYPE, max_chars=2000) if RES_BY_TYPE.exists() else None
    check(BY_TYPE is not None, "per-type results JSON written")
    note("nl_intent rows number 2 in the whole corpus (see Stage 3) - treat any "
         "nl_intent metric as descriptive, never as a statistic.")

    # ---- 8c. significance --------------------------------------------------
    print("\n   [8c] SIGNIFICANCE - stack vs its individual base learners")
    rc, _ = sh([sys.executable, str(SCRIPTS / "20_significance_holdout.py")], cwd=ROOT)
    check(rc == 0, "20_significance_holdout.py exited 0")
    SIGNIF = preview_json(RES_SIGNIF) if RES_SIGNIF.exists() else None
    check(SIGNIF is not None, "significance JSON written")

    rec["results"] = {"headline": HEADLINE, "by_type": BY_TYPE, "significance": SIGNIF}
    note("These three JSONs are the evaluation record; Stage 13 bundles them.")

---
## Stage 9 — Ablation study + internal significance ⟨optional⟩

| Script | Purpose |
|---|---|
| [`14_ablation_study.py`](../scripts/14_ablation_study.py) | The six-condition ablation from Chapter 3 — which components actually earn their place |
| [`13_statistical_significance.py`](../scripts/13_statistical_significance.py) | Significance testing on the internal test split |

### The stale-artifact trap — the single most important note in this notebook

Both scripts take **no arguments**. They read fixed paths:
`data/prepared/rf_{train,val,test}_v2.csv`, `data/prepared/lstm_*.npz`, and `models/*`.

That means they are only valid if those prepared splits were written by **the same training run**
that produced the models currently in `models/`. Stage 5 rewrites them for exactly this reason —
its own comment: *"Without this they load stale .npz/.csv from a different split size and produce
garbage."*

The failure mode is nasty because **it does not crash**: mismatched-but-loadable arrays yield
plausible-looking numbers that are meaningless. So the cell below **verifies mtime ordering**
before running anything: prepared splits must be no older than the models. If they are older, it
refuses and tells you to re-run Stage 5.

In [ ]:
# ==========================================================================
# STAGE 9 - ABLATION + INTERNAL SIGNIFICANCE (optional)
# Guarded: refuses to run against prepared splits older than the models.
# ==========================================================================
RUN_STAGE9 = True

RES_ABLATION = REPORTS / "ablation_study_results.json"
RES_STATS    = REPORTS / "statistical_significance.json"

with stage(
    "Ablation study + internal significance",
    "Six-condition ablation and significance testing on the internal test split, "
    "guarded against the stale-prepared-splits failure that produces plausible "
    "but meaningless numbers.",
    inputs={"rf_train_v2": PREP / "rf_train_v2.csv",
            "rf_test_v2":  PREP / "rf_test_v2.csv",
            "lstm_test":   PREP / "lstm_test.npz",
            "RF model":    MODELS / "rf2.pkl"},
    outputs={"ablation": RES_ABLATION, "significance": RES_STATS},
    params={"RUN_STAGE9": RUN_STAGE9},
    optional=True,
) as rec:

    # Defined locally so Stage 9 can run without Stage 5 having executed this
    # session (e.g. re-running only the evaluation half of the notebook).
    MODEL_FILES = {"RF":         MODELS / "rf2.pkl",
                   "LSTM":       MODELS / "lstm_best.keras",
                   "meta":       MODELS / "meta.pkl",
                   "vectorizer": MODELS / "ngram_vectorizer.pkl"}

    needed = [PREP / "rf_train_v2.csv", PREP / "rf_val_v2.csv", PREP / "rf_test_v2.csv",
              PREP / "lstm_train.npz", PREP / "lstm_val.npz", PREP / "lstm_test.npz"]
    absent = [p for p in needed if not p.exists()]

    if not RUN_STAGE9:
        skip("RUN_STAGE9 is False.")
    elif absent:
        skip("%d prepared split file(s) missing." % len(absent),
             produced_by="Stage 5 (18_train_stacked.py rewrites data/prepared/*)")
        for p in absent:
            print("          missing: %s" % p.name)
    else:
        # ---- the freshness guard -------------------------------------------
        model_mtime = max(p.stat().st_mtime for p in MODEL_FILES.values() if p.exists())
        prep_mtime  = min(p.stat().st_mtime for p in needed)
        stale_by    = model_mtime - prep_mtime
        print("   FRESHNESS GUARD")
        print("      newest model mtime ....... %s"
              % datetime.fromtimestamp(model_mtime, timezone.utc).isoformat()[:19])
        print("      oldest prepared mtime .... %s"
              % datetime.fromtimestamp(prep_mtime, timezone.utc).isoformat()[:19])
        fresh = stale_by <= 60      # 60s slack: Stage 5 writes models then splits
        check(fresh,
              "prepared splits are no older than the models (delta %.0fs) - if this "
              "FAILS the splits are stale and results would be meaningless" % stale_by)

        # Shape agreement is the second, stronger check.
        import pandas as pd, numpy as np
        rf_te   = pd.read_csv(needed[2], nrows=1)
        n_rf_te = sum(1 for _ in open(needed[2], encoding="utf-8")) - 1
        lstm_te = np.load(needed[5])
        print("\n   SHAPE AGREEMENT")
        print("      rf_test_v2.csv ....... %d rows x %d cols" % (n_rf_te, rf_te.shape[1]))
        print("      lstm_test.npz ........ X%s y%s"
              % (lstm_te["X"].shape, lstm_te["y"].shape))
        same_n = (n_rf_te == lstm_te["X"].shape[0])
        check(same_n,
              "RF and LSTM test splits describe the SAME rows (%d vs %d) - a "
              "mismatch means they came from different runs"
              % (n_rf_te, lstm_te["X"].shape[0]))
        check(rf_te.shape[1] - 1 == 319,
              "RF feature width is 319 (19 structural + 300 n-gram); got %d"
              % (rf_te.shape[1] - 1))

        if not (fresh and same_n):
            skip("Refusing to run: prepared splits do not match the current models. "
                 "Re-run Stage 5 (RUN_STAGE5=True) first.")
        else:
            print("\n   [9a] ABLATION")
            rc, _ = sh([sys.executable, str(SCRIPTS / "14_ablation_study.py")], cwd=ROOT)
            check(rc == 0, "14_ablation_study.py exited 0")
            ABLATION = preview_json(RES_ABLATION, max_chars=2200) if RES_ABLATION.exists() else None
            check(ABLATION is not None, "ablation JSON written")

            print("\n   [9b] INTERNAL SIGNIFICANCE")
            rc, _ = sh([sys.executable, str(SCRIPTS / "13_statistical_significance.py")],
                       cwd=ROOT)
            check(rc == 0, "13_statistical_significance.py exited 0")
            STATS = preview_json(RES_STATS) if RES_STATS.exists() else None
            check(STATS is not None, "significance JSON written")
            rec["results"] = {"ablation": ABLATION, "significance": STATS}

---
## Stage 10 — Adversarial red-team probes ⟨optional⟩

Read-only with respect to the models — they load `models/*` and score fixed case files. These are
the honest-failure stages: they are supposed to find weaknesses.

| Script | Probe | Cases | Recorded result |
|---|---|---|---|
| [`16_claude_redteam.py`](../scripts/16_claude_redteam.py) | Hand-crafted adversarial set | `scripts/redteam_cases.json` (`[label, tag, text]`) | ~47 % of hand-crafted attacks evade |
| [`23_claude_evasion_probe.py`](../scripts/23_claude_evasion_probe.py) | Targeted evasion probe | embedded | `claude_evasion_probe.json` |
| [`24_polymorphic_probe.py`](../scripts/24_polymorphic_probe.py) | Polymorphic mutation | embedded | robust to **form** (400/400), fragile to **expression** (2/22) |

### The finding these probes exist to preserve

From the commit history: the detector is **robust to form and fragile to expression**. Mutating
the *syntax* of a known attack (400/400 caught) barely dents it; rephrasing the *intent* into an
unfamiliar expression (2/22 caught) defeats it. That asymmetry is the honest limitation of a
character/n-gram-structural model, and it is the reason the plain-language `nl_injection` rows
were pulled out of training rather than used to paper over the gap.

Do not treat a low score here as a bug to be tuned away. It is a reported result.

In [ ]:
# ==========================================================================
# STAGE 10 - ADVERSARIAL RED-TEAM PROBES (read-only w.r.t. models)
# ==========================================================================
RUN_STAGE10 = True

PROBES = [
    ("hand-crafted red-team", SCRIPTS / "16_claude_redteam.py",
     REPORTS / "claude_redteam_results.json",
     "Fixed adversarial cases from scripts/redteam_cases.json"),
    ("targeted evasion probe", SCRIPTS / "23_claude_evasion_probe.py",
     REPORTS / "claude_evasion_probe.json",
     "Targeted evasion attempts against known decision boundaries"),
    ("polymorphic probe", SCRIPTS / "24_polymorphic_probe.py",
     REPORTS / "polymorphic_probe.json",
     "Form-mutation robustness vs expression fragility"),
]

with stage(
    "Adversarial red-team probes",
    "Score fixed adversarial case sets against the current models to record where "
    "the detector fails - these are reported limitations, not defects to tune away.",
    inputs={"RF": MODELS / "rf2.pkl", "LSTM": MODELS / "lstm_best.keras",
            "red-team cases": SCRIPTS / "redteam_cases.json"},
    outputs={name: out for name, _, out, _ in PROBES},
    params={"RUN_STAGE10": RUN_STAGE10, "probes": len(PROBES)},
    optional=True,
) as rec:

    if not RUN_STAGE10:
        skip("RUN_STAGE10 is False.")
    else:
        PROBE_RESULTS = {}
        for name, script, out, why in PROBES:
            print("\n   PROBE: %s" % name)
            print("      rationale: %s" % why)
            if not script.exists():
                check(False, "%s exists" % script.name); continue
            # PYTHONIOENCODING: the probe scripts print attack payloads containing
            # non-cp1252 characters (full-width quotes, CJK, emoji). On a Windows
            # console that raises UnicodeEncodeError and kills an otherwise
            # healthy run. Force UTF-8 for the child process.
            probe_env = dict(os.environ, PYTHONIOENCODING="utf-8")
            rc, _ = sh([sys.executable, str(script)], cwd=ROOT, timeout=3600,
                       env=probe_env)
            check(rc == 0, "%s exited 0" % script.name)
            if out.exists():
                PROBE_RESULTS[name] = preview_json(out, max_chars=1200)
                check(True, "%s written" % out.name)
            else:
                check(False, "%s written" % out.name)
        rec["probe_results"] = PROBE_RESULTS
        note("Expected shape of the result: high catch rate on form mutations, "
             "low catch rate on rephrased intent. If BOTH are high, verify the "
             "probe actually loaded the current models rather than silently failing.")

---
## Stage 11 — ModSecurity CRS baseline ⟨needs Docker⟩

**Script:** [`scripts/21_modsec_holdout.py`](../scripts/21_modsec_holdout.py)

Scores the same hold-out through **ModSecurity + OWASP CRS at Paranoia Level 1** — the
industry-standard rule-based WAF. This is the comparison that makes the ML result meaningful: a
detection rate means little without the rules-based number on *identical* rows.

**Colab cannot run this** — there is no Docker. Run it locally with
[`docker-compose.pl1.yml`](../../docker-compose.pl1.yml), then copy `reports/modsec_holdout.json`
back. The cell probes the endpoint first and skips cleanly (naming the compose file) rather than
producing a misleading empty result.

In [ ]:
# ==========================================================================
# STAGE 11 - MODSECURITY CRS PL1 BASELINE (requires Docker; skipped on Colab)
# ==========================================================================
RUN_STAGE11   = not IS_COLAB
MODSEC_HOST   = "localhost"
MODSEC_PORT   = 8080
RES_MODSEC    = REPORTS / "modsec_holdout.json"

with stage(
    "ModSecurity CRS PL1 baseline",
    "Score the identical hold-out through a rules-based WAF so the ML detection "
    "rate has a like-for-like industry baseline to be compared against.",
    inputs={"benchmark": LLM_HOLDOUT_CSV,
            "compose file": ROOT.parent / "docker-compose.pl1.yml"},
    outputs={"modsec results": RES_MODSEC},
    params={"RUN_STAGE11": RUN_STAGE11, "endpoint": "%s:%d" % (MODSEC_HOST, MODSEC_PORT)},
    optional=True,
) as rec:

    if not RUN_STAGE11:
        skip("Docker is unavailable here (Colab provides no Docker daemon).",
             produced_by="run locally: docker compose -f docker-compose.pl1.yml up -d, "
                         "then python scripts/21_modsec_holdout.py")
    else:
        import urllib.request, urllib.error
        reachable = False
        try:
            urllib.request.urlopen("http://%s:%d/" % (MODSEC_HOST, MODSEC_PORT), timeout=5)
            reachable = True
        except urllib.error.HTTPError:
            reachable = True          # a 403/404 still proves the WAF is listening
        except Exception as e:
            note("ModSecurity endpoint probe failed: %s" % e)
        check(reachable, "ModSecurity reachable at %s:%d" % (MODSEC_HOST, MODSEC_PORT))

        if not reachable:
            skip("Nothing listening on %s:%d." % (MODSEC_HOST, MODSEC_PORT),
                 produced_by="docker compose -f docker-compose.pl1.yml up -d")
        else:
            rc, _ = sh([sys.executable, str(SCRIPTS / "21_modsec_holdout.py"),
                        "--host", MODSEC_HOST, "--port", str(MODSEC_PORT),
                        "--csv", str(LLM_HOLDOUT_CSV),
                        "--out", str(RES_MODSEC)], cwd=ROOT, timeout=7200)
            check(rc == 0, "21_modsec_holdout.py exited 0")
            MODSEC = preview_json(RES_MODSEC) if RES_MODSEC.exists() else None
            check(MODSEC is not None, "ModSecurity results JSON written")
            rec["results"] = MODSEC

---
## Stage 12 — Multi-seed variance ⟨slow⟩

**Script:** [`scripts/22_multirun_variance.py`](../scripts/22_multirun_variance.py)

Retrains the whole stack **five times under different seeds** and reports mean ± SD.

### Why a single-run number is not reportable

Every headline figure above comes from **one** seeded run. A single number cannot distinguish "the
architecture achieves X" from "this seed achieved X". Section 3.8 / Change 6 requires the mean ± SD,
and this is the only stage that produces it. If you are writing up a headline figure, it should
come from here, not from Stage 8's single run.

**Cost:** 5× a full training run — ~15–30 min on a T4, *hours* on CPU. It is off by default; the
cell refuses to start on CPU unless you explicitly override, because an unattended CPU run will
outlive a Colab session and lose everything.

In [ ]:
# ==========================================================================
# STAGE 12 - MULTI-SEED VARIANCE (5x full training; off by default)
# ==========================================================================
RUN_STAGE12       = False       # set True for the reportable mean +/- SD
VARIANCE_RUNS     = 5
VARIANCE_EPOCHS   = 6
ALLOW_CPU_VARIANCE = False      # override the CPU guard (expect hours)

RES_VARIANCE = REPORTS / "multirun_variance.json"

with stage(
    "Multi-seed variance study",
    "Retrain the stack under 5 seeds to produce the reportable mean +/- SD, since "
    "a single-run figure cannot separate architecture from seed luck.",
    inputs={"honeypot log": HONEYPOT_LOG},
    outputs={"variance results": RES_VARIANCE},
    params={"RUN_STAGE12": RUN_STAGE12, "runs": VARIANCE_RUNS,
            "epochs": VARIANCE_EPOCHS, "gpu": GPU_AVAILABLE},
    optional=True,
) as rec:

    if not RUN_STAGE12:
        skip("RUN_STAGE12 is False - this costs 5x a full training run. "
             "Enable it when you need the reportable mean +/- SD.")
    elif not GPU_AVAILABLE and not ALLOW_CPU_VARIANCE:
        skip("No GPU detected and ALLOW_CPU_VARIANCE is False. On CPU this is "
             "~%d min of training and will outlive a Colab session."
             % (VARIANCE_RUNS * VARIANCE_EPOCHS * 165 // 60),
             produced_by="set ALLOW_CPU_VARIANCE=True to override, or use a T4")
    else:
        note("This OVERWRITES models/ on each of the %d runs; the final models are "
             "from the LAST seed, not necessarily the best." % VARIANCE_RUNS)
        rc, _ = sh([sys.executable, str(SCRIPTS / "22_multirun_variance.py"),
                    "--runs", str(VARIANCE_RUNS),
                    "--epochs", str(VARIANCE_EPOCHS),
                    "--two-layer",
                    "--out", str(RES_VARIANCE)], cwd=ROOT, timeout=28800)
        check(rc == 0, "22_multirun_variance.py exited 0")
        VARIANCE = preview_json(RES_VARIANCE, max_chars=2500) if RES_VARIANCE.exists() else None
        check(VARIANCE is not None, "variance JSON written")
        rec["results"] = VARIANCE
        note("Quote mean +/- SD from this file for any headline figure.")

---
## Stage 13 — Run report + package

Closes the run. Two artifacts:

1. **`reports/run_report_<RUN_ID>.md`** — a human-readable summary: every stage with status,
   duration, failed checks, and the fingerprint of every artifact produced. This is the file to
   open first when a run "worked" but the numbers look wrong; it tells you which stages actually
   ran, which were skipped and why, and whether any check failed without stopping the run.
2. **`aigis_<RUN_ID>.zip`** — the corpora, results, models, run log and report, for download.

The machine-readable `reports/run_log_<RUN_ID>.jsonl` was written incrementally as each stage
finished, so **it survives even if a later stage crashes the kernel.**

In [ ]:
# ==========================================================================
# STAGE 13 - RUN REPORT + PACKAGE
# ==========================================================================
with stage(
    "Run report + package",
    "Emit a human-readable audit of every stage (status, duration, failed checks, "
    "artifact fingerprints) and bundle the run for download.",
    params={"stages_recorded": len(RUN_LOG)},
) as rec:

    total_s   = time.time() - RUN_STARTED
    n_ok      = sum(1 for r in RUN_LOG if r["status"] == "ok")
    n_skip    = sum(1 for r in RUN_LOG if r["status"] == "skipped")
    n_err     = sum(1 for r in RUN_LOG if r["status"] == "error")
    all_checks = [(r["name"], c) for r in RUN_LOG for c in r["checks"]]
    failed     = [(n, c) for n, c in all_checks if not c["ok"]]

    L = []
    L.append("# AI-GIS pipeline run `%s`\n" % RUN_ID)
    L.append("- **Started:** %s" % datetime.fromtimestamp(RUN_STARTED, timezone.utc)
             .isoformat()[:19] + "Z")
    L.append("- **Duration:** %.1f min" % (total_s / 60.0))
    L.append("- **Stages:** %d ok, %d skipped, %d errored" % (n_ok, n_skip, n_err))
    L.append("- **Checks:** %d run, **%d failed**" % (len(all_checks), len(failed)))
    L.append("- **Environment:** python %s, sklearn %s, tensorflow %s, GPU=%s"
             % (ENV.get("python"), ENV.get("sklearn"), ENV.get("tensorflow"),
                ENV.get("gpu_available")))
    L.append("- **Git:** `%s`" % ENV.get("git"))
    L.append("")

    if failed:
        L.append("## FAILED CHECKS (read these first)\n")
        for nm, c in failed:
            L.append("- **%s** - %s" % (nm, c["msg"]))
        L.append("")
    else:
        L.append("## All checks passed\n")

    L.append("## Stages\n")
    L.append("| # | Stage | Status | Duration | Checks | Notes |")
    L.append("|---|---|---|---|---|---|")
    for r in RUN_LOG:
        nf = sum(1 for c in r["checks"] if not c["ok"])
        L.append("| %d | %s | %s | %.1fs | %d/%d | %s |"
                 % (r["stage_seq"], r["name"], r["status"], r.get("seconds", 0),
                    len(r["checks"]) - nf, len(r["checks"]),
                    (r["notes"][0][:70] + "...") if r["notes"] else ""))
    L.append("")

    L.append("## Artifacts produced\n")
    L.append("| Stage | Artifact | Size | SHA-256 |")
    L.append("|---|---|---|---|")
    for r in RUN_LOG:
        for o in r["outputs"]:
            if o.get("exists") and not o.get("is_dir"):
                L.append("| %d | `%s` | %s | `%s` |"
                         % (r["stage_seq"], Path(o["path"]).name,
                            fmt_bytes(o["bytes"]), o["sha256"][:24]))
    L.append("")

    if any("corpus_stats" in r for r in RUN_LOG):
        cs = next(r["corpus_stats"] for r in RUN_LOG if "corpus_stats" in r)
        L.append("## Training corpus\n")
        L.append("- rows **%d**, sessions **%d**, labels %s"
                 % (cs["rows"], cs["sessions"], cs["labels"]))
        L.append("- exact-duplicate payload rate **%.2f%%**, forced_unique_nonce **%d**"
                 % (cs["dup_rate_pct"], cs["forced_unique_nonce"]))
        L.append("")

    if any("holdout_stats" in r for r in RUN_LOG):
        hs = next(r["holdout_stats"] for r in RUN_LOG if "holdout_stats" in r)
        L.append("## Hold-out benchmark\n")
        L.append("- %d rows (attack %d / benign %d)"
                 % (hs["rows"], hs["attack"], hs["benign"]))
        L.append("- FPR resolution floor: **%.3f%%** - a reported 0.0%% means "
                 "'none among %d benign rows'" % (hs["fpr_resolution_pct"], hs["benign"]))
        L.append("- by generator: %s" % hs["by_generator"])
        L.append("")

    L.append("## Reproducing this run\n")
    L.append("```bash")
    L.append("git checkout %s" % ENV.get("git", "HEAD").split()[0])
    L.append("pip install -r dashboard/requirements.txt   # scikit-learn==%s is load-bearing"
             % SKLEARN_PIN)
    L.append("# then run this notebook with the same RUN_STAGE* flags:")
    # globals().get(): a stage cell that was not run this session simply has no
    # flag defined; report that instead of crashing the final report.
    for nm in ("RUN_STAGE2", "RUN_STAGE4", "RUN_STAGE5", "RUN_STAGE6",
               "RUN_STAGE9", "RUN_STAGE10", "RUN_STAGE11", "RUN_STAGE12"):
        L.append("#   %-12s = %s" % (nm, globals().get(nm, "(cell not run)")))
    L.append("```")

    REPORT_MD = REPORTS / ("run_report_%s.md" % RUN_ID)
    REPORT_MD.write_text("\n".join(L), encoding="utf-8")
    print("   Wrote %s" % REPORT_MD)
    print()
    print("\n".join(L[:40]))

    check(REPORT_MD.exists(), "run report written")
    check(n_err == 0, "no stage errored (%d errored)" % n_err)
    check(not failed, "no check failed across the whole run (%d failed)" % len(failed))

    # ---- package ------------------------------------------------------------
    import zipfile
    ZIP_PATH = Path("/content" if IS_COLAB else str(ROOT.parent)) / ("aigis_%s.zip" % RUN_ID)
    to_pack = []
    for pat in ("data/eval/*.csv", "data/eval/*.json", "reports/*.json",
                "reports/*.md", "reports/*.png", "reports/*.jsonl",
                "models/*.pkl", "models/*.keras"):
        to_pack.extend(sorted(ROOT.glob(pat)))
    with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
        for p in to_pack:
            z.write(p, p.relative_to(ROOT))
    print("\n   Packaged %d files -> %s (%s)"
          % (len(to_pack), ZIP_PATH, fmt_bytes(ZIP_PATH.stat().st_size)))
    check(ZIP_PATH.exists() and ZIP_PATH.stat().st_size > 0, "zip bundle written")

    if IS_COLAB:
        try:
            from google.colab import files
            files.download(str(ZIP_PATH))
        except Exception as e:
            note("Auto-download failed (%s) - fetch it from the Files pane." % e)

print("=" * 70)
print("RUN COMPLETE: %s" % RUN_ID)
print("  report   : %s" % REPORT_MD)
print("  run log  : %s" % RUN_LOG_PATH)
print("=" * 70)

---
## Appendix — Debugging playbook

Symptom-first index. Every entry names the stage whose output tells you the answer.

| Symptom | Most likely cause | Where to look |
|---|---|---|
| Metrics look great but collapse on new data | Generated payloads leaked into training | Stage 6 — payloads must be test-only; Stage 4 — session overlap must be 0 |
| Scores differ from a previous run on identical inputs | scikit-learn version drift unpickling the models | Stage 1 — the `SKLEARN_PIN` check |
| Ablation/significance numbers are nonsense but nothing crashed | Stale `data/prepared/*` from a different training run | Stage 9 — freshness guard + shape agreement |
| `val_auc ≈ 0.5`, LSTM appears not to learn | Unseeded bad init, or `mask_zero` not propagating | Stage 5 — trainer highlights; check the seed |
| Reported FPR is "0.0 %" | Only 198 benign rows — resolution floor is 0.51 % | Stage 7 — measurement resolution block |
| Duplicate-heavy corpus, inflated metrics | A generator family's component space shrank | Stage 3 — `forced_unique_nonce` and duplicate rate |
| Ollama fails to extract the model | `zstd` not installed before the Ollama installer ran | Stage 6 — install order |
| `ollama pull` returns 400 on DeepSeek | Used `huihui-ai` (hyphen, HF) not `huihui_ai` (underscore, Ollama) | Stage 6 — tag note |
| Per-type metrics on `nl_intent` look unstable | There are only **2** such rows in the corpus | Stage 3 — family table |
| Red-team catch rate is low | Expected: robust to form, fragile to expression | Stage 10 — this is a reported finding |
| A stage "did nothing" in 0.2 s | It skipped — read the `[SKIP]` line naming the missing input | any stage banner |

### Reading the run log programmatically

```python
import json
recs = [json.loads(l) for l in open(RUN_LOG_PATH, encoding="utf-8")]

# every failed check across the run
[(r["name"], c["msg"]) for r in recs for c in r["checks"] if not c["ok"]]

# what each stage produced, with hashes
{r["name"]: [(o["label"], o.get("sha256", "")[:16]) for o in r["outputs"]] for r in recs}

# slowest stages
sorted(((r["seconds"], r["name"]) for r in recs), reverse=True)[:5]
```

### Known documentation drift

Two docstrings lag their filenames: `webids23_to_honeypot_log_v9.py` still titles itself `v5`, and
`prepare_honeypot_for_training.py` references a `v4` log. Cosmetic, but worth fixing before citing
script versions in Chapter 3.